<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/05_TVAE_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ==================================================================================================
# NOTEBOOK 05 — TVAE BASELINE
# ==================================================================================================
#
# Research Project:
# A Unified Privacy-Preserving Framework for High-Fidelity Synthetic Data Generation
# Using Statistical and Machine Learning Models
#
# Notebook:
# 05 — TVAE Baseline
#
# Purpose:
# Establish a reproducible non-private neural generative baseline using TVAE.
#
# Scope:
#   - Load canonical Notebook 02 native TRAIN data
#   - Validate frozen generative schema
#   - Detect SDV metadata from TRAIN only
#   - Configure and train TVAE
#   - Record training loss/history
#   - Generate training-sized synthetic data
#   - Validate synthetic schema and leakage policy
#   - Persist model, checkpoint, metadata, history, synthetic data and manifest
#   - Verify persisted artifacts
#
# Explicit exclusions:
#   - No validation/test data
#   - No refitting of Notebook 02 preprocessing
#   - No differential privacy
#   - No statistical guidance from Notebook 03
#   - No SPP-GAN architecture
#   - No privacy accounting
#   - No downstream evaluation
#
# ==================================================================================================

In [4]:
# ==================================================================================================
# NOTEBOOK 05 — TVAE BASELINE
# SECTION 2 — LOAD CONFIGURATION
# ==================================================================================================

print("=" * 100)
print("SECTION 2 — LOAD CONFIGURATION")
print("=" * 100)


# ==================================================================================================
# 2.0 — IMPORTS
# ==================================================================================================

from pathlib import Path
import json
import hashlib
import os


# ==================================================================================================
# 2.1 — VERIFY / MOUNT GOOGLE DRIVE
# ==================================================================================================

print()
print("-" * 100)
print("GOOGLE DRIVE VERIFICATION")
print("-" * 100)

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_ROOT = DRIVE_ROOT / "MyDrive"

if not MYDRIVE_ROOT.exists():

    print("Google Drive is not currently mounted.")
    print("Attempting to mount Google Drive...")

    try:
        from google.colab import drive

        drive.mount(
            str(DRIVE_ROOT),
            force_remount=False
        )

    except Exception as exc:

        raise RuntimeError(
            "Google Drive could not be mounted automatically.\n"
            "Please mount Google Drive in the Colab runtime and rerun Section 2."
        ) from exc


if not MYDRIVE_ROOT.exists():

    raise FileNotFoundError(
        "Google Drive is not mounted or /content/drive/MyDrive is unavailable.\n"
        f"Expected path:\n  {MYDRIVE_ROOT}"
    )


if not MYDRIVE_ROOT.is_dir():

    raise NotADirectoryError(
        "Google Drive MyDrive path is not a directory.\n"
        f"Path:\n  {MYDRIVE_ROOT}"
    )


print(
    f"✓ Google Drive verified : {MYDRIVE_ROOT}"
)


# ==================================================================================================
# 2.2 — VERIFY CANONICAL PROJECT ROOT
# ==================================================================================================

print()
print("-" * 100)
print("CANONICAL PROJECT ROOT VERIFICATION")
print("-" * 100)

PROJECT_ROOT = (
    MYDRIVE_ROOT
    / "SPP_GAN_Research"
)


if not PROJECT_ROOT.exists():

    raise FileNotFoundError(
        "Canonical SPP-GAN research project root was not found.\n"
        f"Expected:\n  {PROJECT_ROOT}"
    )


if not PROJECT_ROOT.is_dir():

    raise NotADirectoryError(
        "Canonical SPP-GAN research project root is not a directory.\n"
        f"Expected:\n  {PROJECT_ROOT}"
    )


print(
    f"✓ Project root verified : {PROJECT_ROOT}"
)


# ==================================================================================================
# 2.3 — REPRODUCE NOTEBOOK 00 DIRECTORY CONSTRUCTION
# ==================================================================================================

print()
print("-" * 100)
print("NOTEBOOK 00 DIRECTORY CONSTRUCTION")
print("-" * 100)

RESULTS_ROOT = (
    PROJECT_ROOT
    / "results"
)

NOTEBOOK_RESULTS_ROOT = (
    RESULTS_ROOT
    / "notebooks"
)

NOTEBOOK_00_DIR = (
    NOTEBOOK_RESULTS_ROOT
    / "notebook_00"
)

NOTEBOOK_00_SUBDIRS = {

    "root":
        NOTEBOOK_00_DIR,

    "config":
        NOTEBOOK_00_DIR / "config",

    "environment":
        NOTEBOOK_00_DIR / "environment",

    "manifest":
        NOTEBOOK_00_DIR / "manifest",

    "logs":
        NOTEBOOK_00_DIR / "logs",
}


for key, path in NOTEBOOK_00_SUBDIRS.items():

    print(
        f"{key:<15} : {path}"
    )


# ==================================================================================================
# 2.4 — DEFINE CANONICAL NOTEBOOK 00 ARTIFACT PATHS
# ==================================================================================================

print()
print("-" * 100)
print("NOTEBOOK 00 ARTIFACT PATHS")
print("-" * 100)

NOTEBOOK_00_ARTIFACTS = {

    "config_json":
        NOTEBOOK_00_SUBDIRS["config"]
        / "experiment_config.json",

    "dataset_registry_json":
        NOTEBOOK_00_SUBDIRS["config"]
        / "dataset_registry.json",

    "model_registry_json":
        NOTEBOOK_00_SUBDIRS["config"]
        / "model_registry.json",

    "evaluation_config_json":
        NOTEBOOK_00_SUBDIRS["config"]
        / "evaluation_config.json",

    "privacy_config_json":
        NOTEBOOK_00_SUBDIRS["config"]
        / "privacy_config.json",

    "sppgan_config_json":
        NOTEBOOK_00_SUBDIRS["config"]
        / "sppgan_config.json",

    "environment_json":
        NOTEBOOK_00_SUBDIRS["environment"]
        / "environment.json",

    "manifest_json":
        NOTEBOOK_00_SUBDIRS["manifest"]
        / "notebook_00_manifest.json",

    "configuration_fingerprint_json":
        NOTEBOOK_00_SUBDIRS["manifest"]
        / "configuration_fingerprint.json",
}


for artifact_name, artifact_path in NOTEBOOK_00_ARTIFACTS.items():

    print(
        f"{artifact_name:<32} : {artifact_path}"
    )


# ==================================================================================================
# 2.5 — VERIFY NOTEBOOK 00 CANONICAL DIRECTORIES
# ==================================================================================================

print()
print("-" * 100)
print("NOTEBOOK 00 CANONICAL DIRECTORY VERIFICATION")
print("-" * 100)

DIRECTORY_LABELS = {

    "root":
        "Notebook 00 root",

    "config":
        "Config root",

    "environment":
        "Environment root",

    "manifest":
        "Manifest root",

    "logs":
        "Log root",
}


DIRECTORY_CHECKS = {}


for key, path in NOTEBOOK_00_SUBDIRS.items():

    exists = (
        path.exists()
        and path.is_dir()
    )

    DIRECTORY_CHECKS[key] = exists

    print(
        f"{'✓' if exists else '✗'} "
        f"{DIRECTORY_LABELS[key]:<24} : {path}"
    )


if not all(DIRECTORY_CHECKS.values()):

    missing_directories = [

        f"{DIRECTORY_LABELS[key]}: {path}"

        for key, path in NOTEBOOK_00_SUBDIRS.items()

        if not DIRECTORY_CHECKS[key]
    ]

    print()
    print(
        "✗ Canonical Notebook 00 directory "
        "structure is incomplete."
    )

    print()
    print("Missing directories:")

    for item in missing_directories:

        print(
            f"  - {item}"
        )

    print()
    print(
        "Notebook 05 will NOT create or reconstruct "
        "Notebook 00 directories."
    )

    raise FileNotFoundError(
        "Canonical Notebook 00 directory structure is incomplete."
    )


print()
print(
    "✓ Canonical Notebook 00 directory structure verified."
)


# ==================================================================================================
# 2.6 — VERIFY NOTEBOOK 00 MANIFEST LOCATION
# ==================================================================================================

print()
print("-" * 100)
print("NOTEBOOK 00 MANIFEST LOCATION VERIFICATION")
print("-" * 100)

CANONICAL_NOTEBOOK_00_MANIFEST = (
    NOTEBOOK_00_ARTIFACTS["manifest_json"]
)


if not CANONICAL_NOTEBOOK_00_MANIFEST.is_file():

    print(
        "✗ Canonical Notebook 00 manifest is missing."
    )

    print()
    print("Expected location:")

    print(
        f"  {CANONICAL_NOTEBOOK_00_MANIFEST}"
    )

    manifest_candidates = sorted(

        {
            candidate

            for candidate in PROJECT_ROOT.rglob(
                "notebook_00_manifest.json"
            )

            if candidate.is_file()
        },

        key=lambda p: str(p).lower(),
    )

    print()
    print(
        "Diagnostic search for existing Notebook 00 "
        "manifests under the canonical project root:"
    )

    print(
        f"Candidates found : {len(manifest_candidates)}"
    )

    for candidate in manifest_candidates:

        print(
            f"  - {candidate}"
        )

    print()
    print(
        "Notebook 05 will NOT silently substitute "
        "an alternate Notebook 00 manifest."
    )

    raise FileNotFoundError(
        "Canonical Notebook 00 manifest is missing.\n\n"
        "Expected:\n"
        f"  {CANONICAL_NOTEBOOK_00_MANIFEST}"
    )


print(
    "✓ Canonical Notebook 00 manifest verified:"
)

print(
    f"  {CANONICAL_NOTEBOOK_00_MANIFEST}"
)


# ==================================================================================================
# 2.7 — VERIFY ALL NOTEBOOK 00 ARTIFACT FILES
# ==================================================================================================

print()
print("-" * 100)
print("NOTEBOOK 00 ARTIFACT FILE VERIFICATION")
print("-" * 100)

MISSING_ARTIFACTS = []
EMPTY_ARTIFACTS = []

ARTIFACT_FILE_CHECKS = {}


for artifact_name, artifact_path in NOTEBOOK_00_ARTIFACTS.items():

    exists = artifact_path.is_file()

    nonempty = (
        exists
        and artifact_path.stat().st_size > 0
    )

    passed = (
        exists
        and nonempty
    )

    ARTIFACT_FILE_CHECKS[
        artifact_name
    ] = passed

    if passed:

        print(
            f"✓ {artifact_name:<32} : "
            f"{artifact_path}"
        )

    elif exists:

        print(
            f"✗ {artifact_name:<32} : "
            f"{artifact_path} [0 bytes]"
        )

        EMPTY_ARTIFACTS.append(
            artifact_path
        )

    else:

        print(
            f"✗ {artifact_name:<32} : "
            f"{artifact_path}"
        )

        MISSING_ARTIFACTS.append(
            artifact_path
        )


if MISSING_ARTIFACTS or EMPTY_ARTIFACTS:

    print()
    print(
        "✗ Notebook 00 artifact verification failed."
    )

    if MISSING_ARTIFACTS:

        print()
        print("Missing artifacts:")

        for path in MISSING_ARTIFACTS:

            print(
                f"  - {path}"
            )

    if EMPTY_ARTIFACTS:

        print()
        print("Empty artifacts:")

        for path in EMPTY_ARTIFACTS:

            print(
                f"  - {path}"
            )

    print()
    print(
        "Notebook 05 will NOT reconstruct "
        "Notebook 00 artifacts."
    )

    raise FileNotFoundError(
        "Canonical Notebook 00 artifact set is incomplete "
        "or contains zero-byte files."
    )


print()
print(
    "✓ All Notebook 00 artifacts are present and non-empty."
)


# ==================================================================================================
# 2.8 — JSON LOADER
# ==================================================================================================

def load_json_artifact(
    path: Path,
    artifact_name: str,
):
    """
    Load and validate a persisted Notebook 00 JSON artifact.
    """

    if not path.is_file():

        raise FileNotFoundError(
            f"Notebook 00 artifact '{artifact_name}' "
            "was not found:\n"
            f"  {path}"
        )

    try:

        with path.open(
            "r",
            encoding="utf-8",
        ) as handle:

            data = json.load(handle)

    except json.JSONDecodeError as exc:

        raise ValueError(
            f"Invalid JSON in Notebook 00 artifact "
            f"'{artifact_name}':\n"
            f"  {path}"
        ) from exc

    except OSError as exc:

        raise OSError(
            f"Unable to read Notebook 00 artifact "
            f"'{artifact_name}':\n"
            f"  {path}"
        ) from exc

    if not isinstance(data, dict):

        raise TypeError(
            f"Notebook 00 artifact '{artifact_name}' "
            "must contain a JSON object/dictionary.\n"
            f"Path: {path}\n"
            f"Type: {type(data).__name__}"
        )

    return data


# ==================================================================================================
# 2.9 — LOAD NOTEBOOK 00 ARTIFACTS
# ==================================================================================================

print()
print("-" * 100)
print("LOAD NOTEBOOK 00 CONFIGURATION ARTIFACTS")
print("-" * 100)

NB00_CONFIG = load_json_artifact(
    NOTEBOOK_00_ARTIFACTS["config_json"],
    "config_json",
)

NB00_DATASET_REGISTRY = load_json_artifact(
    NOTEBOOK_00_ARTIFACTS["dataset_registry_json"],
    "dataset_registry_json",
)

NB00_MODEL_REGISTRY = load_json_artifact(
    NOTEBOOK_00_ARTIFACTS["model_registry_json"],
    "model_registry_json",
)

NB00_EVALUATION_CONFIG = load_json_artifact(
    NOTEBOOK_00_ARTIFACTS["evaluation_config_json"],
    "evaluation_config_json",
)

NB00_PRIVACY_CONFIG = load_json_artifact(
    NOTEBOOK_00_ARTIFACTS["privacy_config_json"],
    "privacy_config_json",
)

NB00_SPPGAN_CONFIG = load_json_artifact(
    NOTEBOOK_00_ARTIFACTS["sppgan_config_json"],
    "sppgan_config_json",
)

NB00_ENVIRONMENT = load_json_artifact(
    NOTEBOOK_00_ARTIFACTS["environment_json"],
    "environment_json",
)

NB00_MANIFEST = load_json_artifact(
    NOTEBOOK_00_ARTIFACTS["manifest_json"],
    "manifest_json"
)


print()
print(
    "✓ Notebook 00 configuration artifacts "
    "loaded successfully."
)


# ==================================================================================================
# 2.10 — VERIFY NOTEBOOK 00 MANIFEST STRUCTURE
# ==================================================================================================

print()
print("-" * 100)
print("NOTEBOOK 00 MANIFEST STRUCTURE VERIFICATION")
print("-" * 100)

REQUIRED_MANIFEST_KEYS = {

    "manifest_version",
    "project",
    "environment",
    "configuration",
    "artifact_paths",
}


MISSING_MANIFEST_KEYS = (
    REQUIRED_MANIFEST_KEYS
    - set(NB00_MANIFEST.keys())
)


MANIFEST_STRUCTURE_PASS = (
    len(MISSING_MANIFEST_KEYS) == 0
)


if not MANIFEST_STRUCTURE_PASS:

    print(
        "✗ Notebook 00 manifest structure "
        "verification failed."
    )

    print()
    print("Missing required keys:")

    for key in sorted(
        MISSING_MANIFEST_KEYS
    ):

        print(
            f"  - {key}"
        )

    raise ValueError(
        "Notebook 00 manifest is missing required keys."
    )


print(
    "✓ Notebook 00 manifest structure verified."
)


# ==================================================================================================
# 2.11 — EXTRACT AUTHORITATIVE MANIFEST CONFIGURATION
# ==================================================================================================

MANIFEST_CONFIGURATION = (
    NB00_MANIFEST["configuration"]
)


if not isinstance(
    MANIFEST_CONFIGURATION,
    dict,
):

    raise TypeError(
        "Notebook 00 manifest['configuration'] "
        "must be a dictionary."
    )


print(
    "✓ Authoritative manifest configuration extracted."
)


# ==================================================================================================
# 2.12 — CANONICAL JSON SERIALIZATION
# ==================================================================================================

def canonical_json_bytes(obj):
    """
    Reproduce the Notebook 00 canonical JSON serialization
    used for configuration fingerprinting.

    Notebook 00 fingerprint metadata specifies:
        format       = JSON
        encoding     = UTF-8
        sort_keys    = True
        serializer   = str
    """

    return json.dumps(
        obj,
        sort_keys=True,
        default=str,
        ensure_ascii=False,
        separators=(",", ":"),
    ).encode("UTF-8")


# ==================================================================================================
# 2.13 — VERIFY CONFIGURATION FINGERPRINT
# ==================================================================================================

print()
print("-" * 100)
print("NOTEBOOK 00 CONFIGURATION FINGERPRINT")
print("-" * 100)

CONFIGURATION_FINGERPRINT_PATH = (
    NOTEBOOK_00_ARTIFACTS[
        "configuration_fingerprint_json"
    ]
)


NB00_FINGERPRINT = load_json_artifact(
    CONFIGURATION_FINGERPRINT_PATH,
    "configuration_fingerprint_json",
)


# --------------------------------------------------------------------------------------------------
# Verify fingerprint metadata
# --------------------------------------------------------------------------------------------------

FINGERPRINT_ALGORITHM = (
    NB00_FINGERPRINT.get("algorithm")
)

FINGERPRINT_VERSION = (
    NB00_FINGERPRINT.get("fingerprint_version")
)

FINGERPRINT_OBJECT = (
    NB00_FINGERPRINT.get("fingerprinted_object")
)

FINGERPRINT_HASH = (
    NB00_FINGERPRINT.get("hash")
)


if FINGERPRINT_ALGORITHM != "SHA256":

    raise ValueError(
        "Unsupported or unexpected Notebook 00 "
        "fingerprint algorithm.\n"
        "Expected: SHA256\n"
        f"Found   : {FINGERPRINT_ALGORITHM}"
    )


if FINGERPRINT_VERSION != "1.0":

    raise ValueError(
        "Unexpected Notebook 00 fingerprint version.\n"
        "Expected: 1.0\n"
        f"Found   : {FINGERPRINT_VERSION}"
    )


if FINGERPRINT_OBJECT != "CONFIG_SNAPSHOT":

    raise ValueError(
        "Unexpected Notebook 00 fingerprinted object.\n"
        "Expected: CONFIG_SNAPSHOT\n"
        f"Found   : {FINGERPRINT_OBJECT}"
    )


if FINGERPRINT_HASH is None:

    raise KeyError(
        "Notebook 00 configuration fingerprint "
        "artifact does not contain the required 'hash' field."
    )


if not isinstance(
    FINGERPRINT_HASH,
    str,
):

    raise TypeError(
        "Notebook 00 configuration fingerprint "
        "'hash' must be a string."
    )


EXPECTED_FINGERPRINT = hashlib.sha256(
    canonical_json_bytes(
        MANIFEST_CONFIGURATION
    )
).hexdigest()


PERSISTED_FINGERPRINT = (
    FINGERPRINT_HASH
)


CONFIGURATION_FINGERPRINT_PASS = (
    PERSISTED_FINGERPRINT.lower()
    == EXPECTED_FINGERPRINT.lower()
)


print(
    f"Algorithm             : "
    f"{FINGERPRINT_ALGORITHM}"
)

print(
    f"Fingerprint version   : "
    f"{FINGERPRINT_VERSION}"
)

print(
    f"Fingerprinted object  : "
    f"{FINGERPRINT_OBJECT}"
)

print(
    f"Expected fingerprint  : "
    f"{EXPECTED_FINGERPRINT}"
)

print(
    f"Persisted fingerprint : "
    f"{PERSISTED_FINGERPRINT}"
)

print(
    "Fingerprint status    : "
    + (
        "PASS"
        if CONFIGURATION_FINGERPRINT_PASS
        else "FAIL"
    )
)


if not CONFIGURATION_FINGERPRINT_PASS:

    raise ValueError(
        "Notebook 00 configuration fingerprint "
        "verification failed."
    )


print(
    "✓ Configuration fingerprint verified."
)


# ==================================================================================================
# 2.14 — EXTRACT AND VERIFY MASTER SEED
# ==================================================================================================

print()
print("-" * 100)
print("MASTER SEED VERIFICATION")
print("-" * 100)

# --------------------------------------------------------------------------------------------------
# Notebook 00 canonical reproducibility schema:
#
# seed_policy
# ├── deterministic
# ├── master_seed
# ├── repetition_seed_offset
# ├── repetition_seeds
# └── repetitions
# --------------------------------------------------------------------------------------------------

SEED_POLICY = NB00_CONFIG.get(
    "seed_policy"
)


if not isinstance(
    SEED_POLICY,
    dict,
):

    raise TypeError(
        "Notebook 00 configuration does not contain "
        "a valid 'seed_policy' dictionary."
    )


MASTER_SEED = SEED_POLICY.get(
    "master_seed"
)


if MASTER_SEED is None:

    raise KeyError(
        "Notebook 00 configuration does not contain "
        "'seed_policy.master_seed'."
    )


MASTER_SEED = int(
    MASTER_SEED
)


DETERMINISTIC_EXECUTION = bool(
    SEED_POLICY.get(
        "deterministic",
        False
    )
)


REPETITION_SEED_OFFSET = SEED_POLICY.get(
    "repetition_seed_offset"
)


if REPETITION_SEED_OFFSET is None:

    raise KeyError(
        "Notebook 00 configuration does not contain "
        "'seed_policy.repetition_seed_offset'."
    )


REPETITION_SEED_OFFSET = int(
    REPETITION_SEED_OFFSET
)


REPETITION_SEEDS = SEED_POLICY.get(
    "repetition_seeds"
)


if not isinstance(
    REPETITION_SEEDS,
    dict,
):

    raise TypeError(
        "Notebook 00 'seed_policy.repetition_seeds' "
        "must be a dictionary."
    )


if not REPETITION_SEEDS:

    raise ValueError(
        "Notebook 00 repetition seed registry is empty."
    )


TOP_LEVEL_REPETITIONS = NB00_CONFIG.get(
    "repetitions"
)


if TOP_LEVEL_REPETITIONS is None:

    raise KeyError(
        "Notebook 00 configuration does not contain "
        "the required top-level 'repetitions' field."
    )


TOP_LEVEL_REPETITIONS = int(
    TOP_LEVEL_REPETITIONS
)


SEED_POLICY_REPETITIONS = SEED_POLICY.get(
    "repetitions"
)


if SEED_POLICY_REPETITIONS is None:

    raise KeyError(
        "Notebook 00 seed policy does not contain "
        "'seed_policy.repetitions'."
    )


SEED_POLICY_REPETITIONS = int(
    SEED_POLICY_REPETITIONS
)


if TOP_LEVEL_REPETITIONS != SEED_POLICY_REPETITIONS:

    raise ValueError(
        "Notebook 00 repetition configuration is inconsistent.\n"
        f"Top-level repetitions   : {TOP_LEVEL_REPETITIONS}\n"
        f"Seed-policy repetitions : {SEED_POLICY_REPETITIONS}"
    )


if len(REPETITION_SEEDS) != TOP_LEVEL_REPETITIONS:

    raise ValueError(
        "Notebook 00 repetition seed registry is inconsistent "
        "with the configured repetition count.\n"
        f"Configured repetitions  : {TOP_LEVEL_REPETITIONS}\n"
        f"Seed entries            : {len(REPETITION_SEEDS)}"
    )


if not DETERMINISTIC_EXECUTION:

    raise ValueError(
        "Notebook 00 requires deterministic execution, "
        "but 'seed_policy.deterministic' is False."
    )


NORMALIZED_REPETITION_SEEDS = {}

for repetition_key, seed_value in REPETITION_SEEDS.items():

    repetition_number = int(
        repetition_key
    )

    normalized_seed = int(
        seed_value
    )

    NORMALIZED_REPETITION_SEEDS[
        repetition_number
    ] = normalized_seed


EXPECTED_REPETITION_NUMBERS = set(
    range(
        1,
        TOP_LEVEL_REPETITIONS + 1
    )
)


ACTUAL_REPETITION_NUMBERS = set(
    NORMALIZED_REPETITION_SEEDS.keys()
)


if ACTUAL_REPETITION_NUMBERS != EXPECTED_REPETITION_NUMBERS:

    raise ValueError(
        "Notebook 00 repetition seed registry does not "
        "contain exactly the expected repetition numbers.\n"
        f"Expected : {sorted(EXPECTED_REPETITION_NUMBERS)}\n"
        f"Found    : {sorted(ACTUAL_REPETITION_NUMBERS)}"
    )


EXPECTED_REPETITION_SEEDS = {

    repetition_number:
        MASTER_SEED
        + REPETITION_SEED_OFFSET
        + repetition_number

    for repetition_number in EXPECTED_REPETITION_NUMBERS
}


if NORMALIZED_REPETITION_SEEDS != EXPECTED_REPETITION_SEEDS:

    raise ValueError(
        "Notebook 00 repetition seeds do not match the "
        "frozen master-seed / repetition-offset policy.\n"
        f"Master seed            : {MASTER_SEED}\n"
        f"Repetition offset      : {REPETITION_SEED_OFFSET}\n"
        f"Expected seeds         : {EXPECTED_REPETITION_SEEDS}\n"
        f"Persisted seeds        : {NORMALIZED_REPETITION_SEEDS}"
    )


FROZEN_MASTER_SEED = 2025

MASTER_SEED_PASS = (
    MASTER_SEED == FROZEN_MASTER_SEED
)


if not MASTER_SEED_PASS:

    raise ValueError(
        "Notebook 00 master seed does not match "
        "the frozen research seed.\n"
        f"Expected: {FROZEN_MASTER_SEED}\n"
        f"Found   : {MASTER_SEED}"
    )


print(
    f"Master seed             : {MASTER_SEED}"
)

print(
    f"Deterministic execution : {DETERMINISTIC_EXECUTION}"
)

print(
    f"Repetition seed offset  : {REPETITION_SEED_OFFSET}"
)

print(
    f"Configured repetitions  : {TOP_LEVEL_REPETITIONS}"
)

print(
    "Repetition seeds        :"
)

for repetition_number in sorted(
    NORMALIZED_REPETITION_SEEDS
):

    print(
        f"  Repetition {repetition_number} "
        f"→ seed "
        f"{NORMALIZED_REPETITION_SEEDS[repetition_number]}"
    )


print(
    "✓ Master seed verified."
)

print(
    "✓ Deterministic execution policy verified."
)

print(
    "✓ Repetition seed registry verified."
)


# ==================================================================================================
# 2.15 — VERIFY TVAE REGISTRATION
# ==================================================================================================

print()
print("-" * 100)
print("TVAE REGISTRATION VERIFICATION")
print("-" * 100)

TVAE_REGISTERED = (
    "tvae"
    in NB00_MODEL_REGISTRY
)


if not TVAE_REGISTERED:

    raise KeyError(
        "TVAE is not registered in the "
        "Notebook 00 model registry."
    )


TVAE_CONFIG = (
    NB00_MODEL_REGISTRY["tvae"]
)


if not isinstance(
    TVAE_CONFIG,
    dict,
):

    raise TypeError(
        "Notebook 00 model registry entry "
        "'tvae' must be a dictionary."
    )


print(
    "✓ TVAE registration verified."
)


# ==================================================================================================
# 2.16 — VERIFY DATASET REGISTRY AND TARGETS
# ==================================================================================================

print()
print("-" * 100)
print("DATASET REGISTRY VERIFICATION")
print("-" * 100)

# --------------------------------------------------------------------------------------------------
# Notebook 00 canonical dataset registry schema:
#
# adult_income:
#     target_column: "income"
#
# bank_marketing:
#     target_column: "y"
#
# diabetes_130us:
#     target_column: "readmitted"
#
# Therefore Notebook 05 must consume:
#
#     dataset_cfg["target_column"]
#
# and NOT dataset_cfg["target"].
# --------------------------------------------------------------------------------------------------

EXPECTED_DATASETS = {

    "adult_income":
        "income",

    "bank_marketing":
        "y",

    "diabetes_130us":
        "readmitted",
}


DATASET_REGISTRY_PASS = True


for dataset_id, expected_target in (
    EXPECTED_DATASETS.items()
):

    if dataset_id not in NB00_DATASET_REGISTRY:

        DATASET_REGISTRY_PASS = False

        raise KeyError(
            f"Dataset '{dataset_id}' is missing "
            "from Notebook 00 registry."
        )


    dataset_cfg = (
        NB00_DATASET_REGISTRY[
            dataset_id
        ]
    )


    if not isinstance(
        dataset_cfg,
        dict,
    ):

        DATASET_REGISTRY_PASS = False

        raise TypeError(
            f"Notebook 00 dataset registry entry "
            f"'{dataset_id}' must be a dictionary."
        )


    target = dataset_cfg.get(
        "target_column"
    )


    if target is None:

        DATASET_REGISTRY_PASS = False

        raise KeyError(
            f"Notebook 00 dataset registry entry "
            f"'{dataset_id}' does not contain "
            "the required 'target_column' field."
        )


    if target != expected_target:

        DATASET_REGISTRY_PASS = False

        raise ValueError(
            f"Target mismatch for '{dataset_id}'.\n"
            f"Expected: {expected_target}\n"
            f"Found   : {target}"
        )


    print(
        f"✓ {dataset_id:<20} target = {target}"
    )


print()
print(
    "✓ Dataset registry verified."
)

print(
    "✓ Dataset targets verified."
)


# ==================================================================================================
# 2.17 — TVAE TRAIN-ONLY POLICY
# ==================================================================================================

print()
print("-" * 100)
print("TVAE TRAIN-ONLY POLICY")
print("-" * 100)

TVAE_TRAIN_ONLY_POLICY = {

    "fit_split":
        "train",

    "validation_split_used_for_fitting":
        False,

    "test_split_used_for_fitting":
        False,
}


TVAE_TRAIN_ONLY_POLICY_PASS = (

    TVAE_TRAIN_ONLY_POLICY[
        "fit_split"
    ]
    == "train"

    and

    TVAE_TRAIN_ONLY_POLICY[
        "validation_split_used_for_fitting"
    ]
    is False

    and

    TVAE_TRAIN_ONLY_POLICY[
        "test_split_used_for_fitting"
    ]
    is False
)


if not TVAE_TRAIN_ONLY_POLICY_PASS:

    raise RuntimeError(
        "TVAE TRAIN-only fitting policy "
        "verification failed."
    )


print(
    "✓ TVAE TRAIN-only fitting policy established."
)

print(
    "✓ Validation split excluded from TVAE fitting."
)

print(
    "✓ Test split excluded from TVAE fitting."
)


# ==================================================================================================
# 2.18 — VERIFY EXPERIMENTAL REPETITIONS
# ==================================================================================================

print()
print("-" * 100)
print("EXPERIMENTAL REPETITIONS")
print("-" * 100)

EXPERIMENTAL_REPETITIONS = int(
    NB00_CONFIG["repetitions"]
)


EXPERIMENTAL_REPETITIONS_PASS = (
    EXPERIMENTAL_REPETITIONS == 5
)


if not EXPERIMENTAL_REPETITIONS_PASS:

    raise ValueError(
        "Experimental repetitions do not match "
        "the frozen configuration.\n"
        "Expected: 5\n"
        f"Found   : {EXPERIMENTAL_REPETITIONS}"
    )


print(
    f"✓ Experimental repetitions verified : "
    f"{EXPERIMENTAL_REPETITIONS}"
)


# ==================================================================================================
# 2.19 — CONFIGURATION LOAD SUMMARY
# ==================================================================================================

print()
print("-" * 100)
print("CONFIGURATION LOAD SUMMARY")
print("-" * 100)

print(
    f"Project root             : {PROJECT_ROOT}"
)

print(
    f"Notebook 00 root         : {NOTEBOOK_00_DIR}"
)

print(
    f"Master seed              : {MASTER_SEED}"
)

print(
    f"Experimental repetitions : "
    f"{EXPERIMENTAL_REPETITIONS}"
)

print(
    f"TVAE registered          : "
    f"{TVAE_REGISTERED}"
)

print(
    f"Datasets registered      : "
    f"{len(EXPECTED_DATASETS)}"
)

print(
    f"Fingerprint verified     : "
    f"{CONFIGURATION_FINGERPRINT_PASS}"
)

print(
    f"TRAIN-only policy        : "
    f"{TVAE_TRAIN_ONLY_POLICY_PASS}"
)

print(
    f"Deterministic execution  : "
    f"{DETERMINISTIC_EXECUTION}"
)

print(
    f"Repetition seeds verified: "
    f"{NORMALIZED_REPETITION_SEEDS == EXPECTED_REPETITION_SEEDS}"
)


# ==================================================================================================
# 2.20 — SECTION 2 COMPLETION GATE
# ==================================================================================================

print()
print("=" * 100)
print("SECTION 2 — FINAL VERIFICATION")
print("=" * 100)

SECTION_02_CHECKS = {

    "project_root":
        PROJECT_ROOT.is_dir(),

    "notebook_00_root":
        NOTEBOOK_00_SUBDIRS[
            "root"
        ].is_dir(),

    "config_root":
        NOTEBOOK_00_SUBDIRS[
            "config"
        ].is_dir(),

    "environment_root":
        NOTEBOOK_00_SUBDIRS[
            "environment"
        ].is_dir(),

    "manifest_root":
        NOTEBOOK_00_SUBDIRS[
            "manifest"
        ].is_dir(),

    "log_root":
        NOTEBOOK_00_SUBDIRS[
            "logs"
        ].is_dir(),

    "required_artifacts":
        all(
            path.is_file()
            and
            path.stat().st_size > 0
            for path in NOTEBOOK_00_ARTIFACTS.values()
        ),

    "manifest_structure":
        MANIFEST_STRUCTURE_PASS,

    "configuration_fingerprint":
        CONFIGURATION_FINGERPRINT_PASS,

    "master_seed":
        MASTER_SEED_PASS,

    "deterministic_execution":
        DETERMINISTIC_EXECUTION,

    "repetition_seed_registry":
        NORMALIZED_REPETITION_SEEDS
        == EXPECTED_REPETITION_SEEDS,

    "tvae_registered":
        TVAE_REGISTERED,

    "dataset_registry":
        DATASET_REGISTRY_PASS,

    "tvae_train_only_policy":
        TVAE_TRAIN_ONLY_POLICY_PASS,

    "experimental_repetitions":
        EXPERIMENTAL_REPETITIONS_PASS,
}


PASSED_SECTION_02 = sum(
    bool(value)
    for value in SECTION_02_CHECKS.values()
)


TOTAL_SECTION_02 = len(
    SECTION_02_CHECKS
)


FAILED_SECTION_02 = (
    TOTAL_SECTION_02
    - PASSED_SECTION_02
)


print(
    f"Total checks  : {TOTAL_SECTION_02}"
)

print(
    f"Passed checks : {PASSED_SECTION_02}"
)

print(
    f"Failed checks : {FAILED_SECTION_02}"
)

print()

for check_name, passed in (
    SECTION_02_CHECKS.items()
):

    print(
        f"{'✓' if passed else '✗'} "
        f"{check_name}"
    )


if FAILED_SECTION_02 > 0:

    raise RuntimeError(
        "SECTION 2 — LOAD CONFIGURATION FAILED."
    )


# ==================================================================================================
# 2.21 — FINAL STATUS
# ==================================================================================================

print()
print("=" * 100)
print("SECTION 2 STATUS: COMPLETE / PASS")
print("=" * 100)

print(
    "✓ Google Drive verified"
)

print(
    "✓ Canonical project root verified"
)

print(
    "✓ Notebook 00 canonical directories verified"
)

print(
    "✓ Notebook 00 artifacts verified"
)

print(
    "✓ Notebook 00 configuration loaded"
)

print(
    "✓ Notebook 00 manifest structure verified"
)

print(
    "✓ Configuration fingerprint verified"
)

print(
    "✓ Master seed verified"
)

print(
    "✓ Deterministic execution policy verified"
)

print(
    "✓ Repetition seed registry verified"
)

print(
    "✓ Dataset registry verified"
)

print(
    "✓ Dataset targets verified"
)

print(
    "✓ TVAE registration verified"
)

print(
    "✓ TVAE TRAIN-only policy established"
)

print(
    "✓ Experimental repetitions verified"
)

print(
    "✓ Section 2 completion gate passed"
)

SECTION 2 — LOAD CONFIGURATION

----------------------------------------------------------------------------------------------------
GOOGLE DRIVE VERIFICATION
----------------------------------------------------------------------------------------------------
Google Drive is not currently mounted.
Attempting to mount Google Drive...
Mounted at /content/drive
✓ Google Drive verified : /content/drive/MyDrive

----------------------------------------------------------------------------------------------------
CANONICAL PROJECT ROOT VERIFICATION
----------------------------------------------------------------------------------------------------
✓ Project root verified : /content/drive/MyDrive/SPP_GAN_Research

----------------------------------------------------------------------------------------------------
NOTEBOOK 00 DIRECTORY CONSTRUCTION
----------------------------------------------------------------------------------------------------
root            : /content/drive/MyDrive/SPP_GA

In [6]:
# ==================================================================================================
# SECTION 3 — LOAD TRAINING DATA
# ==================================================================================================

print("=" * 100)
print("SECTION 3 — LOAD TRAINING DATA")
print("=" * 100)

from pathlib import Path
import hashlib
import json
import pandas as pd
import numpy as np


# --------------------------------------------------------------------------------------------------
# 1. GOOGLE DRIVE
# --------------------------------------------------------------------------------------------------

try:
    from google.colab import drive

    DRIVE_ROOT = Path("/content/drive")
    MYDRIVE_ROOT = DRIVE_ROOT / "MyDrive"

    if not (
        DRIVE_ROOT.exists()
        and MYDRIVE_ROOT.exists()
    ):
        drive.mount(
            "/content/drive",
            force_remount=False,
        )

except ImportError:
    pass


# --------------------------------------------------------------------------------------------------
# 2. VERIFY CANONICAL PROJECT ROOT
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research"
)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        "Canonical project root not found:\n"
        f"{PROJECT_ROOT}"
    )

if not PROJECT_ROOT.is_dir():
    raise RuntimeError(
        "Canonical project root is not a directory:\n"
        f"{PROJECT_ROOT}"
    )

print(
    f"✓ Project root verified : {PROJECT_ROOT}"
)


# ==================================================================================================
# 3. LOAD PERSISTED NOTEBOOK 00 CONFIGURATION
# ==================================================================================================

NB00_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "notebook_00"
)

NB00_CONFIG_ROOT = (
    NB00_ROOT
    / "config"
)

NB00_MANIFEST_ROOT = (
    NB00_ROOT
    / "manifest"
)


# --------------------------------------------------------------------------------------------------
# Notebook 00 canonical artifacts required by Section 3
# --------------------------------------------------------------------------------------------------

NB00_CONFIG_PATH = (
    NB00_CONFIG_ROOT
    / "experiment_config.json"
)

NB00_DATASET_REGISTRY_PATH = (
    NB00_CONFIG_ROOT
    / "dataset_registry.json"
)

NB00_MANIFEST_PATH = (
    NB00_MANIFEST_ROOT
    / "notebook_00_manifest.json"
)

NB00_FINGERPRINT_PATH = (
    NB00_MANIFEST_ROOT
    / "configuration_fingerprint.json"
)


REQUIRED_NB00_ARTIFACTS = {
    "experiment_config": NB00_CONFIG_PATH,
    "dataset_registry": NB00_DATASET_REGISTRY_PATH,
    "manifest": NB00_MANIFEST_PATH,
    "configuration_fingerprint": NB00_FINGERPRINT_PATH,
}


# --------------------------------------------------------------------------------------------------
# Verify Notebook 00 artifacts
# --------------------------------------------------------------------------------------------------

for artifact_name, artifact_path in REQUIRED_NB00_ARTIFACTS.items():

    if not artifact_path.exists():
        raise FileNotFoundError(
            f"Required Notebook 00 artifact not found:\n"
            f"Artifact : {artifact_name}\n"
            f"Path     : {artifact_path}"
        )

    if not artifact_path.is_file():
        raise RuntimeError(
            f"Notebook 00 artifact is not a file:\n"
            f"{artifact_path}"
        )

    if artifact_path.stat().st_size == 0:
        raise RuntimeError(
            f"Notebook 00 artifact is empty:\n"
            f"{artifact_path}"
        )


print(
    "✓ Required Notebook 00 artifacts verified."
)


# --------------------------------------------------------------------------------------------------
# JSON loader
# --------------------------------------------------------------------------------------------------

def load_json_artifact(path: Path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as file_handle:

        return json.load(
            file_handle
        )


# --------------------------------------------------------------------------------------------------
# Load authoritative Notebook 00 artifacts
# --------------------------------------------------------------------------------------------------

NB00_CONFIG = load_json_artifact(
    NB00_CONFIG_PATH
)

NB00_DATASET_REGISTRY = load_json_artifact(
    NB00_DATASET_REGISTRY_PATH
)

NB00_MANIFEST = load_json_artifact(
    NB00_MANIFEST_PATH
)

NB00_FINGERPRINT = load_json_artifact(
    NB00_FINGERPRINT_PATH
)


print(
    "✓ Notebook 00 configuration loaded."
)

print(
    "✓ Notebook 00 dataset registry loaded."
)


# --------------------------------------------------------------------------------------------------
# Validate top-level artifact types
# --------------------------------------------------------------------------------------------------

if not isinstance(
    NB00_CONFIG,
    dict,
):
    raise RuntimeError(
        "Notebook 00 experiment configuration "
        "must be a JSON object."
    )

if not isinstance(
    NB00_DATASET_REGISTRY,
    dict,
):
    raise RuntimeError(
        "Notebook 00 dataset registry "
        "must be a JSON object."
    )

if not isinstance(
    NB00_MANIFEST,
    dict,
):
    raise RuntimeError(
        "Notebook 00 manifest "
        "must be a JSON object."
    )

if not isinstance(
    NB00_FINGERPRINT,
    dict,
):
    raise RuntimeError(
        "Notebook 00 configuration fingerprint "
        "must be a JSON object."
    )


# ==================================================================================================
# 4. RESOLVE DATASETS FROM NOTEBOOK 00
# ==================================================================================================

DATASET_IDS = list(
    NB00_CONFIG.get(
        "datasets",
        [],
    )
)

if not DATASET_IDS:

    raise RuntimeError(
        "Notebook 00 experiment configuration "
        "contains no registered datasets."
    )


# --------------------------------------------------------------------------------------------------
# Cross-validate configuration against registry
# --------------------------------------------------------------------------------------------------

CONFIG_DATASET_IDS = set(
    DATASET_IDS
)

REGISTRY_DATASET_IDS = set(
    NB00_DATASET_REGISTRY.keys()
)

MISSING_FROM_REGISTRY = sorted(
    CONFIG_DATASET_IDS
    - REGISTRY_DATASET_IDS
)

if MISSING_FROM_REGISTRY:

    raise RuntimeError(
        "Notebook 00 configuration contains datasets "
        "missing from the dataset registry:\n"
        f"{MISSING_FROM_REGISTRY}"
    )


print(
    f"✓ Authoritative datasets resolved : "
    f"{len(DATASET_IDS)}"
)

for dataset_id in DATASET_IDS:

    print(
        f"  • {dataset_id}"
    )


# ==================================================================================================
# 5. VALIDATE DATASET REGISTRY
# ==================================================================================================

EXPECTED_TARGETS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

EXPECTED_IDENTIFIERS = {
    "adult_income": [],
    "bank_marketing": [],
    "diabetes_130us": [
        "encounter_id",
        "patient_nbr",
    ],
}


for dataset_id in DATASET_IDS:

    dataset_cfg = NB00_DATASET_REGISTRY.get(
        dataset_id
    )

    if dataset_cfg is None:

        raise RuntimeError(
            f"{dataset_id}: missing from Notebook 00 "
            "dataset registry."
        )


    # ----------------------------------------------------------------------------------------------
    # Target
    # ----------------------------------------------------------------------------------------------

    target_column = dataset_cfg.get(
        "target_column"
    )

    if not target_column:

        raise RuntimeError(
            f"{dataset_id}: target_column is missing "
            "from Notebook 00 dataset registry."
        )


    expected_target = EXPECTED_TARGETS.get(
        dataset_id
    )

    if (
        expected_target is not None
        and target_column != expected_target
    ):

        raise RuntimeError(
            f"{dataset_id}: target mismatch.\n"
            f"Expected : {expected_target}\n"
            f"Found    : {target_column}"
        )


    # ----------------------------------------------------------------------------------------------
    # Explicit identifiers
    # ----------------------------------------------------------------------------------------------

    identifier_columns = list(
        dataset_cfg.get(
            "identifier_columns",
            [],
        )
    )


    expected_identifiers = EXPECTED_IDENTIFIERS.get(
        dataset_id
    )

    if (
        expected_identifiers is not None
        and identifier_columns != expected_identifiers
    ):

        raise RuntimeError(
            f"{dataset_id}: identifier registry mismatch.\n"
            f"Expected : {expected_identifiers}\n"
            f"Found    : {identifier_columns}"
        )


print(
    "✓ Notebook 00 dataset registry verified."
)

for dataset_id in DATASET_IDS:

    dataset_cfg = NB00_DATASET_REGISTRY[
        dataset_id
    ]

    identifiers = dataset_cfg.get(
        "identifier_columns",
        [],
    )

    print(
        f"✓ {dataset_id:<20} "
        f"target = {dataset_cfg['target_column']} | "
        f"identifiers = "
        f"{identifiers if identifiers else 'None'}"
    )


# ==================================================================================================
# 6. VERIFY TRAIN-ONLY TVAE POLICY
# ==================================================================================================

DATA_SPLIT_CONFIG = NB00_CONFIG.get(
    "data_split",
    {},
)

FIT_SPLIT = DATA_SPLIT_CONFIG.get(
    "fit_split"
)

VALIDATION_ROLE = DATA_SPLIT_CONFIG.get(
    "validation_role"
)

TEST_ROLE = DATA_SPLIT_CONFIG.get(
    "test_role"
)


if FIT_SPLIT != "train":

    raise RuntimeError(
        "Notebook 00 fit_split is not 'train'.\n"
        f"Found: {FIT_SPLIT}"
    )


if (
    VALIDATION_ROLE
    and str(
        VALIDATION_ROLE
    ).lower()
    not in {
        "downstream_evaluation",
        "validation",
    }
):

    raise RuntimeError(
        "Unexpected Notebook 00 validation role:\n"
        f"{VALIDATION_ROLE}"
    )


if (
    TEST_ROLE
    and str(
        TEST_ROLE
    ).lower()
    not in {
        "final_evaluation",
        "test",
    }
):

    raise RuntimeError(
        "Unexpected Notebook 00 test role:\n"
        f"{TEST_ROLE}"
    )


print(
    "✓ Notebook 00 TRAIN-only fitting policy verified."
)

print(
    f"✓ Fit split : {FIT_SPLIT}"
)


# ==================================================================================================
# 7. VERIFY CANONICAL NOTEBOOK 02 ROOT
# ==================================================================================================

NB02_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_02"
)

if not NB02_ROOT.exists():

    raise FileNotFoundError(
        "Canonical Notebook 02 processed-data root "
        "not found:\n"
        f"{NB02_ROOT}"
    )

if not NB02_ROOT.is_dir():

    raise RuntimeError(
        "Notebook 02 processed-data root "
        "is not a directory:\n"
        f"{NB02_ROOT}"
    )


print(
    f"✓ Notebook 02 root verified : {NB02_ROOT}"
)


# ==================================================================================================
# 8. VERIFY NOTEBOOK 02 NATIVE DIRECTORY
# ==================================================================================================

NB02_NATIVE_ROOT = (
    NB02_ROOT
    / "native"
)

if not NB02_NATIVE_ROOT.exists():

    raise FileNotFoundError(
        "Notebook 02 native directory not found:\n"
        f"{NB02_NATIVE_ROOT}"
    )

if not NB02_NATIVE_ROOT.is_dir():

    raise RuntimeError(
        "Notebook 02 native path is not a directory:\n"
        f"{NB02_NATIVE_ROOT}"
    )


print(
    f"✓ Native output directory verified : "
    f"{NB02_NATIVE_ROOT}"
)


# ==================================================================================================
# 9. VERIFY NATIVE DATASET MANIFEST
# ==================================================================================================

NB02_NATIVE_MANIFEST = (
    NB02_NATIVE_ROOT
    / "native_dataset_manifest.csv"
)

if not NB02_NATIVE_MANIFEST.exists():

    raise FileNotFoundError(
        "Notebook 02 native dataset manifest "
        "not found:\n"
        f"{NB02_NATIVE_MANIFEST}"
    )

if not NB02_NATIVE_MANIFEST.is_file():

    raise RuntimeError(
        "Notebook 02 native dataset manifest "
        "is not a file:\n"
        f"{NB02_NATIVE_MANIFEST}"
    )

if NB02_NATIVE_MANIFEST.stat().st_size == 0:

    raise RuntimeError(
        "Notebook 02 native dataset manifest is empty:\n"
        f"{NB02_NATIVE_MANIFEST}"
    )


print(
    f"✓ Native manifest verified        : "
    f"{NB02_NATIVE_MANIFEST}"
)


# ==================================================================================================
# 10. LOAD NATIVE MANIFEST
# ==================================================================================================

NATIVE_MANIFEST_DF = pd.read_csv(
    NB02_NATIVE_MANIFEST,
    low_memory=False,
)

if NATIVE_MANIFEST_DF.empty:

    raise RuntimeError(
        "Notebook 02 native dataset manifest "
        "contains no records."
    )


print(
    f"✓ Native manifest loaded          : "
    f"{len(NATIVE_MANIFEST_DF)} records"
)


# ==================================================================================================
# 11. VALIDATE NATIVE MANIFEST SCHEMA
# ==================================================================================================

REQUIRED_MANIFEST_COLUMNS = [
    "dataset_id",
    "split",
    "relative_path",
    "absolute_path",
    "rows",
    "columns",
    "preprocessing_feature_columns",
    "generative_columns",
    "target_column",
    "provenance_column",
    "identifier_columns",
    "provenance_present",
    "target_present",
    "identifiers_excluded",
    "file_exists",
    "file_size_bytes",
    "sha256",
    "reload_validation",
    "status",
]


MISSING_MANIFEST_COLUMNS = [
    column
    for column in REQUIRED_MANIFEST_COLUMNS
    if column not in NATIVE_MANIFEST_DF.columns
]


if MISSING_MANIFEST_COLUMNS:

    raise RuntimeError(
        "Notebook 02 native manifest is missing "
        "required fields:\n"
        f"{MISSING_MANIFEST_COLUMNS}"
    )


print(
    f"✓ Native manifest schema verified : "
    f"{len(REQUIRED_MANIFEST_COLUMNS)} required fields"
)


# ==================================================================================================
# 12. VALIDATE DATASET COVERAGE
# ==================================================================================================

MANIFEST_DATASET_IDS = sorted(
    NATIVE_MANIFEST_DF[
        "dataset_id"
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

EXPECTED_DATASET_IDS = sorted(
    DATASET_IDS
)


if MANIFEST_DATASET_IDS != EXPECTED_DATASET_IDS:

    raise RuntimeError(
        "Notebook 02 native manifest dataset coverage "
        "does not match Notebook 00.\n"
        f"Expected : {EXPECTED_DATASET_IDS}\n"
        f"Found    : {MANIFEST_DATASET_IDS}"
    )


print(
    f"✓ Dataset coverage verified       : "
    f"{len(DATASET_IDS)} datasets"
)


# ==================================================================================================
# 13. VALIDATE TRAIN COVERAGE
# ==================================================================================================

TRAIN_MANIFEST_DF = NATIVE_MANIFEST_DF[
    NATIVE_MANIFEST_DF[
        "split"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    == "train"
].copy()


if len(TRAIN_MANIFEST_DF) != len(DATASET_IDS):

    raise RuntimeError(
        "Notebook 02 TRAIN manifest coverage mismatch.\n"
        f"Expected : {len(DATASET_IDS)}\n"
        f"Found    : {len(TRAIN_MANIFEST_DF)}"
    )


print(
    f"✓ TRAIN manifest coverage verified: "
    f"{len(TRAIN_MANIFEST_DF)} records"
)


# ==================================================================================================
# 14. INITIALIZE TRAINING DATA CONTAINERS
# ==================================================================================================

TRAINING_DATA = {}

TRAINING_GENERATIVE_COLUMNS = {}

TRAINING_FEATURE_COLUMNS = {}

TRAINING_TARGET_COLUMNS = {}

TRAINING_PROVENANCE_COLUMNS = {}

TRAINING_IDENTIFIER_COLUMNS = {}

TRAINING_SOURCE_PATHS = {}

TRAINING_SOURCE_HASHES = {}


# ==================================================================================================
# 15. LOAD AND VALIDATE EACH TRAINING DATASET
# ==================================================================================================

for dataset_id in DATASET_IDS:

    print()
    print("-" * 100)
    print(
        f"Loading TRAIN dataset : {dataset_id}"
    )
    print("-" * 100)


    # ----------------------------------------------------------------------------------------------
    # 15.1 Get exactly one TRAIN manifest record
    # ----------------------------------------------------------------------------------------------

    dataset_train_records = TRAIN_MANIFEST_DF[
        TRAIN_MANIFEST_DF[
            "dataset_id"
        ]
        .astype(str)
        .str.strip()
        == str(dataset_id)
    ]


    if len(dataset_train_records) != 1:

        raise RuntimeError(
            f"{dataset_id}: expected exactly one "
            "TRAIN manifest record.\n"
            f"Found: {len(dataset_train_records)}"
        )


    record = dataset_train_records.iloc[0]


    # ----------------------------------------------------------------------------------------------
    # 15.2 Validate manifest status
    # ----------------------------------------------------------------------------------------------

    status_value = str(
        record["status"]
    ).strip().upper()


    if status_value != "PASS":

        raise RuntimeError(
            f"{dataset_id}: Notebook 02 TRAIN "
            "manifest status is not PASS.\n"
            f"Found: {record['status']}"
        )


    # ----------------------------------------------------------------------------------------------
    # 15.3 Validate boolean integrity flags
    # ----------------------------------------------------------------------------------------------

    BOOLEAN_TRUE_VALUES = {
        "true",
        "1",
        "yes",
    }


    for field in [
        "file_exists",
        "provenance_present",
        "target_present",
        "identifiers_excluded",
        "reload_validation",
    ]:

        value = str(
            record[field]
        ).strip().lower()


        if value not in BOOLEAN_TRUE_VALUES:

            raise RuntimeError(
                f"{dataset_id}: Notebook 02 TRAIN "
                f"{field} flag is not TRUE.\n"
                f"Found: {record[field]}"
            )


    # ----------------------------------------------------------------------------------------------
    # 15.4 Resolve persisted TRAIN path
    # ----------------------------------------------------------------------------------------------

    manifest_absolute_path = str(
        record["absolute_path"]
    ).strip()

    manifest_relative_path = str(
        record["relative_path"]
    ).strip()


    if (
        not manifest_absolute_path
        or manifest_absolute_path.lower() == "nan"
    ):

        raise RuntimeError(
            f"{dataset_id}: invalid absolute_path "
            "in Notebook 02 native manifest."
        )


    if (
        not manifest_relative_path
        or manifest_relative_path.lower() == "nan"
    ):

        raise RuntimeError(
            f"{dataset_id}: invalid relative_path "
            "in Notebook 02 native manifest."
        )


    train_path = Path(
        manifest_absolute_path
    )


    # ----------------------------------------------------------------------------------------------
    # Fallback to canonical relative path
    # ----------------------------------------------------------------------------------------------

    if not train_path.exists():

        train_path = (
            NB02_ROOT
            / manifest_relative_path
        )


    if not train_path.exists():

        raise FileNotFoundError(
            f"{dataset_id}: TRAIN file could not be resolved.\n\n"
            f"Manifest absolute path:\n"
            f"{manifest_absolute_path}\n\n"
            f"Manifest relative path:\n"
            f"{manifest_relative_path}\n\n"
            f"Canonical resolved path:\n"
            f"{NB02_ROOT / manifest_relative_path}"
        )


    if not train_path.is_file():

        raise RuntimeError(
            f"{dataset_id}: resolved TRAIN path "
            "is not a file:\n"
            f"{train_path}"
        )


    # ----------------------------------------------------------------------------------------------
    # 15.5 Verify file size
    # ----------------------------------------------------------------------------------------------

    expected_file_size = int(
        record["file_size_bytes"]
    )

    actual_file_size = int(
        train_path.stat().st_size
    )


    if actual_file_size != expected_file_size:

        raise RuntimeError(
            f"{dataset_id}: TRAIN file-size mismatch.\n"
            f"Manifest : {expected_file_size:,} bytes\n"
            f"Actual   : {actual_file_size:,} bytes"
        )


    # ----------------------------------------------------------------------------------------------
    # 15.6 Load TRAIN CSV
    # ----------------------------------------------------------------------------------------------

    train_df = pd.read_csv(
        train_path,
        low_memory=False,
    )


    if train_df.empty:

        raise RuntimeError(
            f"{dataset_id}: TRAIN dataset is empty."
        )


    # ----------------------------------------------------------------------------------------------
    # 15.7 Load authoritative Notebook 00 dataset configuration
    # ----------------------------------------------------------------------------------------------

    dataset_cfg = NB00_DATASET_REGISTRY[
        dataset_id
    ]


    target_column = dataset_cfg[
        "target_column"
    ]


    identifier_columns = list(
        dataset_cfg.get(
            "identifier_columns",
            [],
        )
    )


    # ----------------------------------------------------------------------------------------------
    # 15.8 Validate target column
    # ----------------------------------------------------------------------------------------------

    if target_column not in train_df.columns:

        raise RuntimeError(
            f"{dataset_id}: target column "
            f"'{target_column}' is missing from TRAIN data."
        )


    manifest_target = str(
        record["target_column"]
    ).strip()


    if manifest_target != target_column:

        raise RuntimeError(
            f"{dataset_id}: target mismatch.\n"
            f"Notebook 00 : {target_column}\n"
            f"Notebook 02 : {manifest_target}"
        )


    # ----------------------------------------------------------------------------------------------
    # 15.9 Validate provenance column
    # ----------------------------------------------------------------------------------------------

    provenance_column = str(
        record["provenance_column"]
    ).strip()


    if (
        not provenance_column
        or provenance_column.lower() == "nan"
    ):

        raise RuntimeError(
            f"{dataset_id}: invalid provenance column "
            "in Notebook 02 manifest."
        )


    if provenance_column not in train_df.columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column "
            f"'{provenance_column}' is missing from TRAIN data."
        )


    # ----------------------------------------------------------------------------------------------
    # 15.10 Validate explicit identifiers are excluded
    # ----------------------------------------------------------------------------------------------

    identifiers_present = [
        column
        for column in identifier_columns
        if column in train_df.columns
    ]


    if identifiers_present:

        raise RuntimeError(
            f"{dataset_id}: explicit identifier columns "
            "are present in TRAIN data:\n"
            f"{identifiers_present}"
        )


    # ----------------------------------------------------------------------------------------------
    # 15.11 Construct native generative schema
    #
    # Notebook 02 native files contain:
    #   - modeling/generative columns
    #   - target
    #   - provenance
    #
    # Explicit identifiers are excluded.
    #
    # The provenance column must NOT enter TVAE training.
    # ----------------------------------------------------------------------------------------------

    generative_columns = [
        column
        for column in train_df.columns
        if column != provenance_column
        and column not in identifier_columns
    ]


    feature_columns = [
        column
        for column in generative_columns
        if column != target_column
    ]


    # ----------------------------------------------------------------------------------------------
    # 15.12 Validate row and column counts
    # ----------------------------------------------------------------------------------------------

    expected_rows = int(
        record["rows"]
    )

    expected_columns = int(
        record["columns"]
    )


    actual_rows = len(
        train_df
    )

    actual_columns = len(
        train_df.columns
    )


    if actual_rows != expected_rows:

        raise RuntimeError(
            f"{dataset_id}: TRAIN row-count mismatch.\n"
            f"Manifest : {expected_rows:,}\n"
            f"Loaded   : {actual_rows:,}"
        )


    if actual_columns != expected_columns:

        raise RuntimeError(
            f"{dataset_id}: TRAIN column-count mismatch.\n"
            f"Manifest : {expected_columns}\n"
            f"Loaded   : {actual_columns}"
        )


    # ==================================================================================================
    # 15.13 VALIDATE MANIFEST COLUMN COUNTS
    #
    # IMPORTANT:
    # Notebook 02 stores these fields as COUNTS, not lists of column names.
    #
    #   preprocessing_feature_columns → integer count
    #   generative_columns            → integer count
    #
    # ==================================================================================================

    expected_manifest_generative_count = int(
        record["generative_columns"]
    )

    expected_manifest_feature_count = int(
        record["preprocessing_feature_columns"]
    )


    actual_generative_count = len(
        generative_columns
    )

    actual_feature_count = len(
        feature_columns
    )


    if (
        expected_manifest_generative_count
        != actual_generative_count
    ):

        raise RuntimeError(
            f"{dataset_id}: generative-column count mismatch.\n"
            f"Notebook 02 manifest : "
            f"{expected_manifest_generative_count}\n"
            f"Loaded native schema : "
            f"{actual_generative_count}"
        )


    if (
        expected_manifest_feature_count
        != actual_feature_count
    ):

        raise RuntimeError(
            f"{dataset_id}: preprocessing-feature count mismatch.\n"
            f"Notebook 02 manifest : "
            f"{expected_manifest_feature_count}\n"
            f"Loaded native schema : "
            f"{actual_feature_count}"
        )


    print(
        f"✓ Generative column count verified : "
        f"{actual_generative_count}"
    )

    print(
        f"✓ Feature column count verified    : "
        f"{actual_feature_count}"
    )


    # ----------------------------------------------------------------------------------------------
    # 15.14 Verify SHA-256
    # ----------------------------------------------------------------------------------------------

    expected_sha256 = str(
        record["sha256"]
    ).strip().lower()


    if (
        not expected_sha256
        or expected_sha256 == "nan"
    ):

        raise RuntimeError(
            f"{dataset_id}: invalid SHA-256 "
            "in Notebook 02 manifest."
        )


    sha256_hash = hashlib.sha256()


    with open(
        train_path,
        "rb",
    ) as file_handle:

        for chunk in iter(
            lambda: file_handle.read(
                1024 * 1024
            ),
            b"",
        ):

            sha256_hash.update(
                chunk
            )


    actual_sha256 = (
        sha256_hash.hexdigest().lower()
    )


    if actual_sha256 != expected_sha256:

        raise RuntimeError(
            f"{dataset_id}: SHA-256 mismatch.\n"
            f"Manifest : {expected_sha256}\n"
            f"Actual   : {actual_sha256}"
        )


    # ==================================================================================================
    # 15.15 STORE VALIDATED TRAINING DATA
    # ==================================================================================================

    TRAINING_DATA[
        dataset_id
    ] = train_df


    TRAINING_GENERATIVE_COLUMNS[
        dataset_id
    ] = generative_columns


    TRAINING_FEATURE_COLUMNS[
        dataset_id
    ] = feature_columns


    TRAINING_TARGET_COLUMNS[
        dataset_id
    ] = target_column


    TRAINING_PROVENANCE_COLUMNS[
        dataset_id
    ] = provenance_column


    TRAINING_IDENTIFIER_COLUMNS[
        dataset_id
    ] = identifier_columns.copy()


    TRAINING_SOURCE_PATHS[
        dataset_id
    ] = str(
        train_path
    )


    TRAINING_SOURCE_HASHES[
        dataset_id
    ] = actual_sha256


    # ==================================================================================================
    # 15.16 DATASET VALIDATION REPORT
    # ==================================================================================================

    print(
        f"✓ Rows            : {actual_rows:,}"
    )

    print(
        f"✓ Native columns  : {actual_columns}"
    )

    print(
        f"✓ Generative cols : {actual_generative_count}"
    )

    print(
        f"✓ Feature cols    : {actual_feature_count}"
    )

    print(
        f"✓ Target          : {target_column}"
    )

    print(
        f"✓ Provenance      : {provenance_column}"
    )

    print(
        f"✓ Identifiers     : "
        f"{identifier_columns if identifier_columns else 'None'}"
    )

    print(
        f"✓ Source file     : {train_path}"
    )

    print(
        f"✓ SHA-256         : {actual_sha256}"
    )

    print(
        "✓ TRAIN validation : PASS"
    )


# ==================================================================================================
# 16. FINAL SECTION 3 INTEGRITY GATE
# ==================================================================================================

print()
print("=" * 100)
print("SECTION 3 — TRAINING DATA LOAD SUMMARY")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 16.1 Dataset completeness
# --------------------------------------------------------------------------------------------------

if len(
    TRAINING_DATA
) != len(
    DATASET_IDS
):

    raise RuntimeError(
        "Training dataset count does not match "
        "Notebook 00 dataset configuration."
    )


# --------------------------------------------------------------------------------------------------
# 16.2 Missing datasets
# --------------------------------------------------------------------------------------------------

MISSING_TRAINING_DATASETS = [
    dataset_id
    for dataset_id in DATASET_IDS
    if dataset_id not in TRAINING_DATA
]


if MISSING_TRAINING_DATASETS:

    raise RuntimeError(
        "Missing training datasets:\n"
        f"{MISSING_TRAINING_DATASETS}"
    )


# --------------------------------------------------------------------------------------------------
# 16.3 Unexpected datasets
# --------------------------------------------------------------------------------------------------

UNEXPECTED_TRAINING_DATASETS = [
    dataset_id
    for dataset_id in TRAINING_DATA
    if dataset_id not in DATASET_IDS
]


if UNEXPECTED_TRAINING_DATASETS:

    raise RuntimeError(
        "Unexpected datasets found in TRAINING_DATA:\n"
        f"{UNEXPECTED_TRAINING_DATASETS}"
    )


# --------------------------------------------------------------------------------------------------
# 16.4 Validate all containers
# --------------------------------------------------------------------------------------------------

if set(
    TRAINING_GENERATIVE_COLUMNS.keys()
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "TRAINING_GENERATIVE_COLUMNS coverage mismatch."
    )


if set(
    TRAINING_FEATURE_COLUMNS.keys()
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "TRAINING_FEATURE_COLUMNS coverage mismatch."
    )


if set(
    TRAINING_TARGET_COLUMNS.keys()
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "TRAINING_TARGET_COLUMNS coverage mismatch."
    )


if set(
    TRAINING_PROVENANCE_COLUMNS.keys()
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "TRAINING_PROVENANCE_COLUMNS coverage mismatch."
    )


if set(
    TRAINING_SOURCE_PATHS.keys()
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "TRAINING_SOURCE_PATHS coverage mismatch."
    )


if set(
    TRAINING_SOURCE_HASHES.keys()
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "TRAINING_SOURCE_HASHES coverage mismatch."
    )


# ==================================================================================================
# 17. FINAL TRAINING DATA SUMMARY
# ==================================================================================================

print(
    f"✓ Training datasets loaded : "
    f"{len(TRAINING_DATA)}"
)

print(
    f"✓ Dataset registry count    : "
    f"{len(DATASET_IDS)}"
)

print(
    "✓ All registered datasets loaded from "
    "canonical Notebook 02 TRAIN artifacts."
)

print(
    "✓ TRAIN-only loading policy verified."
)

print(
    "✓ Notebook 00 dataset configuration verified."
)

print(
    "✓ Notebook 02 native manifest integrity verified."
)

print(
    "✓ TRAIN file-size integrity verified."
)

print(
    "✓ TRAIN SHA-256 integrity verified."
)

print(
    "✓ Native row/column counts verified."
)

print(
    "✓ Generative column counts verified."
)

print(
    "✓ Feature column counts verified."
)

print(
    "✓ Target-column consistency verified."
)

print(
    "✓ Provenance-column consistency verified."
)

print(
    "✓ Explicit identifier exclusion verified."
)

print(
    "✓ TRAIN reload validation verified."
)

print()
print("=" * 100)
print("SECTION 3 STATUS: COMPLETE / PASS")
print("=" * 100)

SECTION 3 — LOAD TRAINING DATA
✓ Project root verified : /content/drive/MyDrive/SPP_GAN_Research
✓ Required Notebook 00 artifacts verified.
✓ Notebook 00 configuration loaded.
✓ Notebook 00 dataset registry loaded.
✓ Authoritative datasets resolved : 3
  • adult_income
  • bank_marketing
  • diabetes_130us
✓ Notebook 00 dataset registry verified.
✓ adult_income         target = income | identifiers = None
✓ bank_marketing       target = y | identifiers = None
✓ diabetes_130us       target = readmitted | identifiers = ['encounter_id', 'patient_nbr']
✓ Notebook 00 TRAIN-only fitting policy verified.
✓ Fit split : train
✓ Notebook 02 root verified : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
✓ Native output directory verified : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native
✓ Native manifest verified        : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native/native_dataset_manifest.csv
✓ Native manifest loaded     

In [7]:
# ==================================================================================================
# 4. VALIDATE DATA SCHEMA
# ==================================================================================================

print("=" * 100)
print("SECTION 4 — VALIDATE DATA SCHEMA")
print("=" * 100)

SCHEMA_VALIDATION_RECORDS = []

for dataset_id in DATASET_IDS:

    df = TRAINING_DATA[
        dataset_id
    ]

    generative_columns = (
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    feature_columns = (
        TRAINING_FEATURE_COLUMNS[
            dataset_id
        ]
    )

    target = (
        TRAINING_TARGET_COLUMNS[
            dataset_id
        ]
    )

    provenance = (
        TRAINING_PROVENANCE_COLUMNS[
            dataset_id
        ]
    )

    identifiers = (
        TRAINING_IDENTIFIER_COLUMNS[
            dataset_id
        ]
    )

    # ----------------------------------------------------------------------------------------------
    # Column uniqueness
    # ----------------------------------------------------------------------------------------------

    if not df.columns.is_unique:

        raise RuntimeError(
            f"{dataset_id}: duplicate column names detected."
        )

    # ----------------------------------------------------------------------------------------------
    # Target validation
    # ----------------------------------------------------------------------------------------------

    if target not in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: target not present in "
            "generative schema."
        )

    if target in feature_columns:

        raise RuntimeError(
            f"{dataset_id}: target appears in feature columns."
        )

    # ----------------------------------------------------------------------------------------------
    # Provenance exclusion
    # ----------------------------------------------------------------------------------------------

    if provenance in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance appears in "
            "generative schema."
        )

    # ----------------------------------------------------------------------------------------------
    # Identifier exclusion
    # ----------------------------------------------------------------------------------------------

    leaked_identifiers = (
        set(
            identifiers
        )
        .intersection(
            generative_columns
        )
    )

    if leaked_identifiers:

        raise RuntimeError(
            f"{dataset_id}: identifier leakage detected: "
            f"{leaked_identifiers}"
        )

    # ----------------------------------------------------------------------------------------------
    # Manifest consistency
    # ----------------------------------------------------------------------------------------------

    manifest_record = NATIVE_MANIFEST_DF[
        (
            NATIVE_MANIFEST_DF[
                "dataset_id"
            ]
            == dataset_id
        )
        &
        (
            NATIVE_MANIFEST_DF[
                "split"
            ]
            == "train"
        )
    ].iloc[0]

    expected_rows = int(
        manifest_record[
            "rows"
        ]
    )

    expected_generative_count = int(
        manifest_record[
            "generative_columns"
        ]
    )

    expected_feature_count = int(
        manifest_record[
            "preprocessing_feature_columns"
        ]
    )

    if len(df) != expected_rows:

        raise RuntimeError(
            f"{dataset_id}: row-count mismatch."
        )

    if len(
        generative_columns
    ) != expected_generative_count:

        raise RuntimeError(
            f"{dataset_id}: generative-column count mismatch."
        )

    if len(
        feature_columns
    ) != expected_feature_count:

        raise RuntimeError(
            f"{dataset_id}: feature-column count mismatch."
        )

    # ----------------------------------------------------------------------------------------------
    # Record
    # ----------------------------------------------------------------------------------------------

    SCHEMA_VALIDATION_RECORDS.append(
        {
            "dataset_id": dataset_id,
            "rows": len(df),
            "native_columns": len(df.columns),
            "generative_columns": len(
                generative_columns
            ),
            "feature_columns": len(
                feature_columns
            ),
            "target": target,
            "provenance_excluded": (
                provenance not in generative_columns
            ),
            "identifiers_excluded": (
                len(leaked_identifiers) == 0
            ),
            "status": "PASS",
        }
    )

    print(
        f"✓ {dataset_id:<20} | "
        f"features={len(feature_columns):>3} | "
        f"generative={len(generative_columns):>3} | "
        f"target={target:<12} | PASS"
    )

SCHEMA_VALIDATION_DF = pd.DataFrame(
    SCHEMA_VALIDATION_RECORDS
)

print()
print(
    f"✓ Schema validations: "
    f"{len(SCHEMA_VALIDATION_DF)}"
)

print(
    "✓ SECTION 4 — INPUT SCHEMA: PASS"
)

SECTION 4 — VALIDATE DATA SCHEMA
✓ adult_income         | features= 14 | generative= 15 | target=income       | PASS
✓ bank_marketing       | features= 16 | generative= 17 | target=y            | PASS
✓ diabetes_130us       | features= 47 | generative= 48 | target=readmitted   | PASS

✓ Schema validations: 3
✓ SECTION 4 — INPUT SCHEMA: PASS


In [10]:
# ==================================================================================================
# 5. CONFIGURE TVAE
# ==================================================================================================

print("=" * 100)
print("SECTION 5 — CONFIGURE TVAE")
print("=" * 100)


# ==================================================================================================
# 5.1 VERIFY UPSTREAM DEPENDENCIES
# ==================================================================================================

REQUIRED_SECTION_5_VARIABLES = [
    "PROJECT_ROOT",
    "DATASET_IDS",
    "NB00_CONFIG",
    "NB00_DATASET_REGISTRY",
    "TRAINING_DATA",
    "TRAINING_GENERATIVE_COLUMNS",
    "TRAINING_FEATURE_COLUMNS",
    "TRAINING_TARGET_COLUMNS",
    "TRAINING_PROVENANCE_COLUMNS",
    "TRAINING_IDENTIFIER_COLUMNS",
]

MISSING_SECTION_5_VARIABLES = [
    variable
    for variable in REQUIRED_SECTION_5_VARIABLES
    if variable not in globals()
]

if MISSING_SECTION_5_VARIABLES:

    raise RuntimeError(
        "Required Section 5 dependencies are missing:\n"
        f"{MISSING_SECTION_5_VARIABLES}\n\n"
        "Run the validated Notebook 05 Sections 2–4 first."
    )

print(
    "✓ Required upstream Section 5 dependencies verified."
)


# ==================================================================================================
# 5.2 LOAD AUTHORITATIVE REPRODUCIBILITY POLICY
# ==================================================================================================

SEED_POLICY = NB00_CONFIG.get(
    "seed_policy",
    {},
)

if not SEED_POLICY:

    raise RuntimeError(
        "Notebook 00 seed_policy is missing."
    )


MASTER_SEED = int(
    SEED_POLICY[
        "master_seed"
    ]
)

DETERMINISTIC_EXECUTION = bool(
    SEED_POLICY[
        "deterministic"
    ]
)

REPETITION_SEED_OFFSET = int(
    SEED_POLICY[
        "repetition_seed_offset"
    ]
)

REPETITION_SEEDS = {
    int(repetition): int(seed)
    for repetition, seed
    in SEED_POLICY[
        "repetition_seeds"
    ].items()
}

CONFIGURED_REPETITIONS = int(
    NB00_CONFIG[
        "repetitions"
    ]
)


# --------------------------------------------------------------------------------------------------
# Validate seed registry
# --------------------------------------------------------------------------------------------------

if MASTER_SEED != 2025:

    raise RuntimeError(
        "Unexpected Notebook 00 master seed.\n"
        f"Expected : 2025\n"
        f"Found    : {MASTER_SEED}"
    )


if not DETERMINISTIC_EXECUTION:

    raise RuntimeError(
        "Notebook 00 deterministic execution policy "
        "is disabled."
    )


EXPECTED_REPETITION_SEEDS = {
    repetition:
        MASTER_SEED
        + REPETITION_SEED_OFFSET
        + repetition
    for repetition in range(
        1,
        CONFIGURED_REPETITIONS + 1,
    )
}


if REPETITION_SEEDS != EXPECTED_REPETITION_SEEDS:

    raise RuntimeError(
        "Notebook 00 repetition seed registry mismatch.\n"
        f"Expected : {EXPECTED_REPETITION_SEEDS}\n"
        f"Found    : {REPETITION_SEEDS}"
    )


print(
    f"✓ Master seed             : {MASTER_SEED}"
)

print(
    f"✓ Deterministic execution : "
    f"{DETERMINISTIC_EXECUTION}"
)

print(
    f"✓ Configured repetitions  : "
    f"{CONFIGURED_REPETITIONS}"
)

print(
    f"✓ Repetition seed registry: "
    f"{REPETITION_SEEDS}"
)


# ==================================================================================================
# 5.3 TVAE MODEL CONFIGURATION
# ==================================================================================================

TVAE_CONFIG = {

    # ----------------------------------------------------------------------------------------------
    # Model identity
    # ----------------------------------------------------------------------------------------------

    "model": "TVAESynthesizer",

    # ----------------------------------------------------------------------------------------------
    # TVAE architecture
    # ----------------------------------------------------------------------------------------------

    "embedding_dim": 128,

    "compress_dims": (
        128,
        128,
    ),

    "decompress_dims": (
        128,
        128,
    ),

    # ----------------------------------------------------------------------------------------------
    # Optimization / regularization
    # ----------------------------------------------------------------------------------------------

    "loss_factor": 2,

    "l2scale": 1e-5,

    "batch_size": 500,

    "epochs": 300,

    # ----------------------------------------------------------------------------------------------
    # Data reconstruction behaviour
    # ----------------------------------------------------------------------------------------------

    "enforce_min_max_values": True,

    "enforce_rounding": True,

    # ----------------------------------------------------------------------------------------------
    # Logging
    # ----------------------------------------------------------------------------------------------

    "verbose": False,

    # ----------------------------------------------------------------------------------------------
    # Hardware policy
    #
    # requested_gpu indicates experimental preference.
    # Actual GPU availability is determined separately below.
    # ----------------------------------------------------------------------------------------------

    "requested_gpu": True,

    # ----------------------------------------------------------------------------------------------
    # Sampling policy
    # ----------------------------------------------------------------------------------------------

    "sample_size_policy": "training_rows",

    # ----------------------------------------------------------------------------------------------
    # Data fitting policy
    # ----------------------------------------------------------------------------------------------

    "fit_data_policy": "native_train_only",

    # ----------------------------------------------------------------------------------------------
    # Evaluation split policy
    # ----------------------------------------------------------------------------------------------

    "validation_used": False,

    "test_used": False,

    # ----------------------------------------------------------------------------------------------
    # Privacy / methodology exclusions
    # ----------------------------------------------------------------------------------------------

    "differential_privacy": False,

    "statistical_guidance": False,

    "sppgan_components": False,

    # ----------------------------------------------------------------------------------------------
    # Reproducibility
    # ----------------------------------------------------------------------------------------------

    "master_seed": MASTER_SEED,

    "deterministic_execution": DETERMINISTIC_EXECUTION,

    "repetitions": CONFIGURED_REPETITIONS,

    "repetition_seeds": REPETITION_SEEDS.copy(),
}


# ==================================================================================================
# 5.4 TVAE CONFIGURATION INTEGRITY
# ==================================================================================================

if TVAE_CONFIG["model"] != "TVAESynthesizer":

    raise RuntimeError(
        "Invalid TVAE model identifier."
    )


if TVAE_CONFIG["embedding_dim"] <= 0:

    raise RuntimeError(
        "TVAE embedding_dim must be positive."
    )


if TVAE_CONFIG["batch_size"] <= 0:

    raise RuntimeError(
        "TVAE batch_size must be positive."
    )


if TVAE_CONFIG["epochs"] <= 0:

    raise RuntimeError(
        "TVAE epochs must be positive."
    )


if TVAE_CONFIG["loss_factor"] <= 0:

    raise RuntimeError(
        "TVAE loss_factor must be positive."
    )


if TVAE_CONFIG["l2scale"] < 0:

    raise RuntimeError(
        "TVAE l2scale cannot be negative."
    )


if not isinstance(
    TVAE_CONFIG["compress_dims"],
    tuple,
):

    raise RuntimeError(
        "TVAE compress_dims must be a tuple."
    )


if not isinstance(
    TVAE_CONFIG["decompress_dims"],
    tuple,
):

    raise RuntimeError(
        "TVAE decompress_dims must be a tuple."
    )


if not TVAE_CONFIG["compress_dims"]:

    raise RuntimeError(
        "TVAE compress_dims cannot be empty."
    )


if not TVAE_CONFIG["decompress_dims"]:

    raise RuntimeError(
        "TVAE decompress_dims cannot be empty."
    )


if any(
    dimension <= 0
    for dimension in TVAE_CONFIG[
        "compress_dims"
    ]
):

    raise RuntimeError(
        "All TVAE compress dimensions must be positive."
    )


if any(
    dimension <= 0
    for dimension in TVAE_CONFIG[
        "decompress_dims"
    ]
):

    raise RuntimeError(
        "All TVAE decompress dimensions must be positive."
    )


# ==================================================================================================
# 5.5 VALIDATE EXPERIMENTAL BOUNDARY
# ==================================================================================================

if TVAE_CONFIG[
    "fit_data_policy"
] != "native_train_only":

    raise RuntimeError(
        "TVAE must use native TRAIN-only data."
    )


if TVAE_CONFIG[
    "validation_used"
]:

    raise RuntimeError(
        "Validation data cannot be used "
        "during Notebook 05 TVAE baseline fitting."
    )


if TVAE_CONFIG[
    "test_used"
]:

    raise RuntimeError(
        "Test data cannot be used "
        "during Notebook 05 TVAE baseline fitting."
    )


if TVAE_CONFIG[
    "differential_privacy"
]:

    raise RuntimeError(
        "TVAE baseline must remain non-private."
    )


if TVAE_CONFIG[
    "statistical_guidance"
]:

    raise RuntimeError(
        "Statistical guidance cannot be used "
        "in the TVAE baseline."
    )


if TVAE_CONFIG[
    "sppgan_components"
]:

    raise RuntimeError(
        "SPP-GAN components cannot be used "
        "in the TVAE baseline."
    )


if TVAE_CONFIG[
    "sample_size_policy"
] != "training_rows":

    raise RuntimeError(
        "TVAE synthetic sample-size policy "
        "must be training_rows."
    )


print(
    "✓ TVAE experimental boundary validated."
)

print(
    "✓ TRAIN-only fitting confirmed."
)

print(
    "✓ Validation data excluded."
)

print(
    "✓ Test data excluded."
)

print(
    "✓ Differential privacy disabled."
)

print(
    "✓ Statistical guidance disabled."
)

print(
    "✓ SPP-GAN components disabled."
)


# ==================================================================================================
# 5.6 VERIFY DATASET-SPECIFIC TRAINING INPUTS
# ==================================================================================================

for dataset_id in DATASET_IDS:

    if dataset_id not in TRAINING_DATA:

        raise RuntimeError(
            f"{dataset_id}: TRAINING_DATA is missing."
        )


    if dataset_id not in TRAINING_GENERATIVE_COLUMNS:

        raise RuntimeError(
            f"{dataset_id}: generative schema is missing."
        )


    if dataset_id not in TRAINING_FEATURE_COLUMNS:

        raise RuntimeError(
            f"{dataset_id}: feature schema is missing."
        )


    if dataset_id not in TRAINING_TARGET_COLUMNS:

        raise RuntimeError(
            f"{dataset_id}: target definition is missing."
        )


    if dataset_id not in TRAINING_PROVENANCE_COLUMNS:

        raise RuntimeError(
            f"{dataset_id}: provenance definition is missing."
        )


    if dataset_id not in TRAINING_IDENTIFIER_COLUMNS:

        raise RuntimeError(
            f"{dataset_id}: identifier definition is missing."
        )


    df = TRAINING_DATA[
        dataset_id
    ]


    generative_columns = (
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )


    feature_columns = (
        TRAINING_FEATURE_COLUMNS[
            dataset_id
        ]
    )


    target_column = (
        TRAINING_TARGET_COLUMNS[
            dataset_id
        ]
    )


    provenance_column = (
        TRAINING_PROVENANCE_COLUMNS[
            dataset_id
        ]
    )


    identifier_columns = (
        TRAINING_IDENTIFIER_COLUMNS[
            dataset_id
        ]
    )


    # ----------------------------------------------------------------------------------------------
    # Validate dataframe
    # ----------------------------------------------------------------------------------------------

    if df.empty:

        raise RuntimeError(
            f"{dataset_id}: TRAIN dataframe is empty."
        )


    # ----------------------------------------------------------------------------------------------
    # Validate generative schema
    # ----------------------------------------------------------------------------------------------

    missing_generative_columns = [
        column
        for column in generative_columns
        if column not in df.columns
    ]


    if missing_generative_columns:

        raise RuntimeError(
            f"{dataset_id}: missing generative columns:\n"
            f"{missing_generative_columns}"
        )


    # ----------------------------------------------------------------------------------------------
    # Validate feature schema
    # ----------------------------------------------------------------------------------------------

    missing_feature_columns = [
        column
        for column in feature_columns
        if column not in df.columns
    ]


    if missing_feature_columns:

        raise RuntimeError(
            f"{dataset_id}: missing feature columns:\n"
            f"{missing_feature_columns}"
        )


    # ----------------------------------------------------------------------------------------------
    # Validate target
    # ----------------------------------------------------------------------------------------------

    if target_column not in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: target is not present "
            "in the generative schema."
        )


    if target_column in feature_columns:

        raise RuntimeError(
            f"{dataset_id}: target is present "
            "in the feature schema."
        )


    # ----------------------------------------------------------------------------------------------
    # Validate provenance
    # ----------------------------------------------------------------------------------------------

    if provenance_column in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column is "
            "included in the generative schema."
        )


    # ----------------------------------------------------------------------------------------------
    # Validate explicit identifiers
    # ----------------------------------------------------------------------------------------------

    identifier_leakage = set(
        identifier_columns
    ).intersection(
        set(generative_columns)
    )


    if identifier_leakage:

        raise RuntimeError(
            f"{dataset_id}: identifier leakage detected:\n"
            f"{identifier_leakage}"
        )


print(
    "✓ Dataset-specific TVAE input schemas verified."
)


# ==================================================================================================
# 5.7 RUNTIME / HARDWARE DETECTION
# ==================================================================================================

RUNTIME_METADATA = {
    "python_version": None,
    "platform": None,
    "torch_version": None,
    "cuda_available": False,
    "cuda_version": None,
    "gpu_count": 0,
    "gpu_names": [],
    "requested_gpu": bool(
        TVAE_CONFIG[
            "requested_gpu"
        ]
    ),
    "actual_execution_device": None,
}


# --------------------------------------------------------------------------------------------------
# Python / platform
# --------------------------------------------------------------------------------------------------

import sys
import platform

RUNTIME_METADATA[
    "python_version"
] = sys.version

RUNTIME_METADATA[
    "platform"
] = platform.platform()


# --------------------------------------------------------------------------------------------------
# PyTorch
# --------------------------------------------------------------------------------------------------

try:

    import torch

    RUNTIME_METADATA[
        "torch_version"
    ] = torch.__version__

    RUNTIME_METADATA[
        "cuda_available"
    ] = bool(
        torch.cuda.is_available()
    )

    RUNTIME_METADATA[
        "cuda_version"
    ] = torch.version.cuda

    if torch.cuda.is_available():

        RUNTIME_METADATA[
            "gpu_count"
        ] = int(
            torch.cuda.device_count()
        )

        RUNTIME_METADATA[
            "gpu_names"
        ] = [
            torch.cuda.get_device_name(
                index
            )
            for index in range(
                torch.cuda.device_count()
            )
        ]

        RUNTIME_METADATA[
            "actual_execution_device"
        ] = "cuda"

    else:

        RUNTIME_METADATA[
            "actual_execution_device"
        ] = "cpu"


except ImportError:

    RUNTIME_METADATA[
        "torch_version"
    ] = None

    RUNTIME_METADATA[
        "actual_execution_device"
    ] = "cpu"


# ==================================================================================================
# 5.8 HARDWARE POLICY
# ==================================================================================================

print()
print(
    "RUNTIME / HARDWARE"
)
print("-" * 100)

print(
    f"✓ Python version     : "
    f"{platform.python_version()}"
)

print(
    f"✓ PyTorch version    : "
    f"{RUNTIME_METADATA['torch_version']}"
)

print(
    f"✓ GPU requested      : "
    f"{RUNTIME_METADATA['requested_gpu']}"
)

print(
    f"✓ CUDA available     : "
    f"{RUNTIME_METADATA['cuda_available']}"
)

print(
    f"✓ CUDA version       : "
    f"{RUNTIME_METADATA['cuda_version']}"
)

print(
    f"✓ GPU count          : "
    f"{RUNTIME_METADATA['gpu_count']}"
)


if RUNTIME_METADATA[
    "gpu_names"
]:

    for gpu_index, gpu_name in enumerate(
        RUNTIME_METADATA[
            "gpu_names"
        ]
    ):

        print(
            f"  GPU {gpu_index} : "
            f"{gpu_name}"
        )

else:

    print(
        "  GPU 0 : Not available"
    )


print(
    f"✓ Runtime device     : "
    f"{RUNTIME_METADATA['actual_execution_device']}"
)


# --------------------------------------------------------------------------------------------------
# Important:
# GPU absence is NOT treated as a configuration failure.
#
# The experiment records whether GPU was requested and whether it is
# actually available. The training section must use this metadata
# rather than falsely reporting GPU execution.
# --------------------------------------------------------------------------------------------------

if (
    TVAE_CONFIG[
        "requested_gpu"
    ]
    and not RUNTIME_METADATA[
        "cuda_available"
    ]
):

    print(
        "⚠ GPU was requested but is not available."
    )

    print(
        "⚠ TVAE training must execute on CPU "
        "unless a compatible GPU becomes available."
    )


# ==================================================================================================
# 5.9 SDV ENVIRONMENT DETECTION
# ==================================================================================================

SDV_VERSION = None

try:

    import sdv

    SDV_VERSION = getattr(
        sdv,
        "__version__",
        None,
    )

except ImportError:

    SDV_VERSION = None


RUNTIME_METADATA[
    "sdv_version"
] = SDV_VERSION


if SDV_VERSION is None:

    print(
        "⚠ SDV package version could not be detected "
        "at configuration time."
    )

else:

    print(
        f"✓ SDV version        : "
        f"{SDV_VERSION}"
    )


# ==================================================================================================
# 5.10 TVAE REPRODUCIBILITY POLICY
# ==================================================================================================

if TVAE_CONFIG[
    "master_seed"
] != MASTER_SEED:

    raise RuntimeError(
        "TVAE master seed is inconsistent "
        "with Notebook 00."
    )


if TVAE_CONFIG[
    "repetitions"
] != CONFIGURED_REPETITIONS:

    raise RuntimeError(
        "TVAE repetition count is inconsistent "
        "with Notebook 00."
    )


if TVAE_CONFIG[
    "repetition_seeds"
] != REPETITION_SEEDS:

    raise RuntimeError(
        "TVAE repetition seed registry is inconsistent "
        "with Notebook 00."
    )


print(
    "✓ TVAE master seed linked to Notebook 00."
)

print(
    "✓ TVAE repetition registry linked to Notebook 00."
)

print(
    "✓ Deterministic execution policy linked to Notebook 00."
)


# ==================================================================================================
# 5.11 BUILD RESEARCH CONFIGURATION SNAPSHOT
# ==================================================================================================

TVAE_CONFIGURATION_SNAPSHOT = {

    "model": TVAE_CONFIG[
        "model"
    ],

    "architecture": {
        "embedding_dim":
            TVAE_CONFIG[
                "embedding_dim"
            ],

        "compress_dims":
            list(
                TVAE_CONFIG[
                    "compress_dims"
                ]
            ),

        "decompress_dims":
            list(
                TVAE_CONFIG[
                    "decompress_dims"
                ]
            ),
    },

    "optimization": {
        "loss_factor":
            TVAE_CONFIG[
                "loss_factor"
            ],

        "l2scale":
            TVAE_CONFIG[
                "l2scale"
            ],

        "batch_size":
            TVAE_CONFIG[
                "batch_size"
            ],

        "epochs":
            TVAE_CONFIG[
                "epochs"
            ],
    },

    "data_policy": {
        "fit_data_policy":
            TVAE_CONFIG[
                "fit_data_policy"
            ],

        "sample_size_policy":
            TVAE_CONFIG[
                "sample_size_policy"
            ],

        "validation_used":
            TVAE_CONFIG[
                "validation_used"
            ],

        "test_used":
            TVAE_CONFIG[
                "test_used"
            ],
    },

    "privacy_policy": {
        "differential_privacy":
            TVAE_CONFIG[
                "differential_privacy"
            ],
    },

    "methodology_policy": {
        "statistical_guidance":
            TVAE_CONFIG[
                "statistical_guidance"
            ],

        "sppgan_components":
            TVAE_CONFIG[
                "sppgan_components"
            ],
    },

    "reproducibility": {
        "master_seed":
            MASTER_SEED,

        "deterministic_execution":
            DETERMINISTIC_EXECUTION,

        "repetitions":
            CONFIGURED_REPETITIONS,

        "repetition_seeds":
            REPETITION_SEEDS.copy(),
    },

    "hardware": {
        "gpu_requested":
            RUNTIME_METADATA[
                "requested_gpu"
            ],

        "cuda_available":
            RUNTIME_METADATA[
                "cuda_available"
            ],

        "cuda_version":
            RUNTIME_METADATA[
                "cuda_version"
            ],

        "gpu_count":
            RUNTIME_METADATA[
                "gpu_count"
            ],

        "gpu_names":
            RUNTIME_METADATA[
                "gpu_names"
            ],

        "execution_device":
            RUNTIME_METADATA[
                "actual_execution_device"
            ],
    },

    "environment": {
        "python_version":
            RUNTIME_METADATA[
                "python_version"
            ],

        "platform":
            RUNTIME_METADATA[
                "platform"
            ],

        "torch_version":
            RUNTIME_METADATA[
                "torch_version"
            ],

        "sdv_version":
            RUNTIME_METADATA[
                "sdv_version"
            ],
    },

    "datasets":
        list(
            DATASET_IDS
        ),
}


# ==================================================================================================
# 5.12 CONFIGURATION FINGERPRINT
# ==================================================================================================

TVAE_CONFIGURATION_SERIALIZED = json.dumps(
    TVAE_CONFIGURATION_SNAPSHOT,
    sort_keys=True,
    default=str,
    ensure_ascii=False,
    separators=(
        ",",
        ":",
    ),
).encode(
    "UTF-8"
)


TVAE_CONFIGURATION_FINGERPRINT = (
    hashlib.sha256(
        TVAE_CONFIGURATION_SERIALIZED
    ).hexdigest()
)


print()
print(
    "TVAE CONFIGURATION FINGERPRINT"
)
print("-" * 100)

print(
    f"✓ SHA-256 fingerprint : "
    f"{TVAE_CONFIGURATION_FINGERPRINT}"
)


# ==================================================================================================
# 5.13 FINAL CONFIGURATION REPORT
# ==================================================================================================

print()
print("=" * 100)
print("TVAE CONFIGURATION SUMMARY")
print("=" * 100)

print(
    f"✓ Model               : "
    f"{TVAE_CONFIG['model']}"
)

print(
    f"✓ Epochs              : "
    f"{TVAE_CONFIG['epochs']}"
)

print(
    f"✓ Batch size          : "
    f"{TVAE_CONFIG['batch_size']}"
)

print(
    f"✓ Embedding dimension : "
    f"{TVAE_CONFIG['embedding_dim']}"
)

print(
    f"✓ Compress dimensions : "
    f"{TVAE_CONFIG['compress_dims']}"
)

print(
    f"✓ Decompress dims     : "
    f"{TVAE_CONFIG['decompress_dims']}"
)

print(
    f"✓ Loss factor         : "
    f"{TVAE_CONFIG['loss_factor']}"
)

print(
    f"✓ L2 scale            : "
    f"{TVAE_CONFIG['l2scale']}"
)

print(
    f"✓ GPU requested       : "
    f"{TVAE_CONFIG['requested_gpu']}"
)

print(
    f"✓ GPU available       : "
    f"{RUNTIME_METADATA['cuda_available']}"
)

print(
    f"✓ Actual device       : "
    f"{RUNTIME_METADATA['actual_execution_device']}"
)

print(
    f"✓ SDV version         : "
    f"{SDV_VERSION}"
)

print(
    f"✓ Master seed         : "
    f"{MASTER_SEED}"
)

print(
    f"✓ Repetitions         : "
    f"{CONFIGURED_REPETITIONS}"
)

print(
    f"✓ Validation used     : "
    f"{TVAE_CONFIG['validation_used']}"
)

print(
    f"✓ Test used           : "
    f"{TVAE_CONFIG['test_used']}"
)

print(
    f"✓ Differential privacy: "
    f"{TVAE_CONFIG['differential_privacy']}"
)

print(
    f"✓ Statistical guidance: "
    f"{TVAE_CONFIG['statistical_guidance']}"
)

print(
    f"✓ SPP-GAN components  : "
    f"{TVAE_CONFIG['sppgan_components']}"
)

print(
    f"✓ Configuration hash  : "
    f"{TVAE_CONFIGURATION_FINGERPRINT}"
)


# ==================================================================================================
# 5.14 FINAL SECTION 5 INTEGRITY GATE
# ==================================================================================================

SECTION_5_CHECKS = {

    "model_identity":
        TVAE_CONFIG[
            "model"
        ] == "TVAESynthesizer",

    "positive_embedding_dim":
        TVAE_CONFIG[
            "embedding_dim"
        ] > 0,

    "positive_batch_size":
        TVAE_CONFIG[
            "batch_size"
        ] > 0,

    "positive_epochs":
        TVAE_CONFIG[
            "epochs"
        ] > 0,

    "valid_compress_dims":
        all(
            dimension > 0
            for dimension
            in TVAE_CONFIG[
                "compress_dims"
            ]
        ),

    "valid_decompress_dims":
        all(
            dimension > 0
            for dimension
            in TVAE_CONFIG[
                "decompress_dims"
            ]
        ),

    "train_only_policy":
        TVAE_CONFIG[
            "fit_data_policy"
        ] == "native_train_only",

    "validation_excluded":
        TVAE_CONFIG[
            "validation_used"
        ] is False,

    "test_excluded":
        TVAE_CONFIG[
            "test_used"
        ] is False,

    "non_private_baseline":
        TVAE_CONFIG[
            "differential_privacy"
        ] is False,

    "statistical_guidance_disabled":
        TVAE_CONFIG[
            "statistical_guidance"
        ] is False,

    "sppgan_disabled":
        TVAE_CONFIG[
            "sppgan_components"
        ] is False,

    "sample_size_policy":
        TVAE_CONFIG[
            "sample_size_policy"
        ] == "training_rows",

    "master_seed_consistent":
        TVAE_CONFIG[
            "master_seed"
        ] == MASTER_SEED,

    "deterministic_execution":
        TVAE_CONFIG[
            "deterministic_execution"
        ] is True,

    "repetition_registry":
        TVAE_CONFIG[
            "repetition_seeds"
        ] == REPETITION_SEEDS,

    "dataset_coverage":
        set(DATASET_IDS)
        == set(TRAINING_DATA.keys()),

    "configuration_fingerprint":
        len(
            TVAE_CONFIGURATION_FINGERPRINT
        ) == 64,
}


FAILED_SECTION_5_CHECKS = [
    check_name
    for check_name, check_result
    in SECTION_5_CHECKS.items()
    if not check_result
]


print()
print("=" * 100)
print("SECTION 5 — CONFIGURATION INTEGRITY GATE")
print("=" * 100)

for check_name, check_result in SECTION_5_CHECKS.items():

    status = (
        "PASS"
        if check_result
        else "FAIL"
    )

    print(
        f"{'✓' if check_result else '✗'} "
        f"{check_name:<35} : "
        f"{status}"
    )


if FAILED_SECTION_5_CHECKS:

    raise RuntimeError(
        "SECTION 5 FAILED.\n"
        f"Failed checks: {FAILED_SECTION_5_CHECKS}"
    )


print()
print(
    f"Total checks  : "
    f"{len(SECTION_5_CHECKS)}"
)

print(
    f"Passed checks : "
    f"{len(SECTION_5_CHECKS) - len(FAILED_SECTION_5_CHECKS)}"
)

print(
    f"Failed checks : "
    f"{len(FAILED_SECTION_5_CHECKS)}"
)

print()
print(
    "✓ TVAE configuration validated."
)

print(
    "✓ TVAE baseline remains non-private."
)

print(
    "✓ TVAE baseline remains independent of SPP-GAN."
)

print(
    "✓ TVAE fitting boundary remains TRAIN-only."
)

print(
    "✓ Reproducibility policy inherited from Notebook 00."
)

print(
    "✓ Runtime hardware status recorded."
)

print(
    "✓ TVAE configuration fingerprint generated."
)

print()
print("=" * 100)
print("SECTION 5 STATUS: COMPLETE / PASS")
print("=" * 100)

SECTION 5 — CONFIGURE TVAE
✓ Required upstream Section 5 dependencies verified.
✓ Master seed             : 2025
✓ Deterministic execution : True
✓ Configured repetitions  : 5
✓ Repetition seed registry: {1: 3026, 2: 3027, 3: 3028, 4: 3029, 5: 3030}
✓ TVAE experimental boundary validated.
✓ TRAIN-only fitting confirmed.
✓ Validation data excluded.
✓ Test data excluded.
✓ Differential privacy disabled.
✓ Statistical guidance disabled.
✓ SPP-GAN components disabled.
✓ Dataset-specific TVAE input schemas verified.

RUNTIME / HARDWARE
----------------------------------------------------------------------------------------------------
✓ Python version     : 3.13.15
✓ PyTorch version    : 2.11.0+cu128
✓ GPU requested      : True
✓ CUDA available     : True
✓ CUDA version       : 12.8
✓ GPU count          : 1
  GPU 0 : Tesla T4
✓ Runtime device     : cuda
⚠ SDV package version could not be detected at configuration time.
✓ TVAE master seed linked to Notebook 00.
✓ TVAE repetition registry lin

In [12]:
# ==================================================================================================
# 6. SET SEEDS
# ==================================================================================================

print("=" * 100)
print("SECTION 6 — SET SEEDS")
print("=" * 100)


# ==================================================================================================
# 6.1 REQUIRED DEPENDENCIES
# ==================================================================================================

REQUIRED_SECTION_6_VARIABLES = [
    "DATASET_IDS",
    "MASTER_SEED",
    "DETERMINISTIC_EXECUTION",
    "REPETITION_SEEDS",
    "CONFIGURED_REPETITIONS",
    "TVAE_CONFIG",
]

MISSING_SECTION_6_VARIABLES = [
    variable
    for variable in REQUIRED_SECTION_6_VARIABLES
    if variable not in globals()
]

if MISSING_SECTION_6_VARIABLES:

    raise RuntimeError(
        "Required Section 6 dependencies are missing:\n"
        f"{MISSING_SECTION_6_VARIABLES}\n\n"
        "Run the validated Notebook 05 Sections 2–5 first."
    )


print(
    "✓ Required Section 6 dependencies verified."
)


# ==================================================================================================
# 6.2 IMPORT REPRODUCIBILITY LIBRARIES
# ==================================================================================================

import os
import random
import numpy as np


try:

    import torch

    TORCH_AVAILABLE = True

except ImportError:

    torch = None

    TORCH_AVAILABLE = False


print(
    f"✓ Python random module available."
)

print(
    f"✓ NumPy version           : {np.__version__}"
)

print(
    f"✓ PyTorch available       : {TORCH_AVAILABLE}"
)

if TORCH_AVAILABLE:

    print(
        f"✓ PyTorch version         : {torch.__version__}"
    )


# ==================================================================================================
# 6.3 VERIFY AUTHORITATIVE NOTEBOOK 00 SEED POLICY
# ==================================================================================================

if MASTER_SEED != 2025:

    raise RuntimeError(
        "Unexpected Notebook 00 master seed.\n"
        f"Expected : 2025\n"
        f"Found    : {MASTER_SEED}"
    )


if not DETERMINISTIC_EXECUTION:

    raise RuntimeError(
        "Notebook 00 deterministic execution policy "
        "is disabled."
    )


if CONFIGURED_REPETITIONS != 5:

    raise RuntimeError(
        "Unexpected number of experimental repetitions.\n"
        f"Expected : 5\n"
        f"Found    : {CONFIGURED_REPETITIONS}"
    )


EXPECTED_REPETITION_NUMBERS = list(
    range(
        1,
        CONFIGURED_REPETITIONS + 1,
    )
)


ACTUAL_REPETITION_NUMBERS = sorted(
    REPETITION_SEEDS.keys()
)


if (
    ACTUAL_REPETITION_NUMBERS
    != EXPECTED_REPETITION_NUMBERS
):

    raise RuntimeError(
        "Notebook 00 repetition seed registry "
        "has an invalid repetition-number set.\n"
        f"Expected : {EXPECTED_REPETITION_NUMBERS}\n"
        f"Found    : {ACTUAL_REPETITION_NUMBERS}"
    )


EXPECTED_REPETITION_SEEDS = {
    repetition:
        MASTER_SEED
        + 1000
        + repetition
    for repetition in EXPECTED_REPETITION_NUMBERS
}


if REPETITION_SEEDS != EXPECTED_REPETITION_SEEDS:

    raise RuntimeError(
        "Notebook 00 repetition seed registry "
        "does not match the authoritative seed policy.\n"
        f"Expected : {EXPECTED_REPETITION_SEEDS}\n"
        f"Found    : {REPETITION_SEEDS}"
    )


print(
    "✓ Notebook 00 master seed verified."
)

print(
    "✓ Notebook 00 deterministic execution verified."
)

print(
    "✓ Notebook 00 repetition seed registry verified."
)


# ==================================================================================================
# 6.4 DEFINE DETERMINISTIC DATASET-SPECIFIC SEED POLICY
# ==================================================================================================
#
# Notebook 00 remains the authoritative source for experimental repetition seeds.
#
# Dataset-specific seeds are deterministic derivatives of the authoritative repetition seed.
#
# Formula:
#
#     dataset_seed =
#         repetition_seed
#         + (dataset_index + 1) × 100
#
# This avoids creating an unrelated master seed policy while ensuring that each
# dataset/repetition combination receives a distinct deterministic seed.
#
# The dataset index follows the authoritative DATASET_IDS order from Notebook 00.
#
# ==================================================================================================

DATASET_SEED_OFFSET = 100


# --------------------------------------------------------------------------------------------------
# Dataset-level seed registry for the current default repetition
# --------------------------------------------------------------------------------------------------

DEFAULT_REPETITION = 1

if DEFAULT_REPETITION not in REPETITION_SEEDS:

    raise RuntimeError(
        f"Default repetition {DEFAULT_REPETITION} "
        "is missing from REPETITION_SEEDS."
    )


DEFAULT_REPETITION_SEED = int(
    REPETITION_SEEDS[
        DEFAULT_REPETITION
    ]
)


TVAE_SEED_REGISTRY = {}

for dataset_index, dataset_id in enumerate(
    DATASET_IDS
):

    dataset_seed = (
        DEFAULT_REPETITION_SEED
        + (dataset_index + 1)
        * DATASET_SEED_OFFSET
    )

    TVAE_SEED_REGISTRY[
        dataset_id
    ] = int(
        dataset_seed
    )


# ==================================================================================================
# 6.5 BUILD COMPLETE DATASET × REPETITION SEED REGISTRY
# ==================================================================================================

TVAE_REPETITION_SEED_REGISTRY = {}

for repetition in EXPECTED_REPETITION_NUMBERS:

    repetition_seed = int(
        REPETITION_SEEDS[
            repetition
        ]
    )

    TVAE_REPETITION_SEED_REGISTRY[
        repetition
    ] = {}

    for dataset_index, dataset_id in enumerate(
        DATASET_IDS
    ):

        dataset_seed = (
            repetition_seed
            + (dataset_index + 1)
            * DATASET_SEED_OFFSET
        )

        TVAE_REPETITION_SEED_REGISTRY[
            repetition
        ][
            dataset_id
        ] = int(
            dataset_seed
        )


# ==================================================================================================
# 6.6 VALIDATE SEED UNIQUENESS
# ==================================================================================================

ALL_DERIVED_SEEDS = []

for repetition in EXPECTED_REPETITION_NUMBERS:

    for dataset_id in DATASET_IDS:

        ALL_DERIVED_SEEDS.append(
            TVAE_REPETITION_SEED_REGISTRY[
                repetition
            ][
                dataset_id
            ]
        )


if len(
    ALL_DERIVED_SEEDS
) != len(
    set(ALL_DERIVED_SEEDS)
):

    raise RuntimeError(
        "Duplicate dataset × repetition TVAE seeds detected."
    )


if len(
    set(
        TVAE_SEED_REGISTRY.values()
    )
) != len(
    TVAE_SEED_REGISTRY
):

    raise RuntimeError(
        "Duplicate default-repetition TVAE dataset seeds detected."
    )


print(
    "✓ Dataset × repetition seed uniqueness verified."
)


# ==================================================================================================
# 6.7 SEED EVERYTHING FUNCTION
# ==================================================================================================

def seed_everything(
    seed: int,
):
    """
    Configure deterministic random seeds for Python, NumPy,
    and PyTorch where available.
    """

    seed = int(
        seed
    )


    # ----------------------------------------------------------------------------------------------
    # Python hash seed
    # ----------------------------------------------------------------------------------------------

    os.environ[
        "PYTHONHASHSEED"
    ] = str(
        seed
    )


    # ----------------------------------------------------------------------------------------------
    # Python standard-library RNG
    # ----------------------------------------------------------------------------------------------

    random.seed(
        seed
    )


    # ----------------------------------------------------------------------------------------------
    # NumPy RNG
    # ----------------------------------------------------------------------------------------------

    np.random.seed(
        seed
    )


    # ----------------------------------------------------------------------------------------------
    # PyTorch RNG
    # ----------------------------------------------------------------------------------------------

    if TORCH_AVAILABLE:

        torch.manual_seed(
            seed
        )


        if torch.cuda.is_available():

            torch.cuda.manual_seed(
                seed
            )

            torch.cuda.manual_seed_all(
                seed
            )


    return seed


# ==================================================================================================
# 6.8 CONFIGURE PYTORCH DETERMINISM
# ==================================================================================================

if TORCH_AVAILABLE:

    try:

        torch.backends.cudnn.deterministic = True

        torch.backends.cudnn.benchmark = False

        print(
            "✓ cuDNN deterministic mode configured."
        )

    except Exception as exc:

        print(
            "⚠ cuDNN deterministic configuration "
            f"could not be fully applied: {exc}"
        )


    # ----------------------------------------------------------------------------------------------
    # PyTorch deterministic algorithms
    # ----------------------------------------------------------------------------------------------

    try:

        torch.use_deterministic_algorithms(
            True
        )

        PYTORCH_DETERMINISTIC_ALGORITHMS = True

        print(
            "✓ PyTorch deterministic algorithms enabled."
        )

    except Exception as exc:

        PYTORCH_DETERMINISTIC_ALGORITHMS = False

        print(
            "⚠ PyTorch deterministic algorithms "
            f"could not be fully enabled: {exc}"
        )

else:

    PYTORCH_DETERMINISTIC_ALGORITHMS = False


# ==================================================================================================
# 6.9 CONFIGURE DEFAULT GLOBAL SEED
# ==================================================================================================

seed_everything(
    MASTER_SEED
)


print(
    f"✓ Global master seed initialized : "
    f"{MASTER_SEED}"
)


# ==================================================================================================
# 6.10 DISPLAY AUTHORITATIVE REPETITION SEEDS
# ==================================================================================================

print()
print(
    "AUTHORITATIVE NOTEBOOK 00 REPETITION SEEDS"
)
print("-" * 100)

for repetition in EXPECTED_REPETITION_NUMBERS:

    print(
        f"✓ Repetition {repetition} "
        f"→ seed={REPETITION_SEEDS[repetition]}"
    )


# ==================================================================================================
# 6.11 DISPLAY DEFAULT TVAE DATASET SEEDS
# ==================================================================================================

print()
print(
    "TVAE DATASET SEEDS — REPETITION 1"
)
print("-" * 100)

for dataset_id in DATASET_IDS:

    print(
        f"✓ {dataset_id:<20} | "
        f"seed={TVAE_SEED_REGISTRY[dataset_id]}"
    )


# ==================================================================================================
# 6.12 DISPLAY COMPLETE DATASET × REPETITION REGISTRY
# ==================================================================================================

print()
print(
    "TVAE DATASET × REPETITION SEED REGISTRY"
)
print("-" * 100)

for repetition in EXPECTED_REPETITION_NUMBERS:

    print(
        f"Repetition {repetition} "
        f"(base seed={REPETITION_SEEDS[repetition]})"
    )

    for dataset_id in DATASET_IDS:

        print(
            f"  • {dataset_id:<20} → "
            f"{TVAE_REPETITION_SEED_REGISTRY[repetition][dataset_id]}"
        )


# ==================================================================================================
# 6.13 FINAL SEED POLICY SNAPSHOT
# ==================================================================================================

TVAE_SEED_POLICY = {

    "master_seed":
        MASTER_SEED,

    "deterministic_execution":
        DETERMINISTIC_EXECUTION,

    "configured_repetitions":
        CONFIGURED_REPETITIONS,

    "authoritative_repetition_seeds":
        {
            str(repetition):
                int(
                    REPETITION_SEEDS[
                        repetition
                    ]
                )
            for repetition
            in EXPECTED_REPETITION_NUMBERS
        },

    "dataset_seed_offset":
        DATASET_SEED_OFFSET,

    "default_repetition":
        DEFAULT_REPETITION,

    "default_repetition_seed":
        DEFAULT_REPETITION_SEED,

    "dataset_seed_registry":
        {
            dataset_id:
                int(
                    TVAE_SEED_REGISTRY[
                        dataset_id
                    ]
                )
            for dataset_id
            in DATASET_IDS
        },

    "dataset_repetition_seed_registry":
        {
            str(repetition):
                {
                    dataset_id:
                        int(
                            TVAE_REPETITION_SEED_REGISTRY[
                                repetition
                            ][
                                dataset_id
                            ]
                        )
                    for dataset_id
                    in DATASET_IDS
                }
            for repetition
            in EXPECTED_REPETITION_NUMBERS
        },

    "pythonhashseed":
        os.environ.get(
            "PYTHONHASHSEED"
        ),

    "pytorch_available":
        TORCH_AVAILABLE,

    "pytorch_deterministic_algorithms":
        PYTORCH_DETERMINISTIC_ALGORITHMS,

    "cuda_available":
        bool(
            torch.cuda.is_available()
        )
        if TORCH_AVAILABLE
        else False,
}


# ==================================================================================================
# 6.14 FINAL SEED INTEGRITY GATE
# ==================================================================================================

SECTION_6_CHECKS = {

    "random_module_available":
        "random" in globals(),

    "master_seed_consistent":
        MASTER_SEED == 2025,

    "deterministic_execution":
        DETERMINISTIC_EXECUTION is True,

    "configured_repetitions":
        CONFIGURED_REPETITIONS == 5,

    "repetition_numbers":
        ACTUAL_REPETITION_NUMBERS
        == EXPECTED_REPETITION_NUMBERS,

    "repetition_seed_registry":
        REPETITION_SEEDS
        == EXPECTED_REPETITION_SEEDS,

    "dataset_coverage":
        set(DATASET_IDS)
        == set(TRAINING_DATA.keys()),

    "dataset_seed_registry":
        set(TVAE_SEED_REGISTRY.keys())
        == set(DATASET_IDS),

    "dataset_seed_uniqueness":
        len(ALL_DERIVED_SEEDS)
        == len(set(ALL_DERIVED_SEEDS)),

    "default_seed_uniqueness":
        len(
            TVAE_SEED_REGISTRY.values()
        )
        == len(
            set(
                TVAE_SEED_REGISTRY.values()
            )
        ),

    "global_seed_initialized":
        os.environ.get(
            "PYTHONHASHSEED"
        ) == str(
            MASTER_SEED
        ),

    "pytorch_seed_support":
        TORCH_AVAILABLE,

    "seed_policy_snapshot":
        isinstance(
            TVAE_SEED_POLICY,
            dict,
        ),

    "dataset_repetition_registry":
        len(
            TVAE_REPETITION_SEED_REGISTRY
        )
        == CONFIGURED_REPETITIONS,
}


FAILED_SECTION_6_CHECKS = [
    check_name
    for check_name, check_result
    in SECTION_6_CHECKS.items()
    if not check_result
]


print()
print("=" * 100)
print("SECTION 6 — SEED INTEGRITY GATE")
print("=" * 100)


for check_name, check_result in SECTION_6_CHECKS.items():

    print(
        f"{'✓' if check_result else '✗'} "
        f"{check_name:<35} : "
        f"{'PASS' if check_result else 'FAIL'}"
    )


if FAILED_SECTION_6_CHECKS:

    raise RuntimeError(
        "SECTION 6 FAILED.\n"
        f"Failed checks: {FAILED_SECTION_6_CHECKS}"
    )


print()
print(
    f"Total checks  : "
    f"{len(SECTION_6_CHECKS)}"
)

print(
    f"Passed checks : "
    f"{len(SECTION_6_CHECKS) - len(FAILED_SECTION_6_CHECKS)}"
)

print(
    f"Failed checks : "
    f"{len(FAILED_SECTION_6_CHECKS)}"
)

print()
print(
    "✓ Python random seed configured."
)

print(
    "✓ NumPy seed configured."
)

print(
    "✓ PyTorch seed configured."
)

print(
    "✓ CUDA seeds configured where available."
)

print(
    "✓ Dataset-specific deterministic seeds configured."
)

print(
    "✓ Repetition-specific deterministic seeds configured."
)

print(
    "✓ Notebook 00 remains the authoritative seed source."
)

print(
    "✓ Seed policy is reproducible and auditable."
)

print()
print("=" * 100)
print("SECTION 6 STATUS: COMPLETE / PASS")
print("=" * 100)

SECTION 6 — SET SEEDS
✓ Required Section 6 dependencies verified.
✓ Python random module available.
✓ NumPy version           : 2.1.3
✓ PyTorch available       : True
✓ PyTorch version         : 2.11.0+cu128
✓ Notebook 00 master seed verified.
✓ Notebook 00 deterministic execution verified.
✓ Notebook 00 repetition seed registry verified.
✓ Dataset × repetition seed uniqueness verified.
✓ cuDNN deterministic mode configured.
✓ PyTorch deterministic algorithms enabled.
✓ Global master seed initialized : 2025

AUTHORITATIVE NOTEBOOK 00 REPETITION SEEDS
----------------------------------------------------------------------------------------------------
✓ Repetition 1 → seed=3026
✓ Repetition 2 → seed=3027
✓ Repetition 3 → seed=3028
✓ Repetition 4 → seed=3029
✓ Repetition 5 → seed=3030

TVAE DATASET SEEDS — REPETITION 1
----------------------------------------------------------------------------------------------------
✓ adult_income         | seed=3126
✓ bank_marketing       | seed=3226
✓

In [17]:
# ==================================================================================================
# NOTEBOOK 05 — SDV ENVIRONMENT SETUP
# ==================================================================================================

!pip install -q "sdv==1.38.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.4/215.4 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.7/213.7 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 9.5 MB/s eta 0:00:00


In [18]:
# ==================================================================================================
# VERIFY SDV INSTALLATION
# ==================================================================================================

import sys
import sdv

print("=" * 100)
print("SDV ENVIRONMENT VERIFICATION")
print("=" * 100)

print(f"Python version : {sys.version.split()[0]}")
print(f"SDV version    : {sdv.__version__}")

from sdv.metadata import SingleTableMetadata
from sdv.single_table import TVAESynthesizer

print(f"Metadata class : {SingleTableMetadata.__name__}")
print(f"TVAE class     : {TVAESynthesizer.__name__}")

print()
print("✓ SDV installed successfully.")
print("✓ SingleTableMetadata available.")
print("✓ TVAESynthesizer available.")

SDV ENVIRONMENT VERIFICATION
Python version : 3.13.15
SDV version    : 1.38.3
Metadata class : SingleTableMetadata
TVAE class     : TVAESynthesizer

✓ SDV installed successfully.
✓ SingleTableMetadata available.
✓ TVAESynthesizer available.


In [19]:
# ==================================================================================================
# 7. INITIALIZE TVAE
# ==================================================================================================

print("=" * 100)
print("SECTION 7 — INITIALIZE TVAE")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Import SDV Components
# --------------------------------------------------------------------------------------------------

try:
    import sdv
    from sdv.metadata import SingleTableMetadata
    from sdv.single_table import TVAESynthesizer

except ImportError as exc:
    raise ImportError(
        "SDV is required for Notebook 05. "
        "Install the validated SDV version before continuing."
    ) from exc


# --------------------------------------------------------------------------------------------------
# 2. Capture Exact SDV Version
# --------------------------------------------------------------------------------------------------

SDV_VERSION = getattr(sdv, "__version__", None)

if not SDV_VERSION:
    raise RuntimeError(
        "Unable to determine the installed SDV version."
    )

print(f"✓ SDV version : {SDV_VERSION}")
print(f"✓ TVAE class  : {TVAESynthesizer.__name__}")
print("✓ SDV TVAE dependencies loaded.")


# --------------------------------------------------------------------------------------------------
# 3. Validate Required Classes
# --------------------------------------------------------------------------------------------------

if SingleTableMetadata is None:
    raise RuntimeError("SingleTableMetadata is unavailable.")

if TVAESynthesizer is None:
    raise RuntimeError("TVAESynthesizer is unavailable.")

print("✓ SingleTableMetadata available.")
print("✓ TVAESynthesizer available.")


# --------------------------------------------------------------------------------------------------
# 4. Validate SDV Version Against Installed Runtime
# --------------------------------------------------------------------------------------------------

EXPECTED_SDV_VERSION = "1.38.3"

if SDV_VERSION != EXPECTED_SDV_VERSION:
    raise RuntimeError(
        f"Unexpected SDV version: {SDV_VERSION}. "
        f"Expected pinned version: {EXPECTED_SDV_VERSION}."
    )

print(f"✓ SDV version pin verified : {EXPECTED_SDV_VERSION}")


# --------------------------------------------------------------------------------------------------
# 5. Section 7 Integrity Gate
# --------------------------------------------------------------------------------------------------

SECTION_7_CHECKS = {
    "sdv_import": True,
    "sdv_version_available": bool(SDV_VERSION),
    "sdv_version_pin": SDV_VERSION == EXPECTED_SDV_VERSION,
    "metadata_class_available": SingleTableMetadata is not None,
    "tvae_class_available": TVAESynthesizer is not None,
}

print()
print("=" * 100)
print("SECTION 7 — TVAE INITIALIZATION INTEGRITY GATE")
print("=" * 100)

for check_name, check_result in SECTION_7_CHECKS.items():
    print(
        f"{'✓' if check_result else '✗'} "
        f"{check_name:<35} : "
        f"{'PASS' if check_result else 'FAIL'}"
    )

TOTAL_CHECKS = len(SECTION_7_CHECKS)
PASSED_CHECKS = sum(SECTION_7_CHECKS.values())
FAILED_CHECKS = TOTAL_CHECKS - PASSED_CHECKS

print()
print(f"Total checks  : {TOTAL_CHECKS}")
print(f"Passed checks : {PASSED_CHECKS}")
print(f"Failed checks : {FAILED_CHECKS}")

if FAILED_CHECKS > 0:
    raise RuntimeError(
        "SECTION 7 FAILED. TVAE initialization cannot proceed."
    )

print()
print("✓ SDV dependency verified.")
print("✓ Exact SDV version captured.")
print("✓ SDV version pin verified.")
print("✓ TVAE synthesizer verified.")
print("✓ Metadata class verified.")

print()
print("=" * 100)
print("SECTION 7 STATUS: COMPLETE / PASS")
print("=" * 100)

SECTION 7 — INITIALIZE TVAE
✓ SDV version : 1.38.3
✓ TVAE class  : TVAESynthesizer
✓ SDV TVAE dependencies loaded.
✓ SingleTableMetadata available.
✓ TVAESynthesizer available.
✓ SDV version pin verified : 1.38.3

SECTION 7 — TVAE INITIALIZATION INTEGRITY GATE
✓ sdv_import                          : PASS
✓ sdv_version_available               : PASS
✓ sdv_version_pin                     : PASS
✓ metadata_class_available            : PASS
✓ tvae_class_available                : PASS

Total checks  : 5
Passed checks : 5
Failed checks : 0

✓ SDV dependency verified.
✓ Exact SDV version captured.
✓ SDV version pin verified.
✓ TVAE synthesizer verified.
✓ Metadata class verified.

SECTION 7 STATUS: COMPLETE / PASS


In [24]:
# ==================================================================================================
# 8. TRAIN TVAE
# ==================================================================================================

print("=" * 100)
print("SECTION 8 — TRAIN TVAE")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 0. Runtime Imports
# --------------------------------------------------------------------------------------------------

import gc
import inspect
import json
import time

from datetime import datetime, timezone
from pathlib import Path


# --------------------------------------------------------------------------------------------------
# 1. Notebook Identity
# --------------------------------------------------------------------------------------------------

NOTEBOOK_ID = "05"
NOTEBOOK_VERSION = "1.0"


# --------------------------------------------------------------------------------------------------
# 2. Validate Required Dependencies
# --------------------------------------------------------------------------------------------------

required_objects = {
    "PROJECT_ROOT": "PROJECT_ROOT",
    "DATASET_IDS": "DATASET_IDS",
    "TRAINING_DATA": "TRAINING_DATA",
    "TRAINING_GENERATIVE_COLUMNS": "TRAINING_GENERATIVE_COLUMNS",
    "TVAE_CONFIG": "TVAE_CONFIG",
    "TVAE_SEED_REGISTRY": "TVAE_SEED_REGISTRY",
    "seed_everything": "seed_everything",
    "SingleTableMetadata": "SingleTableMetadata",
    "TVAESynthesizer": "TVAESynthesizer",
}

missing_objects = [
    variable_name
    for variable_name in required_objects
    if variable_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "SECTION 8 dependency failure. Missing required objects: "
        + ", ".join(missing_objects)
    )

print("✓ Required Section 8 dependencies verified.")


# --------------------------------------------------------------------------------------------------
# 3. Validate SDV Runtime
# --------------------------------------------------------------------------------------------------

try:
    import sdv
except ImportError as exc:
    raise RuntimeError(
        "SDV is not available. "
        "Install the validated SDV 1.38.3 environment before training."
    ) from exc


SDV_VERSION = getattr(
    sdv,
    "__version__",
    None,
)

EXPECTED_SDV_VERSION = "1.38.3"

if SDV_VERSION != EXPECTED_SDV_VERSION:
    raise RuntimeError(
        f"Unexpected SDV version: {SDV_VERSION}. "
        f"Expected: {EXPECTED_SDV_VERSION}."
    )

print(
    f"✓ SDV version verified : {SDV_VERSION}"
)


# --------------------------------------------------------------------------------------------------
# 4. Validate Notebook 05 Output Directories
# --------------------------------------------------------------------------------------------------

NB05_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "notebook_05"
)

NB05_METADATA_ROOT = (
    NB05_ROOT
    / "metadata"
)

NB05_HISTORY_ROOT = (
    NB05_ROOT
    / "history"
)

NB05_CHECKPOINT_ROOT = (
    NB05_ROOT
    / "checkpoints"
)

NB05_TRAINING_ROOT = (
    NB05_ROOT
    / "training"
)

for directory in [
    NB05_ROOT,
    NB05_METADATA_ROOT,
    NB05_HISTORY_ROOT,
    NB05_CHECKPOINT_ROOT,
    NB05_TRAINING_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

print(
    "✓ Notebook 05 output directories verified."
)

print(
    f"✓ Notebook 05 root : {NB05_ROOT}"
)


# --------------------------------------------------------------------------------------------------
# 5. Validate Dataset Registry
# --------------------------------------------------------------------------------------------------

EXPECTED_DATASET_IDS = {
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
}

if set(DATASET_IDS) != EXPECTED_DATASET_IDS:
    raise RuntimeError(
        "DATASET_IDS does not match the authoritative Notebook 00 dataset registry."
    )

if set(TRAINING_DATA.keys()) != EXPECTED_DATASET_IDS:
    raise RuntimeError(
        "TRAINING_DATA coverage does not match DATASET_IDS."
    )

if set(
    TRAINING_GENERATIVE_COLUMNS.keys()
) != EXPECTED_DATASET_IDS:
    raise RuntimeError(
        "TRAINING_GENERATIVE_COLUMNS coverage does not match DATASET_IDS."
    )

if set(
    TVAE_SEED_REGISTRY.keys()
) != EXPECTED_DATASET_IDS:
    raise RuntimeError(
        "TVAE_SEED_REGISTRY coverage does not match DATASET_IDS."
    )

print(
    f"✓ Authoritative datasets : {len(DATASET_IDS)}"
)


# --------------------------------------------------------------------------------------------------
# 6. Validate TVAE Configuration
# --------------------------------------------------------------------------------------------------

REQUIRED_TVAE_CONFIG_KEYS = {
    "model",
    "embedding_dim",
    "compress_dims",
    "decompress_dims",
    "loss_factor",
    "l2scale",
    "batch_size",
    "epochs",
    "enforce_min_max_values",
    "enforce_rounding",
    "verbose",
}

missing_tvae_config = (
    REQUIRED_TVAE_CONFIG_KEYS
    - set(TVAE_CONFIG.keys())
)

if missing_tvae_config:
    raise RuntimeError(
        "TVAE_CONFIG is missing required keys: "
        + ", ".join(
            sorted(
                missing_tvae_config
            )
        )
    )

if TVAE_CONFIG["model"] != "TVAESynthesizer":
    raise RuntimeError(
        "TVAE_CONFIG['model'] must be 'TVAESynthesizer'."
    )

if int(
    TVAE_CONFIG["epochs"]
) <= 0:
    raise RuntimeError(
        "TVAE epochs must be greater than zero."
    )

if int(
    TVAE_CONFIG["batch_size"]
) <= 0:
    raise RuntimeError(
        "TVAE batch size must be greater than zero."
    )

print(
    "✓ TVAE configuration validated."
)

print(
    f"✓ Epochs     : {TVAE_CONFIG['epochs']}"
)

print(
    f"✓ Batch size : {TVAE_CONFIG['batch_size']}"
)


# --------------------------------------------------------------------------------------------------
# 7. Resolve GPU Configuration
# --------------------------------------------------------------------------------------------------

if "requested_gpu" in TVAE_CONFIG:

    GPU_REQUESTED = bool(
        TVAE_CONFIG["requested_gpu"]
    )

elif "enable_gpu" in TVAE_CONFIG:

    GPU_REQUESTED = bool(
        TVAE_CONFIG["enable_gpu"]
    )

else:

    raise RuntimeError(
        "TVAE_CONFIG must contain 'requested_gpu'."
    )


try:

    import torch

    CUDA_AVAILABLE = bool(
        torch.cuda.is_available()
    )

    GPU_COUNT = (
        torch.cuda.device_count()
        if CUDA_AVAILABLE
        else 0
    )

    PYTORCH_VERSION = str(
        torch.__version__
    )

    CUDA_VERSION = (
        torch.version.cuda
        if CUDA_AVAILABLE
        else None
    )

except Exception as exc:

    raise RuntimeError(
        "Unable to validate PyTorch/CUDA runtime."
    ) from exc


if GPU_REQUESTED and not CUDA_AVAILABLE:

    raise RuntimeError(
        "GPU was requested by TVAE_CONFIG, "
        "but CUDA is unavailable."
    )


GPU_NAMES = []

if CUDA_AVAILABLE:

    for gpu_index in range(
        GPU_COUNT
    ):

        GPU_NAMES.append(
            torch.cuda.get_device_name(
                gpu_index
            )
        )


print(
    f"✓ GPU requested : {GPU_REQUESTED}"
)

print(
    f"✓ CUDA available: {CUDA_AVAILABLE}"
)

print(
    f"✓ GPU count     : {GPU_COUNT}"
)

for gpu_index, gpu_name in enumerate(
    GPU_NAMES
):

    print(
        f"✓ GPU {gpu_index}         : {gpu_name}"
    )


# --------------------------------------------------------------------------------------------------
# 8. Validate TVAE Constructor Compatibility
# --------------------------------------------------------------------------------------------------

TVAE_SIGNATURE = inspect.signature(
    TVAESynthesizer.__init__
)

TVAE_SUPPORTED_PARAMETERS = set(
    TVAE_SIGNATURE.parameters.keys()
)

REQUIRED_TVAE_PARAMETERS = {
    "metadata",
    "embedding_dim",
    "compress_dims",
    "decompress_dims",
    "loss_factor",
    "l2scale",
    "batch_size",
    "epochs",
    "enforce_min_max_values",
    "enforce_rounding",
    "verbose",
    "enable_gpu",
}

missing_tvae_parameters = (
    REQUIRED_TVAE_PARAMETERS
    - TVAE_SUPPORTED_PARAMETERS
)

if missing_tvae_parameters:

    raise RuntimeError(
        "The installed SDV TVAE API does not support required "
        "parameters: "
        + ", ".join(
            sorted(
                missing_tvae_parameters
            )
        )
    )

print(
    "✓ SDV 1.38.3 TVAE constructor compatibility verified."
)


# --------------------------------------------------------------------------------------------------
# 9. Check Supported Randomness Parameter
# --------------------------------------------------------------------------------------------------

RANDOM_STATE_SUPPORTED = (
    "random_state"
    in TVAE_SUPPORTED_PARAMETERS
)

if RANDOM_STATE_SUPPORTED:

    print(
        "✓ TVAE constructor supports random_state."
    )

else:

    print(
        "⚠ TVAE constructor does not expose random_state; "
        "Notebook 00/Section 6 global seed policy remains authoritative."
    )


# --------------------------------------------------------------------------------------------------
# 10. Initialize Result Containers
# --------------------------------------------------------------------------------------------------

TVAE_MODELS = {}

TVAE_TRAINING_RECORDS = []

TVAE_LOSS_HISTORY = {}

TVAE_METADATA_RECORDS = {}

TVAE_CHECKPOINT_RECORDS = {}


# --------------------------------------------------------------------------------------------------
# 11. Dataset-Specific Training Loop
# --------------------------------------------------------------------------------------------------

for dataset_index, dataset_id in enumerate(
    DATASET_IDS
):

    print()
    print("-" * 100)
    print(
        f"Training Dataset : {dataset_id}"
    )
    print("-" * 100)


    # ----------------------------------------------------------------------------------------------
    # 11.1 Dataset-Specific Seed
    # ----------------------------------------------------------------------------------------------

    if dataset_id not in TVAE_SEED_REGISTRY:

        raise RuntimeError(
            f"{dataset_id}: no TVAE seed registered."
        )

    seed = int(
        TVAE_SEED_REGISTRY[
            dataset_id
        ]
    )

    seed_everything(
        seed
    )

    print(
        f"Seed    : {seed}"
    )


    # ----------------------------------------------------------------------------------------------
    # 11.2 Canonical Artifact Paths
    # ----------------------------------------------------------------------------------------------

    metadata_path = (
        NB05_METADATA_ROOT
        / dataset_id
        / "tvae_metadata.json"
    )

    history_path = (
        NB05_HISTORY_ROOT
        / dataset_id
        / "tvae_loss_history.csv"
    )

    checkpoint_directory = (
        NB05_CHECKPOINT_ROOT
        / dataset_id
    )

    checkpoint_model_path = (
        checkpoint_directory
        / "tvae_checkpoint.pkl"
    )

    checkpoint_metadata_path = (
        checkpoint_directory
        / "training_checkpoint.json"
    )


    # ----------------------------------------------------------------------------------------------
    # 11.3 Load TRAIN Data Only
    # ----------------------------------------------------------------------------------------------

    train_df = TRAINING_DATA[
        dataset_id
    ].copy()

    generative_columns = list(
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    missing_columns = [
        column
        for column in generative_columns
        if column not in train_df.columns
    ]

    if missing_columns:

        raise RuntimeError(
            f"{dataset_id}: missing required generative columns: "
            + ", ".join(
                missing_columns
            )
        )

    train_df = train_df[
        generative_columns
    ].copy()

    expected_rows = len(
        train_df
    )

    expected_columns = len(
        train_df.columns
    )

    print(
        f"Rows    : {expected_rows:,}"
    )

    print(
        f"Columns : {expected_columns}"
    )


    # ----------------------------------------------------------------------------------------------
    # 11.4 Validate TRAIN Dataset
    # ----------------------------------------------------------------------------------------------

    if expected_rows == 0:

        raise RuntimeError(
            f"{dataset_id}: TRAIN dataset is empty."
        )

    if train_df.columns.duplicated().any():

        duplicate_columns = (
            train_df.columns[
                train_df.columns.duplicated()
            ]
            .tolist()
        )

        raise RuntimeError(
            f"{dataset_id}: duplicate columns detected: "
            + ", ".join(
                duplicate_columns
            )
        )


    # ----------------------------------------------------------------------------------------------
    # 11.5 Validate Identifier Leakage
    # ----------------------------------------------------------------------------------------------

    if dataset_id == "diabetes_130us":

        forbidden_identifiers = {
            "encounter_id",
            "patient_nbr",
        }

        identifier_leaks = (
            forbidden_identifiers
            & set(train_df.columns)
        )

        if identifier_leaks:

            raise RuntimeError(
                f"{dataset_id}: explicit identifier leakage detected: "
                + ", ".join(
                    sorted(
                        identifier_leaks
                    )
                )
            )


    # ----------------------------------------------------------------------------------------------
    # 11.6 Check for Existing Complete Artifact
    # ----------------------------------------------------------------------------------------------

    existing_checkpoint = (
        checkpoint_model_path.exists()
    )

    existing_history = (
        history_path.exists()
    )

    existing_metadata = (
        metadata_path.exists()
    )


    if (
        existing_checkpoint
        and existing_history
        and existing_metadata
    ):

        print()
        print(
            "✓ Existing TVAE artifacts detected."
        )

        print(
            f"  Checkpoint : {checkpoint_model_path}"
        )

        print(
            f"  History    : {history_path}"
        )

        print(
            f"  Metadata   : {metadata_path}"
        )

        # ------------------------------------------------------------------------------------------
        # 11.6.1 Validate Existing Metadata
        # ------------------------------------------------------------------------------------------

        persisted_metadata = (
            SingleTableMetadata.load_from_json(
                filepath=str(
                    metadata_path
                )
            )
        )

        persisted_metadata.validate()

        persisted_metadata_columns = set(
            persisted_metadata.to_dict()
            .get(
                "columns",
                {}
            )
            .keys()
        )

        current_train_columns = set(
            train_df.columns
        )

        if (
            persisted_metadata_columns
            != current_train_columns
        ):

            raise RuntimeError(
                f"{dataset_id}: existing metadata does not "
                f"match current TRAIN schema."
            )

        print(
            "✓ Existing metadata validated against TRAIN schema."
        )


        # ------------------------------------------------------------------------------------------
        # 11.6.2 Validate Existing Loss History
        # ------------------------------------------------------------------------------------------

        existing_loss_df = pd.read_csv(
            history_path
        )

        if existing_loss_df.empty:

            raise RuntimeError(
                f"{dataset_id}: existing loss history is empty."
            )

        print(
            f"✓ Existing loss history validated "
            f"({len(existing_loss_df):,} records)."
        )


        # ------------------------------------------------------------------------------------------
        # 11.6.3 Reload Existing TVAE Checkpoint
        # ------------------------------------------------------------------------------------------

        try:

            persisted_synthesizer = (
                TVAESynthesizer.load(
                    filepath=str(
                        checkpoint_model_path
                    )
                )
            )

        except Exception as exc:

            raise RuntimeError(
                f"{dataset_id}: existing TVAE checkpoint "
                f"could not be reloaded."
            ) from exc

        print(
            "✓ Existing TVAE checkpoint successfully reloaded."
        )


        # ------------------------------------------------------------------------------------------
        # 11.6.4 Validate Existing Checkpoint Metadata
        # ------------------------------------------------------------------------------------------

        checkpoint_size_bytes = (
            checkpoint_model_path.stat()
            .st_size
        )

        if checkpoint_size_bytes <= 0:

            raise RuntimeError(
                f"{dataset_id}: existing checkpoint is empty."
            )


        # ------------------------------------------------------------------------------------------
        # 11.6.5 Recover Existing Training Metadata
        # ------------------------------------------------------------------------------------------

        recovered_runtime_seconds = None
        recovered_created_utc = None

        if checkpoint_metadata_path.exists():

            try:

                with open(
                    checkpoint_metadata_path,
                    "r",
                    encoding="utf-8",
                ) as handle:

                    existing_checkpoint_record = json.load(
                        handle
                    )

                recovered_runtime_seconds = (
                    existing_checkpoint_record.get(
                        "training_runtime_seconds"
                    )
                )

                recovered_created_utc = (
                    existing_checkpoint_record.get(
                        "created_utc"
                    )
                )

            except Exception:

                recovered_runtime_seconds = None
                recovered_created_utc = None


        # ------------------------------------------------------------------------------------------
        # 11.6.6 Reuse Existing Artifact
        # ------------------------------------------------------------------------------------------

        TVAE_MODELS[
            dataset_id
        ] = {

            "model_path": str(
                checkpoint_model_path
            ),

            "metadata_path": str(
                metadata_path
            ),

            "history_path": str(
                history_path
            ),

            "seed": seed,

            "sdv_version": SDV_VERSION,

            "random_state_supported": (
                RANDOM_STATE_SUPPORTED
            ),

            "artifact_status": "REUSED_EXISTING",
        }


        TVAE_LOSS_HISTORY[
            dataset_id
        ] = existing_loss_df


        TVAE_METADATA_RECORDS[
            dataset_id
        ] = {

            "metadata_path": str(
                metadata_path
            ),

            "training_rows": expected_rows,

            "training_columns": expected_columns,

            "sdv_version": SDV_VERSION,

            "artifact_status": "REUSED_EXISTING",
        }


        checkpoint_record = {

            "notebook_id": NOTEBOOK_ID,

            "notebook_version": NOTEBOOK_VERSION,

            "dataset_id": dataset_id,

            "model": "TVAESynthesizer",

            "sdv_version": SDV_VERSION,

            "training_policy": "TRAIN_ONLY",

            "seed": seed,

            "random_state_supported": (
                RANDOM_STATE_SUPPORTED
            ),

            "random_state_used": (
                seed
                if RANDOM_STATE_SUPPORTED
                else None
            ),

            "training_rows": expected_rows,

            "training_columns": expected_columns,

            "epochs": int(
                TVAE_CONFIG[
                    "epochs"
                ]
            ),

            "batch_size": int(
                TVAE_CONFIG[
                    "batch_size"
                ]
            ),

            "embedding_dim": int(
                TVAE_CONFIG[
                    "embedding_dim"
                ]
            ),

            "compress_dims": list(
                TVAE_CONFIG[
                    "compress_dims"
                ]
            ),

            "decompress_dims": list(
                TVAE_CONFIG[
                    "decompress_dims"
                ]
            ),

            "loss_factor": float(
                TVAE_CONFIG[
                    "loss_factor"
                ]
            ),

            "l2scale": float(
                TVAE_CONFIG[
                    "l2scale"
                ]
            ),

            "enforce_min_max_values": bool(
                TVAE_CONFIG[
                    "enforce_min_max_values"
                ]
            ),

            "enforce_rounding": bool(
                TVAE_CONFIG[
                    "enforce_rounding"
                ]
            ),

            "gpu_requested": GPU_REQUESTED,

            "cuda_available": CUDA_AVAILABLE,

            "gpu_count": GPU_COUNT,

            "gpu_names": GPU_NAMES,

            "pytorch_version": PYTORCH_VERSION,

            "cuda_version": (
                str(
                    CUDA_VERSION
                )
                if CUDA_VERSION
                else None
            ),

            "loss_history_rows": len(
                existing_loss_df
            ),

            "training_runtime_seconds": (
                recovered_runtime_seconds
            ),

            "checkpoint_model_path": str(
                checkpoint_model_path
            ),

            "checkpoint_size_bytes": (
                checkpoint_size_bytes
            ),

            "history_path": str(
                history_path
            ),

            "metadata_path": str(
                metadata_path
            ),

            "artifact_status": "REUSED_EXISTING",

            "created_utc": (
                recovered_created_utc
                if recovered_created_utc
                else datetime.now(
                    timezone.utc
                ).isoformat()
            ),

            "revalidated_utc": datetime.now(
                timezone.utc
            ).isoformat(),

            "status": "PASS",
        }


        TVAE_CHECKPOINT_RECORDS[
            dataset_id
        ] = checkpoint_record


        TVAE_TRAINING_RECORDS.append(
            {

                "dataset_id": dataset_id,

                "seed": seed,

                "random_state_supported": (
                    RANDOM_STATE_SUPPORTED
                ),

                "random_state_used": (
                    seed
                    if RANDOM_STATE_SUPPORTED
                    else None
                ),

                "training_rows": expected_rows,

                "training_columns": expected_columns,

                "epochs": int(
                    TVAE_CONFIG[
                        "epochs"
                    ]
                ),

                "batch_size": int(
                    TVAE_CONFIG[
                        "batch_size"
                    ]
                ),

                "sdv_version": SDV_VERSION,

                "gpu_requested": GPU_REQUESTED,

                "cuda_available": CUDA_AVAILABLE,

                "gpu_count": GPU_COUNT,

                "training_runtime_seconds": (
                    recovered_runtime_seconds
                ),

                "loss_records": len(
                    existing_loss_df
                ),

                "checkpoint_size_bytes": (
                    checkpoint_size_bytes
                ),

                "artifact_status": "REUSED_EXISTING",

                "status": "PASS",
            }
        )


        print()
        print(
            f"✓ Existing {dataset_id} TVAE artifact reused."
        )

        print(
            "✓ No duplicate 300-epoch training performed."
        )


        del train_df
        del persisted_metadata
        del persisted_synthesizer
        del existing_loss_df

        gc.collect()

        continue


    # ----------------------------------------------------------------------------------------------
    # 11.7 Build SDV Metadata From TRAIN Only
    # ----------------------------------------------------------------------------------------------

    metadata = SingleTableMetadata()

    metadata.detect_from_dataframe(
        data=train_df
    )

    metadata.validate()

    metadata_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    if metadata_path.exists():

        persisted_metadata = (
            SingleTableMetadata.load_from_json(
                filepath=str(
                    metadata_path
                )
            )
        )

        persisted_metadata.validate()

        persisted_metadata_columns = set(
            persisted_metadata.to_dict()
            .get(
                "columns",
                {}
            )
            .keys()
        )

        if persisted_metadata_columns != set(
            train_df.columns
        ):

            raise RuntimeError(
                f"{dataset_id}: existing metadata "
                f"does not match TRAIN schema."
            )

        metadata = persisted_metadata

        print(
            "✓ Existing canonical metadata loaded and validated."
        )

    else:

        metadata.save_to_json(
            filepath=str(
                metadata_path
            )
        )

        print(
            f"✓ Metadata saved : {metadata_path}"
        )


    # ----------------------------------------------------------------------------------------------
    # 11.8 Final Metadata Schema Validation
    # ----------------------------------------------------------------------------------------------

    metadata_columns = set(
        metadata.to_dict()
        .get(
            "columns",
            {}
        )
        .keys()
    )

    if metadata_columns != set(
        train_df.columns
    ):

        raise RuntimeError(
            f"{dataset_id}: SDV metadata does not match "
            f"TRAIN generative schema."
        )

    print(
        "✓ Persisted SDV metadata matches TRAIN generative schema."
    )


    # ----------------------------------------------------------------------------------------------
    # 11.9 Initialize TVAE
    # ----------------------------------------------------------------------------------------------

    synthesizer_kwargs = {

        "embedding_dim": TVAE_CONFIG[
            "embedding_dim"
        ],

        "compress_dims": TVAE_CONFIG[
            "compress_dims"
        ],

        "decompress_dims": TVAE_CONFIG[
            "decompress_dims"
        ],

        "loss_factor": TVAE_CONFIG[
            "loss_factor"
        ],

        "l2scale": TVAE_CONFIG[
            "l2scale"
        ],

        "batch_size": TVAE_CONFIG[
            "batch_size"
        ],

        "epochs": TVAE_CONFIG[
            "epochs"
        ],

        "enforce_min_max_values": TVAE_CONFIG[
            "enforce_min_max_values"
        ],

        "enforce_rounding": TVAE_CONFIG[
            "enforce_rounding"
        ],

        "verbose": TVAE_CONFIG[
            "verbose"
        ],

        "enable_gpu": GPU_REQUESTED,
    }


    # ----------------------------------------------------------------------------------------------
    # 11.10 Pass Explicit Seed Only If Supported
    # ----------------------------------------------------------------------------------------------

    if RANDOM_STATE_SUPPORTED:

        synthesizer_kwargs[
            "random_state"
        ] = seed


    # ----------------------------------------------------------------------------------------------
    # 11.11 Validate Constructor Arguments
    # ----------------------------------------------------------------------------------------------

    unsupported_parameters = {
        parameter
        for parameter in synthesizer_kwargs
        if parameter
        not in TVAE_SUPPORTED_PARAMETERS
    }

    if unsupported_parameters:

        raise RuntimeError(
            f"{dataset_id}: unsupported TVAE constructor "
            f"parameters: "
            + ", ".join(
                sorted(
                    unsupported_parameters
                )
            )
        )


    # ----------------------------------------------------------------------------------------------
    # 11.12 Initialize Synthesizer
    # ----------------------------------------------------------------------------------------------

    synthesizer = TVAESynthesizer(
        metadata,
        **synthesizer_kwargs,
    )

    print(
        "✓ TVAE synthesizer initialized."
    )

    if RANDOM_STATE_SUPPORTED:

        print(
            f"✓ TVAE random_state : {seed}"
        )


    # ----------------------------------------------------------------------------------------------
    # 11.13 Train
    # ----------------------------------------------------------------------------------------------

    print(
        "Starting TVAE training..."
    )

    train_start = time.perf_counter()

    synthesizer.fit(
        train_df
    )

    train_end = time.perf_counter()

    training_runtime_seconds = (
        train_end
        - train_start
    )

    print()
    print(
        f"✓ Training completed in "
        f"{training_runtime_seconds:.2f} seconds"
    )


    # ----------------------------------------------------------------------------------------------
    # 11.14 Retrieve Loss History
    # ----------------------------------------------------------------------------------------------

    loss_df = (
        synthesizer.get_loss_values()
    )

    if not isinstance(
        loss_df,
        pd.DataFrame
    ):

        loss_df = pd.DataFrame(
            loss_df
        )

    loss_df = loss_df.copy()

    if loss_df.empty:

        raise RuntimeError(
            f"{dataset_id}: TVAE returned an empty loss history."
        )

    TVAE_LOSS_HISTORY[
        dataset_id
    ] = loss_df

    print(
        f"✓ Loss records : {len(loss_df)}"
    )


    # ----------------------------------------------------------------------------------------------
    # 11.15 Save Loss History
    # ----------------------------------------------------------------------------------------------

    history_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    loss_df.to_csv(
        history_path,
        index=False,
    )

    if not history_path.exists():

        raise RuntimeError(
            f"{dataset_id}: loss history was not persisted."
        )

    print(
        f"✓ History saved : {history_path}"
    )


    # ----------------------------------------------------------------------------------------------
    # 11.16 Save TVAE Checkpoint
    # ----------------------------------------------------------------------------------------------

    checkpoint_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    synthesizer.save(
        filepath=str(
            checkpoint_model_path
        )
    )

    if not checkpoint_model_path.exists():

        raise RuntimeError(
            f"{dataset_id}: TVAE checkpoint was not persisted."
        )

    print(
        f"✓ Checkpoint saved : "
        f"{checkpoint_model_path}"
    )


    # ----------------------------------------------------------------------------------------------
    # 11.17 Validate Checkpoint Size
    # ----------------------------------------------------------------------------------------------

    checkpoint_size_bytes = (
        checkpoint_model_path.stat()
        .st_size
    )

    if checkpoint_size_bytes <= 0:

        raise RuntimeError(
            f"{dataset_id}: checkpoint file is empty."
        )


    # ----------------------------------------------------------------------------------------------
    # 11.18 Save Training Checkpoint Metadata
    # ----------------------------------------------------------------------------------------------

    checkpoint_record = {

        "notebook_id": NOTEBOOK_ID,

        "notebook_version": NOTEBOOK_VERSION,

        "dataset_id": dataset_id,

        "model": "TVAESynthesizer",

        "sdv_version": SDV_VERSION,

        "training_policy": "TRAIN_ONLY",

        "seed": seed,

        "random_state_supported": (
            RANDOM_STATE_SUPPORTED
        ),

        "random_state_used": (
            seed
            if RANDOM_STATE_SUPPORTED
            else None
        ),

        "training_rows": expected_rows,

        "training_columns": expected_columns,

        "epochs": int(
            TVAE_CONFIG[
                "epochs"
            ]
        ),

        "batch_size": int(
            TVAE_CONFIG[
                "batch_size"
            ]
        ),

        "embedding_dim": int(
            TVAE_CONFIG[
                "embedding_dim"
            ]
        ),

        "compress_dims": list(
            TVAE_CONFIG[
                "compress_dims"
            ]
        ),

        "decompress_dims": list(
            TVAE_CONFIG[
                "decompress_dims"
            ]
        ),

        "loss_factor": float(
            TVAE_CONFIG[
                "loss_factor"
            ]
        ),

        "l2scale": float(
            TVAE_CONFIG[
                "l2scale"
            ]
        ),

        "enforce_min_max_values": bool(
            TVAE_CONFIG[
                "enforce_min_max_values"
            ]
        ),

        "enforce_rounding": bool(
            TVAE_CONFIG[
                "enforce_rounding"
            ]
        ),

        "gpu_requested": GPU_REQUESTED,

        "cuda_available": CUDA_AVAILABLE,

        "gpu_count": GPU_COUNT,

        "gpu_names": GPU_NAMES,

        "pytorch_version": PYTORCH_VERSION,

        "cuda_version": (
            str(
                CUDA_VERSION
            )
            if CUDA_VERSION
            else None
        ),

        "loss_history_rows": len(
            loss_df
        ),

        "training_runtime_seconds": (
            training_runtime_seconds
        ),

        "checkpoint_model_path": str(
            checkpoint_model_path
        ),

        "checkpoint_size_bytes": (
            checkpoint_size_bytes
        ),

        "history_path": str(
            history_path
        ),

        "metadata_path": str(
            metadata_path
        ),

        "artifact_status": "NEWLY_TRAINED",

        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),

        "status": "PASS",
    }


    with open(
        checkpoint_metadata_path,
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            checkpoint_record,
            handle,
            indent=2,
        )


    if not checkpoint_metadata_path.exists():

        raise RuntimeError(
            f"{dataset_id}: checkpoint metadata was not persisted."
        )


    TVAE_CHECKPOINT_RECORDS[
        dataset_id
    ] = checkpoint_record


    # ----------------------------------------------------------------------------------------------
    # 11.19 Training Record
    # ----------------------------------------------------------------------------------------------

    TVAE_TRAINING_RECORDS.append(
        {

            "dataset_id": dataset_id,

            "seed": seed,

            "random_state_supported": (
                RANDOM_STATE_SUPPORTED
            ),

            "random_state_used": (
                seed
                if RANDOM_STATE_SUPPORTED
                else None
            ),

            "training_rows": expected_rows,

            "training_columns": expected_columns,

            "epochs": int(
                TVAE_CONFIG[
                    "epochs"
                ]
            ),

            "batch_size": int(
                TVAE_CONFIG[
                    "batch_size"
                ]
            ),

            "sdv_version": SDV_VERSION,

            "gpu_requested": GPU_REQUESTED,

            "cuda_available": CUDA_AVAILABLE,

            "gpu_count": GPU_COUNT,

            "training_runtime_seconds": (
                training_runtime_seconds
            ),

            "loss_records": len(
                loss_df
            ),

            "checkpoint_size_bytes": (
                checkpoint_size_bytes
            ),

            "artifact_status": "NEWLY_TRAINED",

            "status": "PASS",
        }
    )


    # ----------------------------------------------------------------------------------------------
    # 11.20 Metadata Record
    # ----------------------------------------------------------------------------------------------

    TVAE_METADATA_RECORDS[
        dataset_id
    ] = {

        "metadata_path": str(
            metadata_path
        ),

        "training_rows": expected_rows,

        "training_columns": expected_columns,

        "sdv_version": SDV_VERSION,

        "artifact_status": "NEWLY_TRAINED",
    }


    # ----------------------------------------------------------------------------------------------
    # 11.21 Lightweight Model Reference
    # ----------------------------------------------------------------------------------------------

    TVAE_MODELS[
        dataset_id
    ] = {

        "model_path": str(
            checkpoint_model_path
        ),

        "metadata_path": str(
            metadata_path
        ),

        "history_path": str(
            history_path
        ),

        "seed": seed,

        "sdv_version": SDV_VERSION,

        "random_state_supported": (
            RANDOM_STATE_SUPPORTED
        ),

        "artifact_status": "NEWLY_TRAINED",
    }


    # ----------------------------------------------------------------------------------------------
    # 11.22 Release Temporary Objects
    # ----------------------------------------------------------------------------------------------

    del train_df
    del metadata
    del synthesizer
    del loss_df

    gc.collect()

    print(
        f"✓ Memory cleanup completed for {dataset_id}."
    )


# --------------------------------------------------------------------------------------------------
# 12. Final Training Coverage Validation
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 8 — TRAINING COVERAGE VALIDATION")
print("=" * 100)


trained_dataset_ids = {
    record[
        "dataset_id"
    ]
    for record
    in TVAE_TRAINING_RECORDS
}

expected_dataset_ids = set(
    DATASET_IDS
)

if trained_dataset_ids != expected_dataset_ids:

    missing_datasets = (
        expected_dataset_ids
        - trained_dataset_ids
    )

    unexpected_datasets = (
        trained_dataset_ids
        - expected_dataset_ids
    )

    raise RuntimeError(
        "SECTION 8 training coverage failure. "
        f"Missing: {sorted(missing_datasets)}; "
        f"Unexpected: {sorted(unexpected_datasets)}."
    )


# --------------------------------------------------------------------------------------------------
# 13. Final Artifact Coverage Validation
# --------------------------------------------------------------------------------------------------

if len(
    TVAE_CHECKPOINT_RECORDS
) != len(DATASET_IDS):

    raise RuntimeError(
        "SECTION 8 checkpoint coverage is incomplete."
    )

if len(
    TVAE_METADATA_RECORDS
) != len(DATASET_IDS):

    raise RuntimeError(
        "SECTION 8 metadata coverage is incomplete."
    )

if len(
    TVAE_LOSS_HISTORY
) != len(DATASET_IDS):

    raise RuntimeError(
        "SECTION 8 loss-history coverage is incomplete."
    )


print(
    f"✓ Datasets expected   : {len(DATASET_IDS)}"
)

print(
    f"✓ Datasets processed  : {len(TVAE_TRAINING_RECORDS)}"
)

print(
    f"✓ Checkpoints         : {len(TVAE_CHECKPOINT_RECORDS)}"
)

print(
    f"✓ Metadata artifacts  : {len(TVAE_METADATA_RECORDS)}"
)

print(
    f"✓ Loss histories      : {len(TVAE_LOSS_HISTORY)}"
)


# --------------------------------------------------------------------------------------------------
# 14. Final File-Level Validation
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    checkpoint_record = (
        TVAE_CHECKPOINT_RECORDS[
            dataset_id
        ]
    )

    checkpoint_path = Path(
        checkpoint_record[
            "checkpoint_model_path"
        ]
    )

    metadata_path = Path(
        checkpoint_record[
            "metadata_path"
        ]
    )

    history_path = Path(
        checkpoint_record[
            "history_path"
        ]
    )

    if not checkpoint_path.exists():

        raise RuntimeError(
            f"{dataset_id}: checkpoint file missing."
        )

    if not metadata_path.exists():

        raise RuntimeError(
            f"{dataset_id}: metadata file missing."
        )

    if not history_path.exists():

        raise RuntimeError(
            f"{dataset_id}: history file missing."
        )

    if checkpoint_path.stat().st_size <= 0:

        raise RuntimeError(
            f"{dataset_id}: checkpoint file is empty."
        )

    if metadata_path.stat().st_size <= 0:

        raise RuntimeError(
            f"{dataset_id}: metadata file is empty."
        )

    if history_path.stat().st_size <= 0:

        raise RuntimeError(
            f"{dataset_id}: history file is empty."
        )

print(
    "✓ All persisted TVAE artifact files verified."
)


# --------------------------------------------------------------------------------------------------
# 15. Final Section 8 Integrity Gate
# --------------------------------------------------------------------------------------------------

SECTION_8_CHECKS = {

    "all_datasets_processed": (
        trained_dataset_ids
        == expected_dataset_ids
    ),

    "training_records_complete": (
        len(
            TVAE_TRAINING_RECORDS
        )
        == len(DATASET_IDS)
    ),

    "checkpoint_records_complete": (
        len(
            TVAE_CHECKPOINT_RECORDS
        )
        == len(DATASET_IDS)
    ),

    "metadata_records_complete": (
        len(
            TVAE_METADATA_RECORDS
        )
        == len(DATASET_IDS)
    ),

    "loss_history_complete": (
        len(
            TVAE_LOSS_HISTORY
        )
        == len(DATASET_IDS)
    ),

    "sdv_version_consistent": (
        all(
            record[
                "sdv_version"
            ]
            == EXPECTED_SDV_VERSION
            for record
            in TVAE_TRAINING_RECORDS
        )
    ),

    "train_only_policy": (
        all(
            checkpoint_record.get(
                "training_policy"
            )
            == "TRAIN_ONLY"
            for checkpoint_record
            in TVAE_CHECKPOINT_RECORDS.values()
        )
    ),

    "positive_training_rows": (
        all(
            record[
                "training_rows"
            ] > 0
            for record
            in TVAE_TRAINING_RECORDS
        )
    ),

    "positive_loss_records": (
        all(
            record[
                "loss_records"
            ] > 0
            for record
            in TVAE_TRAINING_RECORDS
        )
    ),

    "checkpoint_files_exist": (
        all(
            Path(
                record[
                    "checkpoint_model_path"
                ]
            ).exists()
            for record
            in TVAE_CHECKPOINT_RECORDS.values()
        )
    ),

    "metadata_files_exist": (
        all(
            Path(
                record[
                    "metadata_path"
                ]
            ).exists()
            for record
            in TVAE_CHECKPOINT_RECORDS.values()
        )
    ),

    "history_files_exist": (
        all(
            Path(
                record[
                    "history_path"
                ]
            ).exists()
            for record
            in TVAE_CHECKPOINT_RECORDS.values()
        )
    ),

    "diabetes_identifiers_excluded": (
        not (
            {
                "encounter_id",
                "patient_nbr",
            }
            & set(
                TRAINING_GENERATIVE_COLUMNS[
                    "diabetes_130us"
                ]
            )
        )
    ),

    "notebook_identity_recorded": (
        NOTEBOOK_ID == "05"
        and bool(
            NOTEBOOK_VERSION
        )
    ),

    "sdv_runtime_recorded": (
        SDV_VERSION == EXPECTED_SDV_VERSION
    ),

    "gpu_runtime_recorded": (
        isinstance(
            GPU_NAMES,
            list
        )
    ),
}


print()
print("=" * 100)
print("SECTION 8 — TVAE TRAINING INTEGRITY GATE")
print("=" * 100)


for check_name, check_result in (
    SECTION_8_CHECKS.items()
):

    print(
        f"{'✓' if check_result else '✗'} "
        f"{check_name:<42} : "
        f"{'PASS' if check_result else 'FAIL'}"
    )


TOTAL_CHECKS = len(
    SECTION_8_CHECKS
)

PASSED_CHECKS = sum(
    SECTION_8_CHECKS.values()
)

FAILED_CHECKS = (
    TOTAL_CHECKS
    - PASSED_CHECKS
)


print()
print(
    f"Total checks  : {TOTAL_CHECKS}"
)

print(
    f"Passed checks : {PASSED_CHECKS}"
)

print(
    f"Failed checks : {FAILED_CHECKS}"
)


if FAILED_CHECKS > 0:

    raise RuntimeError(
        "SECTION 8 FAILED. "
        "TVAE training integrity requirements were not satisfied."
    )


# --------------------------------------------------------------------------------------------------
# 16. Final TVAE Training Summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 8 — FINAL TVAE TRAINING SUMMARY")
print("=" * 100)


for record in TVAE_TRAINING_RECORDS:

    runtime_value = record[
        "training_runtime_seconds"
    ]

    runtime_text = (
        f"{runtime_value:.2f}s"
        if runtime_value is not None
        else "RECOVERED_ARTIFACT"
    )

    print(
        f"{record['dataset_id']:<20} | "
        f"Rows={record['training_rows']:,} | "
        f"Columns={record['training_columns']} | "
        f"Epochs={record['epochs']} | "
        f"Seed={record['seed']} | "
        f"Runtime={runtime_text} | "
        f"Loss records={record['loss_records']} | "
        f"Artifact={record['artifact_status']} | "
        f"PASS"
    )


print()
print("✓ All authoritative TRAIN datasets processed.")
print("✓ TVAE checkpoints available for all datasets.")
print("✓ TRAIN-only policy maintained.")
print("✓ SDV metadata persisted and validated.")
print("✓ TVAE loss histories persisted and validated.")
print("✓ TVAE checkpoints persisted/revalidated.")
print("✓ Existing valid artifacts were reused where available.")
print("✓ No unnecessary duplicate TVAE training performed.")
print("✓ Training provenance recorded.")
print("✓ GPU/runtime provenance recorded.")
print("✓ Dataset coverage verified.")
print("✓ Identifier leakage policy verified.")
print("✓ Notebook identity recorded.")
print("✓ Section 8 integrity gate passed.")

print()
print("=" * 100)
print("SECTION 8 STATUS: COMPLETE / PASS")
print("=" * 100)

SECTION 8 — TRAIN TVAE
✓ Required Section 8 dependencies verified.
✓ SDV version verified : 1.38.3
✓ Notebook 05 output directories verified.
✓ Notebook 05 root : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05
✓ Authoritative datasets : 3
✓ TVAE configuration validated.
✓ Epochs     : 300
✓ Batch size : 500
✓ GPU requested : True
✓ CUDA available: True
✓ GPU count     : 1
✓ GPU 0         : Tesla T4
✓ SDV 1.38.3 TVAE constructor compatibility verified.
⚠ TVAE constructor does not expose random_state; Notebook 00/Section 6 global seed policy remains authoritative.

----------------------------------------------------------------------------------------------------
Training Dataset : adult_income
----------------------------------------------------------------------------------------------------
Seed    : 3126
Rows    : 34,189
Columns : 15

✓ Existing TVAE artifacts detected.
  Checkpoint : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05/checkp

/usr/local/lib/python3.13/dist-packages/sdv/_utils.py:514: FutureWarning: The 'load' function will be deprecated in future versions of SDV. Please use 'utils.load_synthesizer' instead.
  warnings.warn(


✓ Existing TVAE checkpoint successfully reloaded.

✓ Existing adult_income TVAE artifact reused.
✓ No duplicate 300-epoch training performed.

----------------------------------------------------------------------------------------------------
Training Dataset : bank_marketing
----------------------------------------------------------------------------------------------------
Seed    : 3226
Rows    : 31,647
Columns : 17
✓ Metadata saved : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05/metadata/bank_marketing/tvae_metadata.json
✓ Persisted SDV metadata matches TRAIN generative schema.
✓ TVAE synthesizer initialized.
Starting TVAE training...


/usr/local/lib/python3.13/dist-packages/sdv/single_table/base.py:183: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)



✓ Training completed in 299.23 seconds
✓ Loss records : 19200
✓ History saved : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05/history/bank_marketing/tvae_loss_history.csv
✓ Checkpoint saved : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05/checkpoints/bank_marketing/tvae_checkpoint.pkl
✓ Memory cleanup completed for bank_marketing.

----------------------------------------------------------------------------------------------------
Training Dataset : diabetes_130us
----------------------------------------------------------------------------------------------------
Seed    : 3326
Rows    : 71,236
Columns : 48
✓ Metadata saved : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05/metadata/diabetes_130us/tvae_metadata.json
✓ Persisted SDV metadata matches TRAIN generative schema.
✓ TVAE synthesizer initialized.
Starting TVAE training...


/usr/local/lib/python3.13/dist-packages/sdv/single_table/base.py:183: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)



✓ Training completed in 936.16 seconds
✓ Loss records : 42900
✓ History saved : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05/history/diabetes_130us/tvae_loss_history.csv
✓ Checkpoint saved : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05/checkpoints/diabetes_130us/tvae_checkpoint.pkl
✓ Memory cleanup completed for diabetes_130us.

SECTION 8 — TRAINING COVERAGE VALIDATION
✓ Datasets expected   : 3
✓ Datasets processed  : 3
✓ Checkpoints         : 3
✓ Metadata artifacts  : 3
✓ Loss histories      : 3
✓ All persisted TVAE artifact files verified.

SECTION 8 — TVAE TRAINING INTEGRITY GATE
✓ all_datasets_processed                     : PASS
✓ training_records_complete                  : PASS
✓ checkpoint_records_complete                : PASS
✓ metadata_records_complete                  : PASS
✓ loss_history_complete                      : PASS
✓ sdv_version_consistent                     : PASS
✓ train_only_policy                          : 

In [27]:
# ==================================================================================================
# 9. RECORD TRAINING HISTORY
# ==================================================================================================

print("=" * 100)
print("SECTION 9 — RECORD TRAINING HISTORY")
print("=" * 100)

# -----------------------------------------------------------------------------------------------
# 1. Build consolidated TVAE training-history DataFrame
# -----------------------------------------------------------------------------------------------

TVAE_TRAINING_HISTORY_DF = pd.DataFrame(
    TVAE_TRAINING_RECORDS
)

# -----------------------------------------------------------------------------------------------
# 2. Validate dataset-level record coverage
# -----------------------------------------------------------------------------------------------

if len(TVAE_TRAINING_HISTORY_DF) != len(DATASET_IDS):

    raise RuntimeError(
        "TVAE training history does not contain "
        "exactly one record per dataset."
    )

if "dataset_id" not in TVAE_TRAINING_HISTORY_DF.columns:

    raise RuntimeError(
        "TVAE training history is missing the required "
        "'dataset_id' column."
    )

if (
    set(TVAE_TRAINING_HISTORY_DF["dataset_id"])
    != set(DATASET_IDS)
):

    raise RuntimeError(
        "TVAE training history dataset coverage does not "
        "match the authoritative DATASET_IDS."
    )

# -----------------------------------------------------------------------------------------------
# 3. Validate training status
# -----------------------------------------------------------------------------------------------

if "status" not in TVAE_TRAINING_HISTORY_DF.columns:

    raise RuntimeError(
        "TVAE training history is missing the required "
        "'status' column."
    )

if (
    TVAE_TRAINING_HISTORY_DF["status"] != "PASS"
).any():

    failed_datasets = (
        TVAE_TRAINING_HISTORY_DF.loc[
            TVAE_TRAINING_HISTORY_DF["status"] != "PASS",
            "dataset_id",
        ]
        .tolist()
    )

    raise RuntimeError(
        "One or more TVAE training records are not PASS: "
        f"{failed_datasets}"
    )

# -----------------------------------------------------------------------------------------------
# 4. Validate core training provenance fields
# -----------------------------------------------------------------------------------------------

REQUIRED_TRAINING_HISTORY_COLUMNS = [
    "dataset_id",
    "seed",
    "epochs",
    "status",
]

missing_columns = [
    column
    for column in REQUIRED_TRAINING_HISTORY_COLUMNS
    if column not in TVAE_TRAINING_HISTORY_DF.columns
]

if missing_columns:

    raise RuntimeError(
        "TVAE training history is missing required columns: "
        f"{missing_columns}"
    )

# -----------------------------------------------------------------------------------------------
# 5. Validate seed and epoch values
# -----------------------------------------------------------------------------------------------

if (
    TVAE_TRAINING_HISTORY_DF["seed"]
    .isna()
    .any()
):

    raise RuntimeError(
        "One or more TVAE training records have a missing seed."
    )

if (
    TVAE_TRAINING_HISTORY_DF["epochs"]
    .isna()
    .any()
):

    raise RuntimeError(
        "One or more TVAE training records have a missing epoch count."
    )

if (
    TVAE_TRAINING_HISTORY_DF["epochs"].astype(int)
    != int(TVAE_CONFIG["epochs"])
).any():

    raise RuntimeError(
        "One or more TVAE training records do not match "
        f"the authoritative epoch configuration "
        f"({TVAE_CONFIG['epochs']})."
    )

# -----------------------------------------------------------------------------------------------
# 6. Validate canonical loss-history artifacts
# -----------------------------------------------------------------------------------------------

if len(TVAE_CHECKPOINT_RECORDS) != len(DATASET_IDS):

    raise RuntimeError(
        "TVAE checkpoint records do not contain exactly "
        "one record per authoritative dataset."
    )

for dataset_id in DATASET_IDS:

    if dataset_id not in TVAE_CHECKPOINT_RECORDS:

        raise RuntimeError(
            f"{dataset_id}: checkpoint record is missing."
        )

    history_path = Path(
        TVAE_CHECKPOINT_RECORDS[
            dataset_id
        ]["history_path"]
    )

    if not history_path.exists():

        raise RuntimeError(
            f"{dataset_id}: loss-history file missing."
        )

    if history_path.stat().st_size <= 0:

        raise RuntimeError(
            f"{dataset_id}: loss-history file is empty."
        )

# -----------------------------------------------------------------------------------------------
# 7. Validate loss-record counts where available
# -----------------------------------------------------------------------------------------------

if "loss_records" in TVAE_TRAINING_HISTORY_DF.columns:

    if (
        TVAE_TRAINING_HISTORY_DF["loss_records"]
        .isna()
        .any()
    ):

        raise RuntimeError(
            "One or more TVAE training records have "
            "a missing loss-record count."
        )

    if (
        TVAE_TRAINING_HISTORY_DF["loss_records"].astype(int)
        <= 0
    ).any():

        raise RuntimeError(
            "One or more TVAE training records contain "
            "a non-positive loss-record count."
        )

# -----------------------------------------------------------------------------------------------
# 8. Persist consolidated training summary
# -----------------------------------------------------------------------------------------------

TVAE_TRAINING_SUMMARY_PATH = (
    NB05_HISTORY_ROOT
    / "tvae_training_summary.csv"
)

TVAE_TRAINING_HISTORY_DF.to_csv(
    TVAE_TRAINING_SUMMARY_PATH,
    index=False,
)

if not TVAE_TRAINING_SUMMARY_PATH.exists():

    raise RuntimeError(
        "TVAE training summary was not persisted."
    )

if TVAE_TRAINING_SUMMARY_PATH.stat().st_size <= 0:

    raise RuntimeError(
        "TVAE training summary file is empty."
    )

# -----------------------------------------------------------------------------------------------
# 9. Final output
# -----------------------------------------------------------------------------------------------

print()

print(
    TVAE_TRAINING_HISTORY_DF.to_string(
        index=False
    )
)

print()

print(
    f"✓ Training summary saved : "
    f"{TVAE_TRAINING_SUMMARY_PATH}"
)

print(
    "✓ Dataset coverage validated."
)

print(
    "✓ Training status validated."
)

print(
    "✓ Seed provenance validated."
)

print(
    "✓ Epoch configuration validated."
)

print(
    "✓ Loss-history artifacts validated."
)

print(
    "✓ Loss-record coverage validated."
)

print(
    "✓ Consolidated TVAE training history validated."
)

print()
print(
    "SECTION 9 STATUS: COMPLETE / PASS"
)

SECTION 9 — RECORD TRAINING HISTORY

    dataset_id  seed  random_state_supported random_state_used  training_rows  training_columns  epochs  batch_size sdv_version  gpu_requested  cuda_available  gpu_count  training_runtime_seconds  loss_records  checkpoint_size_bytes artifact_status status
  adult_income  3126                   False              None          34189                15     300         500      1.38.3           True            True          1                       NaN         20700                1078122 REUSED_EXISTING   PASS
bank_marketing  3226                   False              None          31647                17     300         500      1.38.3           True            True          1                299.232635         19200                1057925   NEWLY_TRAINED   PASS
diabetes_130us  3326                   False              None          71236                48     300         500      1.38.3           True            True          1                936.156400

In [30]:
# ==================================================================================================
# 10. SAVE CHECKPOINTS
# ==================================================================================================

print("=" * 100)
print("SECTION 10 — SAVE CHECKPOINTS")
print("=" * 100)

# -----------------------------------------------------------------------------------------------
# 0. SHA-256 helper
# -----------------------------------------------------------------------------------------------

import hashlib
import json
from pathlib import Path

def calculate_sha256(
    file_path,
    chunk_size=1024 * 1024,
):

    file_path = Path(file_path)

    sha256 = hashlib.sha256()

    with open(
        file_path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            sha256.update(
                chunk
            )

    return sha256.hexdigest()


print(
    "✓ SHA-256 helper verified."
)

# -----------------------------------------------------------------------------------------------
# 1. Validate required Section 10 dependencies
# -----------------------------------------------------------------------------------------------

required_objects = [
    "DATASET_IDS",
    "TVAE_CHECKPOINT_RECORDS",
    "TVAE_TRAINING_HISTORY_DF",
    "NB05_CHECKPOINT_ROOT",
    "NOTEBOOK_ID",
    "NOTEBOOK_VERSION",
    "SDV_VERSION",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:

    raise RuntimeError(
        "Section 10 is missing required objects: "
        f"{missing_objects}"
    )

# -----------------------------------------------------------------------------------------------
# 2. Initialize validation records
# -----------------------------------------------------------------------------------------------

TVAE_CHECKPOINT_VALIDATION_RECORDS = []

# -----------------------------------------------------------------------------------------------
# 3. Validate checkpoint artifacts
# -----------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    if dataset_id not in TVAE_CHECKPOINT_RECORDS:

        raise RuntimeError(
            f"{dataset_id}: TVAE checkpoint record is missing."
        )

    checkpoint_record = (
        TVAE_CHECKPOINT_RECORDS[
            dataset_id
        ]
    )

    # -------------------------------------------------------------------------------------------
    # Checkpoint model
    # -------------------------------------------------------------------------------------------

    checkpoint_model_path = Path(
        checkpoint_record[
            "checkpoint_model_path"
        ]
    )

    if not checkpoint_model_path.exists():

        raise FileNotFoundError(
            f"{dataset_id}: TVAE checkpoint model missing."
        )

    checkpoint_model_size = (
        checkpoint_model_path.stat().st_size
    )

    if checkpoint_model_size <= 0:

        raise RuntimeError(
            f"{dataset_id}: TVAE checkpoint model is empty."
        )

    # -------------------------------------------------------------------------------------------
    # Checkpoint metadata
    # -------------------------------------------------------------------------------------------

    checkpoint_metadata_path = (
        NB05_CHECKPOINT_ROOT
        / dataset_id
        / "training_checkpoint.json"
    )

    checkpoint_metadata_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    # -------------------------------------------------------------------------------------------
    # Recover missing metadata for previously validated existing artifacts
    # -------------------------------------------------------------------------------------------

    if not checkpoint_metadata_path.exists():

        artifact_status = (
            checkpoint_record.get(
                "artifact_status"
            )
        )

        if artifact_status != "REUSED_EXISTING":

            raise FileNotFoundError(
                f"{dataset_id}: checkpoint metadata missing."
            )

        print()
        print(
            f"⚠ {dataset_id}: training_checkpoint.json missing."
        )

        print(
            "  Existing TVAE model/history/metadata artifacts "
            "were already validated in Section 8."
        )

        print(
            "  Recovering checkpoint provenance metadata "
            "without retraining."
        )

        training_rows = (
            TVAE_TRAINING_HISTORY_DF[
                TVAE_TRAINING_HISTORY_DF[
                    "dataset_id"
                ] == dataset_id
            ]
        )

        if len(training_rows) != 1:

            raise RuntimeError(
                f"{dataset_id}: expected exactly one "
                "training-history record for recovery."
            )

        training_row = (
            training_rows.iloc[0]
            .to_dict()
        )

        recovered_checkpoint_metadata = {
            "notebook_id": NOTEBOOK_ID,
            "notebook_version": NOTEBOOK_VERSION,
            "dataset_id": dataset_id,
            "artifact_status": "REUSED_EXISTING",
            "status": "PASS",
            "recovery_status": "RECOVERED_PROVENANCE",
            "recovery_reason": (
                "Existing valid TVAE checkpoint was created successfully "
                "before a prior post-training provenance-writing failure."
            ),
            "training_policy": "native_train_only",
            "validation_used": False,
            "test_used": False,
            "differential_privacy": False,
            "statistical_guidance": False,
            "sppgan_components": False,
            "seed": int(
                training_row["seed"]
            ),
            "epochs": int(
                training_row["epochs"]
            ),
            "batch_size": int(
                training_row["batch_size"]
            ),
            "training_rows": int(
                training_row["training_rows"]
            ),
            "training_columns": int(
                training_row["training_columns"]
            ),
            "sdv_version": str(
                training_row["sdv_version"]
            ),
            "gpu_requested": bool(
                training_row["gpu_requested"]
            ),
            "cuda_available": bool(
                training_row["cuda_available"]
            ),
            "gpu_count": int(
                training_row["gpu_count"]
            ),
            "random_state_supported": bool(
                training_row["random_state_supported"]
            ),
            "random_state_used": (
                None
                if pd.isna(
                    training_row[
                        "random_state_used"
                    ]
                )
                else training_row[
                    "random_state_used"
                ]
            ),
            "training_runtime_seconds": (
                None
                if pd.isna(
                    training_row[
                        "training_runtime_seconds"
                    ]
                )
                else float(
                    training_row[
                        "training_runtime_seconds"
                    ]
                )
            ),
            "loss_records": int(
                training_row["loss_records"]
            ),
            "checkpoint_model_path": str(
                checkpoint_model_path
            ),
            "checkpoint_model_size_bytes": int(
                checkpoint_model_size
            ),
            "created_utc": datetime.now(
                timezone.utc
            ).isoformat(),
        }

        with open(
            checkpoint_metadata_path,
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                recovered_checkpoint_metadata,
                f,
                indent=2,
                ensure_ascii=False,
            )

        print(
            f"✓ Recovered checkpoint metadata : "
            f"{checkpoint_metadata_path}"
        )

    # -------------------------------------------------------------------------------------------
    # Validate metadata file
    # -------------------------------------------------------------------------------------------

    checkpoint_metadata_size = (
        checkpoint_metadata_path.stat().st_size
    )

    if checkpoint_metadata_size <= 0:

        raise RuntimeError(
            f"{dataset_id}: checkpoint metadata is empty."
        )

    with open(
        checkpoint_metadata_path,
        "r",
        encoding="utf-8",
    ) as f:

        checkpoint_metadata = json.load(f)

    if not isinstance(
        checkpoint_metadata,
        dict,
    ):

        raise RuntimeError(
            f"{dataset_id}: checkpoint metadata is not "
            "a valid JSON object."
        )

    # -------------------------------------------------------------------------------------------
    # Dataset identity
    # -------------------------------------------------------------------------------------------

    if (
        checkpoint_metadata.get(
            "dataset_id"
        )
        != dataset_id
    ):

        raise RuntimeError(
            f"{dataset_id}: checkpoint metadata dataset mismatch."
        )

    # -------------------------------------------------------------------------------------------
    # Status
    # -------------------------------------------------------------------------------------------

    if (
        checkpoint_metadata.get(
            "status"
        )
        != "PASS"
    ):

        raise RuntimeError(
            f"{dataset_id}: checkpoint metadata status is not PASS."
        )

    # -------------------------------------------------------------------------------------------
    # Notebook identity
    # -------------------------------------------------------------------------------------------

    if (
        checkpoint_metadata.get(
            "notebook_id"
        )
        != NOTEBOOK_ID
    ):

        raise RuntimeError(
            f"{dataset_id}: checkpoint notebook identity mismatch."
        )

    # -------------------------------------------------------------------------------------------
    # SDV version
    # -------------------------------------------------------------------------------------------

    if (
        checkpoint_metadata.get(
            "sdv_version"
        )
        != SDV_VERSION
    ):

        raise RuntimeError(
            f"{dataset_id}: checkpoint SDV version mismatch. "
            f"Found={checkpoint_metadata.get('sdv_version')}, "
            f"Expected={SDV_VERSION}"
        )

    # -------------------------------------------------------------------------------------------
    # SHA-256 integrity
    # -------------------------------------------------------------------------------------------

    model_sha256 = calculate_sha256(
        checkpoint_model_path
    )

    metadata_sha256 = calculate_sha256(
        checkpoint_metadata_path
    )

    # -------------------------------------------------------------------------------------------
    # Compare model hash with Section 8 record when available
    # -------------------------------------------------------------------------------------------

    recorded_model_sha256 = (
        checkpoint_record.get(
            "checkpoint_sha256"
        )
    )

    if (
        recorded_model_sha256 is not None
        and recorded_model_sha256 != model_sha256
    ):

        raise RuntimeError(
            f"{dataset_id}: checkpoint SHA-256 mismatch."
        )

    # -------------------------------------------------------------------------------------------
    # Record validation
    # -------------------------------------------------------------------------------------------

    TVAE_CHECKPOINT_VALIDATION_RECORDS.append(
        {
            "dataset_id": dataset_id,
            "checkpoint_model_path": str(
                checkpoint_model_path
            ),
            "checkpoint_model_size_bytes": (
                checkpoint_model_size
            ),
            "checkpoint_model_sha256": (
                model_sha256
            ),
            "checkpoint_metadata_path": str(
                checkpoint_metadata_path
            ),
            "checkpoint_metadata_size_bytes": (
                checkpoint_metadata_size
            ),
            "checkpoint_metadata_sha256": (
                metadata_sha256
            ),
            "notebook_id": checkpoint_metadata.get(
                "notebook_id"
            ),
            "sdv_version": checkpoint_metadata.get(
                "sdv_version"
            ),
            "artifact_status": checkpoint_record.get(
                "artifact_status"
            ),
            "status": "PASS",
        }
    )

    print(
        f"✓ {dataset_id:<20} | "
        f"checkpoint PASS"
    )

# -----------------------------------------------------------------------------------------------
# 4. Validate complete checkpoint coverage
# -----------------------------------------------------------------------------------------------

TVAE_CHECKPOINT_DF = pd.DataFrame(
    TVAE_CHECKPOINT_VALIDATION_RECORDS
)

if len(
    TVAE_CHECKPOINT_DF
) != len(DATASET_IDS):

    raise RuntimeError(
        "Checkpoint validation does not contain exactly "
        "one record per authoritative dataset."
    )

if (
    set(
        TVAE_CHECKPOINT_DF[
            "dataset_id"
        ]
    )
    != set(DATASET_IDS)
):

    raise RuntimeError(
        "Checkpoint validation dataset coverage does not "
        "match DATASET_IDS."
    )

if (
    TVAE_CHECKPOINT_DF[
        "status"
    ] != "PASS"
).any():

    raise RuntimeError(
        "One or more TVAE checkpoint validation records "
        "are not PASS."
    )

# -----------------------------------------------------------------------------------------------
# 5. Persist checkpoint registry
# -----------------------------------------------------------------------------------------------

TVAE_CHECKPOINT_REPORT_PATH = (
    NB05_CHECKPOINT_ROOT
    / "checkpoint_registry.csv"
)

TVAE_CHECKPOINT_DF.to_csv(
    TVAE_CHECKPOINT_REPORT_PATH,
    index=False,
)

if not TVAE_CHECKPOINT_REPORT_PATH.exists():

    raise RuntimeError(
        "Checkpoint registry was not persisted."
    )

if TVAE_CHECKPOINT_REPORT_PATH.stat().st_size <= 0:

    raise RuntimeError(
        "Checkpoint registry is empty."
    )

# -----------------------------------------------------------------------------------------------
# 6. Final summary
# -----------------------------------------------------------------------------------------------

print()
print(
    f"✓ Checkpoints verified : "
    f"{len(TVAE_CHECKPOINT_DF)}"
)

print(
    f"✓ Checkpoint registry  : "
    f"{TVAE_CHECKPOINT_REPORT_PATH}"
)

print(
    "✓ Checkpoint model files validated."
)

print(
    "✓ Checkpoint metadata validated."
)

print(
    "✓ Dataset-to-checkpoint mapping validated."
)

print(
    "✓ SHA-256 integrity validated."
)

print(
    "✓ Notebook identity validated."
)

print(
    "✓ SDV version consistency validated."
)

print(
    "✓ Checkpoint coverage validated."
)

print()
print(
    "SECTION 10 STATUS: COMPLETE / PASS"
)

SECTION 10 — SAVE CHECKPOINTS
✓ SHA-256 helper verified.
✓ adult_income         | checkpoint PASS
✓ bank_marketing       | checkpoint PASS
✓ diabetes_130us       | checkpoint PASS

✓ Checkpoints verified : 3
✓ Checkpoint registry  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05/checkpoints/checkpoint_registry.csv
✓ Checkpoint model files validated.
✓ Checkpoint metadata validated.
✓ Dataset-to-checkpoint mapping validated.
✓ SHA-256 integrity validated.
✓ Notebook identity validated.
✓ SDV version consistency validated.
✓ Checkpoint coverage validated.

SECTION 10 STATUS: COMPLETE / PASS


In [32]:
# ==================================================================================================
# 11. GENERATE SYNTHETIC DATA
# ==================================================================================================

print("=" * 100)
print("SECTION 11 — GENERATE SYNTHETIC DATA")
print("=" * 100)

from sdv.utils import load_synthesizer

TVAE_SYNTHETIC_DATA = {}

TVAE_GENERATION_RUNTIME_RECORDS = []

# -----------------------------------------------------------------------------------------------
# 1. Dataset-specific target and identifier definitions
# -----------------------------------------------------------------------------------------------

TVAE_TARGET_COLUMNS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

TVAE_EXCLUDED_IDENTIFIER_COLUMNS = {
    "adult_income": [],
    "bank_marketing": [],
    "diabetes_130us": [
        "encounter_id",
        "patient_nbr",
    ],
}

# -----------------------------------------------------------------------------------------------
# 2. Validate target configuration
# -----------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    if dataset_id not in TVAE_TARGET_COLUMNS:

        raise RuntimeError(
            f"{dataset_id}: target column definition is missing."
        )

    if dataset_id not in TVAE_EXCLUDED_IDENTIFIER_COLUMNS:

        raise RuntimeError(
            f"{dataset_id}: identifier exclusion definition is missing."
        )

# -----------------------------------------------------------------------------------------------
# 3. Generate synthetic data from frozen TVAE checkpoints
# -----------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    print()
    print(
        f"Generating synthetic data : {dataset_id}"
    )

    # -------------------------------------------------------------------------------------------
    # Dataset-specific authoritative seed
    # -------------------------------------------------------------------------------------------

    seed = TVAE_SEED_REGISTRY[
        dataset_id
    ]

    seed_everything(
        seed
    )

    # -------------------------------------------------------------------------------------------
    # Authoritative training-row count
    # -------------------------------------------------------------------------------------------

    training_rows = len(
        TRAINING_DATA[
            dataset_id
        ]
    )

    if training_rows <= 0:

        raise RuntimeError(
            f"{dataset_id}: authoritative TRAIN dataset "
            "contains no rows."
        )

    # -------------------------------------------------------------------------------------------
    # Expected frozen generative schema
    # -------------------------------------------------------------------------------------------

    expected_columns = (
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    if len(expected_columns) == 0:

        raise RuntimeError(
            f"{dataset_id}: expected generative schema is empty."
        )

    # -------------------------------------------------------------------------------------------
    # Checkpoint validation
    # -------------------------------------------------------------------------------------------

    if dataset_id not in TVAE_CHECKPOINT_RECORDS:

        raise RuntimeError(
            f"{dataset_id}: TVAE checkpoint record is missing."
        )

    checkpoint_path = Path(
        TVAE_CHECKPOINT_RECORDS[
            dataset_id
        ]["checkpoint_model_path"]
    )

    if not checkpoint_path.exists():

        raise FileNotFoundError(
            f"{dataset_id}: TVAE checkpoint not found: "
            f"{checkpoint_path}"
        )

    if checkpoint_path.stat().st_size <= 0:

        raise RuntimeError(
            f"{dataset_id}: TVAE checkpoint is empty."
        )

    # -------------------------------------------------------------------------------------------
    # Load frozen TVAE synthesizer
    # -------------------------------------------------------------------------------------------

    synthesizer = load_synthesizer(
        filepath=str(
            checkpoint_path
        )
    )

    print(
        "✓ Frozen TVAE checkpoint loaded."
    )

    # -------------------------------------------------------------------------------------------
    # Generate synthetic data
    # -------------------------------------------------------------------------------------------

    generation_start = time.perf_counter()

    synthetic_df = synthesizer.sample(
        num_rows=training_rows
    )

    generation_end = time.perf_counter()

    generation_runtime_seconds = (
        generation_end
        - generation_start
    )

    # -------------------------------------------------------------------------------------------
    # Basic output validation
    # -------------------------------------------------------------------------------------------

    if not isinstance(
        synthetic_df,
        pd.DataFrame
    ):

        raise RuntimeError(
            f"{dataset_id}: TVAE did not return a DataFrame."
        )

    if len(
        synthetic_df
    ) != training_rows:

        raise RuntimeError(
            f"{dataset_id}: generated row count mismatch. "
            f"Expected={training_rows}, "
            f"Actual={len(synthetic_df)}"
        )

    # -------------------------------------------------------------------------------------------
    # Column-count validation
    # -------------------------------------------------------------------------------------------

    if len(
        synthetic_df.columns
    ) != len(expected_columns):

        raise RuntimeError(
            f"{dataset_id}: generated column count mismatch. "
            f"Expected={len(expected_columns)}, "
            f"Actual={len(synthetic_df.columns)}"
        )

    # -------------------------------------------------------------------------------------------
    # Exact schema validation
    # -------------------------------------------------------------------------------------------

    if list(
        synthetic_df.columns
    ) != list(
        expected_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: generated schema mismatch."
        )

    # -------------------------------------------------------------------------------------------
    # Duplicate-column validation
    # -------------------------------------------------------------------------------------------

    if synthetic_df.columns.duplicated().any():

        duplicate_columns = (
            synthetic_df.columns[
                synthetic_df.columns.duplicated()
            ]
            .tolist()
        )

        raise RuntimeError(
            f"{dataset_id}: generated dataset contains "
            f"duplicate columns: {duplicate_columns}"
        )

    # -------------------------------------------------------------------------------------------
    # Target-column validation
    # -------------------------------------------------------------------------------------------

    target_column = (
        TVAE_TARGET_COLUMNS[
            dataset_id
        ]
    )

    if target_column not in synthetic_df.columns:

        raise RuntimeError(
            f"{dataset_id}: target column "
            f"'{target_column}' missing from synthetic data."
        )

    # -------------------------------------------------------------------------------------------
    # Identifier leakage validation
    # -------------------------------------------------------------------------------------------

    identifier_columns = (
        TVAE_EXCLUDED_IDENTIFIER_COLUMNS[
            dataset_id
        ]
    )

    leaked_identifiers = [
        column
        for column in identifier_columns
        if column in synthetic_df.columns
    ]

    if leaked_identifiers:

        raise RuntimeError(
            f"{dataset_id}: identifier leakage detected. "
            f"Identifiers present in synthetic data: "
            f"{leaked_identifiers}"
        )

    # -------------------------------------------------------------------------------------------
    # Store validated synthetic dataset
    # -------------------------------------------------------------------------------------------

    TVAE_SYNTHETIC_DATA[
        dataset_id
    ] = synthetic_df

    # -------------------------------------------------------------------------------------------
    # Record generation provenance
    # -------------------------------------------------------------------------------------------

    TVAE_GENERATION_RUNTIME_RECORDS.append(
        {
            "dataset_id": dataset_id,
            "seed": seed,
            "rows_requested": training_rows,
            "rows_generated": len(
                synthetic_df
            ),
            "columns_generated": len(
                synthetic_df.columns
            ),
            "target_column": target_column,
            "identifier_columns_excluded": (
                len(leaked_identifiers) == 0
            ),
            "generation_runtime_seconds": (
                generation_runtime_seconds
            ),
            "checkpoint_path": str(
                checkpoint_path
            ),
            "sdv_version": SDV_VERSION,
            "status": "PASS",
        }
    )

    print(
        f"✓ rows={len(synthetic_df):,} | "
        f"columns={len(synthetic_df.columns)} | "
        f"runtime={generation_runtime_seconds:.3f}s"
    )

    print(
        "✓ Generated schema validated."
    )

    print(
        "✓ Target column validated."
    )

    print(
        "✓ Identifier leakage check passed."
    )

    del synthesizer

    gc.collect()

# -----------------------------------------------------------------------------------------------
# 4. Validate complete generation coverage
# -----------------------------------------------------------------------------------------------

if len(
    TVAE_SYNTHETIC_DATA
) != len(DATASET_IDS):

    raise RuntimeError(
        "Synthetic-data generation did not complete "
        "for every authoritative dataset."
    )

if (
    set(
        TVAE_SYNTHETIC_DATA.keys()
    )
    != set(DATASET_IDS)
):

    raise RuntimeError(
        "Synthetic-data generation coverage does not "
        "match DATASET_IDS."
    )

if len(
    TVAE_GENERATION_RUNTIME_RECORDS
) != len(DATASET_IDS):

    raise RuntimeError(
        "Synthetic generation runtime records do not contain "
        "exactly one record per dataset."
    )

TVAE_GENERATION_RUNTIME_DF = pd.DataFrame(
    TVAE_GENERATION_RUNTIME_RECORDS
)

if (
    TVAE_GENERATION_RUNTIME_DF[
        "status"
    ] != "PASS"
).any():

    raise RuntimeError(
        "One or more synthetic generation records are not PASS."
    )

# -----------------------------------------------------------------------------------------------
# 5. Final generation summary
# -----------------------------------------------------------------------------------------------

print()

print(
    TVAE_GENERATION_RUNTIME_DF.to_string(
        index=False
    )
)

print()

print(
    f"✓ Synthetic datasets generated : "
    f"{len(TVAE_SYNTHETIC_DATA)}"
)

print(
    "✓ Synthetic row counts validated."
)

print(
    "✓ Synthetic schemas validated."
)

print(
    "✓ Target columns validated."
)

print(
    "✓ Identifier leakage checks passed."
)

print(
    "✓ Generation provenance recorded."
)

print()
print(
    "SECTION 11 STATUS: COMPLETE / PASS"
)

SECTION 11 — GENERATE SYNTHETIC DATA

Generating synthetic data : adult_income
✓ Frozen TVAE checkpoint loaded.
✓ rows=34,189 | columns=15 | runtime=0.622s
✓ Generated schema validated.
✓ Target column validated.
✓ Identifier leakage check passed.

Generating synthetic data : bank_marketing
✓ Frozen TVAE checkpoint loaded.
✓ rows=31,647 | columns=17 | runtime=0.613s
✓ Generated schema validated.
✓ Target column validated.
✓ Identifier leakage check passed.

Generating synthetic data : diabetes_130us
✓ Frozen TVAE checkpoint loaded.
✓ rows=71,236 | columns=48 | runtime=7.216s
✓ Generated schema validated.
✓ Target column validated.
✓ Identifier leakage check passed.

    dataset_id  seed  rows_requested  rows_generated  columns_generated target_column  identifier_columns_excluded  generation_runtime_seconds                                                                                                      checkpoint_path sdv_version status
  adult_income  3126           34189          

In [33]:
# ==================================================================================================
# 12. VALIDATE SYNTHETIC DATA
# ==================================================================================================

print("=" * 100)
print("SECTION 12 — VALIDATE SYNTHETIC DATA")
print("=" * 100)

TVAE_SYNTHETIC_VALIDATION_RECORDS = []

# -----------------------------------------------------------------------------------------------
# 1. Validate every generated synthetic dataset
# -----------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    if dataset_id not in TVAE_SYNTHETIC_DATA:

        raise RuntimeError(
            f"{dataset_id}: synthetic dataset is missing."
        )

    synthetic_df = (
        TVAE_SYNTHETIC_DATA[
            dataset_id
        ]
    )

    expected_columns = (
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    expected_rows = len(
        TRAINING_DATA[
            dataset_id
        ]
    )

    target = (
        TRAINING_TARGET_COLUMNS[
            dataset_id
        ]
    )

    provenance = (
        TRAINING_PROVENANCE_COLUMNS[
            dataset_id
        ]
    )

    identifiers = (
        TRAINING_IDENTIFIER_COLUMNS[
            dataset_id
        ]
    )

    # -------------------------------------------------------------------------------------------
    # DataFrame validation
    # -------------------------------------------------------------------------------------------

    if not isinstance(
        synthetic_df,
        pd.DataFrame
    ):

        raise RuntimeError(
            f"{dataset_id}: synthetic object is not a DataFrame."
        )

    # -------------------------------------------------------------------------------------------
    # Row-count validation
    # -------------------------------------------------------------------------------------------

    if len(
        synthetic_df
    ) != expected_rows:

        raise RuntimeError(
            f"{dataset_id}: synthetic row count mismatch. "
            f"Expected={expected_rows}, "
            f"Actual={len(synthetic_df)}"
        )

    # -------------------------------------------------------------------------------------------
    # Schema validation
    # -------------------------------------------------------------------------------------------

    if list(
        synthetic_df.columns
    ) != list(
        expected_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: synthetic schema mismatch."
        )

    # -------------------------------------------------------------------------------------------
    # Duplicate-column validation
    # -------------------------------------------------------------------------------------------

    if synthetic_df.columns.duplicated().any():

        duplicate_columns = (
            synthetic_df.columns[
                synthetic_df.columns.duplicated()
            ]
            .tolist()
        )

        raise RuntimeError(
            f"{dataset_id}: duplicate synthetic columns detected: "
            f"{duplicate_columns}"
        )

    # -------------------------------------------------------------------------------------------
    # Target validation
    # -------------------------------------------------------------------------------------------

    if target not in synthetic_df.columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target}' missing."
        )

    # -------------------------------------------------------------------------------------------
    # Provenance leakage validation
    # -------------------------------------------------------------------------------------------

    if provenance in synthetic_df.columns:

        raise RuntimeError(
            f"{dataset_id}: provenance leakage detected: "
            f"'{provenance}'"
        )

    # -------------------------------------------------------------------------------------------
    # Explicit identifier leakage validation
    # -------------------------------------------------------------------------------------------

    leaked_identifiers = (
        set(
            identifiers
        ).intersection(
            synthetic_df.columns
        )
    )

    if leaked_identifiers:

        raise RuntimeError(
            f"{dataset_id}: identifier leakage detected: "
            f"{sorted(leaked_identifiers)}"
        )

    # -------------------------------------------------------------------------------------------
    # Native generative-column policy
    # -------------------------------------------------------------------------------------------

    if len(
        synthetic_df.columns
    ) != len(
        expected_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: synthetic column count mismatch."
        )

    # -------------------------------------------------------------------------------------------
    # Record successful validation
    # -------------------------------------------------------------------------------------------

    TVAE_SYNTHETIC_VALIDATION_RECORDS.append(
        {
            "dataset_id": dataset_id,
            "rows": len(
                synthetic_df
            ),
            "columns": len(
                synthetic_df.columns
            ),
            "target_column": target,
            "target_present": True,
            "provenance_column": provenance,
            "provenance_excluded": True,
            "identifier_count": len(
                identifiers
            ),
            "identifiers_excluded": True,
            "schema_valid": True,
            "sample_size_valid": True,
            "status": "PASS",
        }
    )

    print(
        f"✓ {dataset_id:<20} | "
        f"rows={len(synthetic_df):,} | "
        f"columns={len(synthetic_df.columns)} | "
        f"schema PASS | leakage PASS"
    )

# -----------------------------------------------------------------------------------------------
# 2. Build validation DataFrame
# -----------------------------------------------------------------------------------------------

TVAE_SYNTHETIC_VALIDATION_DF = pd.DataFrame(
    TVAE_SYNTHETIC_VALIDATION_RECORDS
)

# -----------------------------------------------------------------------------------------------
# 3. Validate complete dataset coverage
# -----------------------------------------------------------------------------------------------

if len(
    TVAE_SYNTHETIC_VALIDATION_DF
) != len(DATASET_IDS):

    raise RuntimeError(
        "Synthetic validation does not contain exactly "
        "one record per authoritative dataset."
    )

if (
    set(
        TVAE_SYNTHETIC_VALIDATION_DF[
            "dataset_id"
        ]
    )
    != set(DATASET_IDS)
):

    raise RuntimeError(
        "Synthetic validation dataset coverage does not "
        "match DATASET_IDS."
    )

# -----------------------------------------------------------------------------------------------
# 4. Validate all records are PASS
# -----------------------------------------------------------------------------------------------

if (
    TVAE_SYNTHETIC_VALIDATION_DF[
        "status"
    ] != "PASS"
).any():

    raise RuntimeError(
        "One or more synthetic validation records are not PASS."
    )

# -----------------------------------------------------------------------------------------------
# 5. Persist synthetic validation registry
# -----------------------------------------------------------------------------------------------

TVAE_SYNTHETIC_VALIDATION_PATH = (
    NB05_HISTORY_ROOT
    / "tvae_synthetic_validation.csv"
)

TVAE_SYNTHETIC_VALIDATION_DF.to_csv(
    TVAE_SYNTHETIC_VALIDATION_PATH,
    index=False,
)

if not TVAE_SYNTHETIC_VALIDATION_PATH.exists():

    raise RuntimeError(
        "Synthetic validation registry was not persisted."
    )

if TVAE_SYNTHETIC_VALIDATION_PATH.stat().st_size <= 0:

    raise RuntimeError(
        "Synthetic validation registry is empty."
    )

# -----------------------------------------------------------------------------------------------
# 6. Final validation summary
# -----------------------------------------------------------------------------------------------

print()
print(
    TVAE_SYNTHETIC_VALIDATION_DF.to_string(
        index=False
    )
)

print()
print(
    f"✓ Synthetic validations : "
    f"{len(TVAE_SYNTHETIC_VALIDATION_DF)}"
)

print(
    f"✓ Validation registry   : "
    f"{TVAE_SYNTHETIC_VALIDATION_PATH}"
)

print(
    "✓ Row counts validated."
)

print(
    "✓ Exact generative schemas validated."
)

print(
    "✓ Target columns validated."
)

print(
    "✓ Provenance columns excluded."
)

print(
    "✓ Explicit identifiers excluded."
)

print(
    "✓ Complete dataset coverage validated."
)

print(
    "✓ All validation records are PASS."
)

print()
print(
    "SECTION 12 STATUS: COMPLETE / PASS"
)

SECTION 12 — VALIDATE SYNTHETIC DATA
✓ adult_income         | rows=34,189 | columns=15 | schema PASS | leakage PASS
✓ bank_marketing       | rows=31,647 | columns=17 | schema PASS | leakage PASS
✓ diabetes_130us       | rows=71,236 | columns=48 | schema PASS | leakage PASS

    dataset_id  rows  columns target_column  target_present   provenance_column  provenance_excluded  identifier_count  identifiers_excluded  schema_valid  sample_size_valid status
  adult_income 34189       15        income            True __original_row_id__                 True                 0                  True          True               True   PASS
bank_marketing 31647       17             y            True __original_row_id__                 True                 0                  True          True               True   PASS
diabetes_130us 71236       48    readmitted            True __original_row_id__                 True                 2                  True          True               True   PASS



In [34]:
# ==================================================================================================
# 13. RECORD RUNTIME
# ==================================================================================================

print("=" * 100)
print("SECTION 13 — RECORD RUNTIME")
print("=" * 100)

# -----------------------------------------------------------------------------------------------
# 1. Build training and generation runtime DataFrames
# -----------------------------------------------------------------------------------------------

TVAE_TRAINING_RUNTIME_DF = pd.DataFrame(
    TVAE_TRAINING_RECORDS
)

TVAE_GENERATION_RUNTIME_DF = pd.DataFrame(
    TVAE_GENERATION_RUNTIME_RECORDS
)

# -----------------------------------------------------------------------------------------------
# 2. Validate required runtime columns
# -----------------------------------------------------------------------------------------------

required_training_columns = [
    "dataset_id",
    "seed",
    "training_rows",
    "training_columns",
    "training_runtime_seconds",
]

required_generation_columns = [
    "dataset_id",
    "seed",
    "rows_requested",
    "rows_generated",
    "generation_runtime_seconds",
]

missing_training_columns = [
    column
    for column in required_training_columns
    if column not in TVAE_TRAINING_RUNTIME_DF.columns
]

if missing_training_columns:

    raise RuntimeError(
        "TVAE training runtime records are missing columns: "
        f"{missing_training_columns}"
    )

missing_generation_columns = [
    column
    for column in required_generation_columns
    if column not in TVAE_GENERATION_RUNTIME_DF.columns
]

if missing_generation_columns:

    raise RuntimeError(
        "TVAE generation runtime records are missing columns: "
        f"{missing_generation_columns}"
    )

# -----------------------------------------------------------------------------------------------
# 3. Merge training and generation runtime records
# -----------------------------------------------------------------------------------------------

TVAE_RUNTIME_DF = (
    TVAE_TRAINING_RUNTIME_DF[
        required_training_columns
    ]
    .merge(
        TVAE_GENERATION_RUNTIME_DF[
            required_generation_columns
        ],
        on=[
            "dataset_id",
            "seed",
        ],
        how="inner",
        validate="one_to_one",
    )
)

# -----------------------------------------------------------------------------------------------
# 4. Validate dataset coverage
# -----------------------------------------------------------------------------------------------

if len(
    TVAE_RUNTIME_DF
) != len(DATASET_IDS):

    raise RuntimeError(
        "TVAE runtime record count mismatch."
    )

if (
    set(
        TVAE_RUNTIME_DF[
            "dataset_id"
        ]
    )
    != set(DATASET_IDS)
):

    raise RuntimeError(
        "TVAE runtime dataset coverage does not "
        "match DATASET_IDS."
    )

# -----------------------------------------------------------------------------------------------
# 5. Validate training/requested/generated sample sizes
# -----------------------------------------------------------------------------------------------

if not (
    TVAE_RUNTIME_DF[
        "training_rows"
    ]
    ==
    TVAE_RUNTIME_DF[
        "rows_requested"
    ]
).all():

    raise RuntimeError(
        "Training/requested sample-size mismatch."
    )

if not (
    TVAE_RUNTIME_DF[
        "rows_requested"
    ]
    ==
    TVAE_RUNTIME_DF[
        "rows_generated"
    ]
).all():

    raise RuntimeError(
        "Requested/generated row-count mismatch."
    )

# -----------------------------------------------------------------------------------------------
# 6. Validate training-column counts
# -----------------------------------------------------------------------------------------------

if (
    TVAE_RUNTIME_DF[
        "training_columns"
    ]
    <= 0
).any():

    raise RuntimeError(
        "One or more training-column counts are non-positive."
    )

# -----------------------------------------------------------------------------------------------
# 7. Validate generation runtime
#
#    Generation was performed in the current run, so all generation runtimes
#    must be finite and non-negative.
# -----------------------------------------------------------------------------------------------

if not np.isfinite(
    TVAE_RUNTIME_DF[
        "generation_runtime_seconds"
    ]
).all():

    raise RuntimeError(
        "Non-finite generation runtime values detected."
    )

if (
    TVAE_RUNTIME_DF[
        "generation_runtime_seconds"
    ]
    < 0
).any():

    raise RuntimeError(
        "Negative generation runtime values detected."
    )

# -----------------------------------------------------------------------------------------------
# 8. Validate training runtime where available
#
#    REUSED_EXISTING artifacts may legitimately have unavailable historical
#    training runtime. Do not fabricate or impute this value.
# -----------------------------------------------------------------------------------------------

training_runtime_available = (
    TVAE_RUNTIME_DF[
        "training_runtime_seconds"
    ].notna()
)

if training_runtime_available.any():

    available_training_runtimes = (
        TVAE_RUNTIME_DF.loc[
            training_runtime_available,
            "training_runtime_seconds"
        ]
    )

    if not np.isfinite(
        available_training_runtimes
    ).all():

        raise RuntimeError(
            "Non-finite available training runtime values detected."
        )

    if (
        available_training_runtimes < 0
    ).any():

        raise RuntimeError(
            "Negative available training runtime values detected."
        )

# -----------------------------------------------------------------------------------------------
# 9. Calculate total runtime only when training runtime is available
# -----------------------------------------------------------------------------------------------

TVAE_RUNTIME_DF[
    "total_runtime_seconds"
] = np.nan

training_runtime_available = (
    TVAE_RUNTIME_DF[
        "training_runtime_seconds"
    ].notna()
)

TVAE_RUNTIME_DF.loc[
    training_runtime_available,
    "total_runtime_seconds"
] = (
    TVAE_RUNTIME_DF.loc[
        training_runtime_available,
        "training_runtime_seconds"
    ]
    +
    TVAE_RUNTIME_DF.loc[
        training_runtime_available,
        "generation_runtime_seconds"
    ]
)

# -----------------------------------------------------------------------------------------------
# 10. Add runtime availability metadata
# -----------------------------------------------------------------------------------------------

TVAE_RUNTIME_DF[
    "training_runtime_available"
] = (
    TVAE_RUNTIME_DF[
        "training_runtime_seconds"
    ].notna()
)

TVAE_RUNTIME_DF[
    "total_runtime_available"
] = (
    TVAE_RUNTIME_DF[
        "total_runtime_seconds"
    ].notna()
)

# -----------------------------------------------------------------------------------------------
# 11. Persist runtime registry
# -----------------------------------------------------------------------------------------------

TVAE_RUNTIME_REPORT_PATH = (
    NB05_HISTORY_ROOT
    / "tvae_runtime_summary.csv"
)

TVAE_RUNTIME_DF.to_csv(
    TVAE_RUNTIME_REPORT_PATH,
    index=False,
)

if not TVAE_RUNTIME_REPORT_PATH.exists():

    raise RuntimeError(
        "TVAE runtime summary was not persisted."
    )

if TVAE_RUNTIME_REPORT_PATH.stat().st_size <= 0:

    raise RuntimeError(
        "TVAE runtime summary is empty."
    )

# -----------------------------------------------------------------------------------------------
# 12. Final output
# -----------------------------------------------------------------------------------------------

print()

print(
    TVAE_RUNTIME_DF.to_string(
        index=False
    )
)

print()

print(
    f"✓ Runtime summary saved : "
    f"{TVAE_RUNTIME_REPORT_PATH}"
)

print(
    "✓ Dataset coverage validated."
)

print(
    "✓ Training/requested/generated row counts validated."
)

print(
    "✓ Generation runtimes validated."
)

print(
    "✓ Available training runtimes validated."
)

print(
    "✓ Missing historical runtime values preserved."
)

print(
    "✓ Total runtime calculated where training runtime is available."
)

print()
print(
    "SECTION 13 STATUS: COMPLETE / PASS"
)

SECTION 13 — RECORD RUNTIME

    dataset_id  seed  training_rows  training_columns  training_runtime_seconds  rows_requested  rows_generated  generation_runtime_seconds  total_runtime_seconds  training_runtime_available  total_runtime_available
  adult_income  3126          34189                15                       NaN           34189           34189                    0.621571                    NaN                       False                    False
bank_marketing  3226          31647                17                299.232635           31647           31647                    0.613010             299.845645                        True                     True
diabetes_130us  3326          71236                48                936.156400           71236           71236                    7.216061             943.372461                        True                     True

✓ Runtime summary saved : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05/history/tv

In [36]:
# ==================================================================================================
# 14. SAVE MODEL
# ==================================================================================================

print("=" * 100)
print("SECTION 14 — SAVE MODEL")
print("=" * 100)

import shutil
from pathlib import Path

# -----------------------------------------------------------------------------------------------
# 0. Resolve canonical Notebook 05 model directory
# -----------------------------------------------------------------------------------------------

if "NB05_ROOT" not in globals():

    raise RuntimeError(
        "NB05_ROOT is not defined. "
        "Notebook 05 canonical root is required for Section 14."
    )

NB05_MODEL_ROOT = (
    NB05_ROOT
    / "models"
)

NB05_MODEL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    f"✓ Canonical model root : {NB05_MODEL_ROOT}"
)

# -----------------------------------------------------------------------------------------------
# 1. Validate required dependencies
# -----------------------------------------------------------------------------------------------

required_objects = [
    "DATASET_IDS",
    "TVAE_CHECKPOINT_RECORDS",
    "calculate_sha256",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:

    raise RuntimeError(
        "Section 14 is missing required objects: "
        f"{missing_objects}"
    )

# -----------------------------------------------------------------------------------------------
# 2. Initialize model records
# -----------------------------------------------------------------------------------------------

TVAE_MODEL_RECORDS = []

# -----------------------------------------------------------------------------------------------
# 3. Copy validated TVAE checkpoints to canonical model locations
# -----------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    if dataset_id not in TVAE_CHECKPOINT_RECORDS:

        raise RuntimeError(
            f"{dataset_id}: validated TVAE checkpoint record is missing."
        )

    checkpoint_path = Path(
        TVAE_CHECKPOINT_RECORDS[
            dataset_id
        ]["checkpoint_model_path"]
    )

    # -------------------------------------------------------------------------------------------
    # Validate source checkpoint
    # -------------------------------------------------------------------------------------------

    if not checkpoint_path.exists():

        raise FileNotFoundError(
            f"{dataset_id}: source TVAE checkpoint does not exist."
        )

    if checkpoint_path.stat().st_size <= 0:

        raise RuntimeError(
            f"{dataset_id}: source TVAE checkpoint is empty."
        )

    # -------------------------------------------------------------------------------------------
    # Canonical model directory
    # -------------------------------------------------------------------------------------------

    model_directory = (
        NB05_MODEL_ROOT
        / dataset_id
    )

    model_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    model_path = (
        model_directory
        / "tvae.pkl"
    )

    # -------------------------------------------------------------------------------------------
    # Copy checkpoint to canonical model location
    # -------------------------------------------------------------------------------------------

    shutil.copy2(
        checkpoint_path,
        model_path,
    )

    # -------------------------------------------------------------------------------------------
    # Validate copied model
    # -------------------------------------------------------------------------------------------

    if not model_path.exists():

        raise RuntimeError(
            f"{dataset_id}: canonical TVAE model was not created."
        )

    model_size = (
        model_path.stat().st_size
    )

    if model_size <= 0:

        raise RuntimeError(
            f"{dataset_id}: canonical TVAE model is empty."
        )

    # -------------------------------------------------------------------------------------------
    # Cryptographic integrity validation
    # -------------------------------------------------------------------------------------------

    source_sha256 = calculate_sha256(
        checkpoint_path
    )

    model_sha256 = calculate_sha256(
        model_path
    )

    if source_sha256 != model_sha256:

        raise RuntimeError(
            f"{dataset_id}: canonical TVAE model SHA-256 does not "
            "match the source checkpoint SHA-256."
        )

    # -------------------------------------------------------------------------------------------
    # File-size integrity validation
    # -------------------------------------------------------------------------------------------

    source_size = (
        checkpoint_path.stat().st_size
    )

    if source_size != model_size:

        raise RuntimeError(
            f"{dataset_id}: canonical TVAE model size does not "
            "match the source checkpoint size."
        )

    # -------------------------------------------------------------------------------------------
    # Record canonical model
    # -------------------------------------------------------------------------------------------

    TVAE_MODEL_RECORDS.append(
        {
            "dataset_id": dataset_id,
            "model_path": str(
                model_path
            ),
            "relative_path": str(
                model_path.relative_to(
                    NB05_ROOT
                )
            ),
            "source_checkpoint_path": str(
                checkpoint_path
            ),
            "file_size_bytes": model_size,
            "source_checkpoint_sha256": source_sha256,
            "sha256": model_sha256,
            "artifact_copy_verified": True,
            "status": "PASS",
        }
    )

    print(
        f"✓ {dataset_id:<20} | "
        f"model saved | "
        f"{model_size:,} bytes | "
        f"SHA-256 PASS"
    )

# -----------------------------------------------------------------------------------------------
# 4. Build model registry
# -----------------------------------------------------------------------------------------------

TVAE_MODEL_DF = pd.DataFrame(
    TVAE_MODEL_RECORDS
)

# -----------------------------------------------------------------------------------------------
# 5. Validate model coverage
# -----------------------------------------------------------------------------------------------

if len(
    TVAE_MODEL_DF
) != len(DATASET_IDS):

    raise RuntimeError(
        "Canonical TVAE model record count mismatch."
    )

if (
    set(
        TVAE_MODEL_DF[
            "dataset_id"
        ]
    )
    != set(DATASET_IDS)
):

    raise RuntimeError(
        "Canonical TVAE model dataset coverage does not "
        "match DATASET_IDS."
    )

# -----------------------------------------------------------------------------------------------
# 6. Validate model records
# -----------------------------------------------------------------------------------------------

if (
    TVAE_MODEL_DF[
        "status"
    ] != "PASS"
).any():

    raise RuntimeError(
        "One or more canonical TVAE model records are not PASS."
    )

if not (
    TVAE_MODEL_DF[
        "artifact_copy_verified"
    ]
).all():

    raise RuntimeError(
        "One or more canonical TVAE model copies failed integrity verification."
    )

# -----------------------------------------------------------------------------------------------
# 7. Persist model registry
# -----------------------------------------------------------------------------------------------

TVAE_MODEL_REGISTRY_PATH = (
    NB05_MODEL_ROOT
    / "tvae_model_registry.csv"
)

TVAE_MODEL_DF.to_csv(
    TVAE_MODEL_REGISTRY_PATH,
    index=False,
)

if not TVAE_MODEL_REGISTRY_PATH.exists():

    raise RuntimeError(
        "TVAE model registry was not persisted."
    )

if TVAE_MODEL_REGISTRY_PATH.stat().st_size <= 0:

    raise RuntimeError(
        "TVAE model registry is empty."
    )

# -----------------------------------------------------------------------------------------------
# 8. Final summary
# -----------------------------------------------------------------------------------------------

print()
print(
    f"✓ Canonical TVAE models : "
    f"{len(TVAE_MODEL_DF)}"
)

print(
    f"✓ Model registry        : "
    f"{TVAE_MODEL_REGISTRY_PATH}"
)

print(
    "✓ Source checkpoints validated."
)

print(
    "✓ Canonical model files validated."
)

print(
    "✓ File sizes verified."
)

print(
    "✓ SHA-256 copy integrity verified."
)

print(
    "✓ Dataset coverage validated."
)

print(
    "✓ Model registry persisted."
)

print()
print(
    "SECTION 14 STATUS: COMPLETE / PASS"
)

SECTION 14 — SAVE MODEL
✓ Canonical model root : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05/models
✓ adult_income         | model saved | 1,078,122 bytes | SHA-256 PASS
✓ bank_marketing       | model saved | 1,057,925 bytes | SHA-256 PASS
✓ diabetes_130us       | model saved | 3,484,028 bytes | SHA-256 PASS

✓ Canonical TVAE models : 3
✓ Model registry        : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05/models/tvae_model_registry.csv
✓ Source checkpoints validated.
✓ Canonical model files validated.
✓ File sizes verified.
✓ SHA-256 copy integrity verified.
✓ Dataset coverage validated.
✓ Model registry persisted.

SECTION 14 STATUS: COMPLETE / PASS


In [38]:
# ==================================================================================================
# 15. SAVE SYNTHETIC DATA
# ==================================================================================================

print("=" * 100)
print("SECTION 15 — SAVE SYNTHETIC DATA")
print("=" * 100)

import pandas as pd
from pathlib import Path

# -----------------------------------------------------------------------------------------------
# 0. Validate required dependencies
# -----------------------------------------------------------------------------------------------

required_objects = [
    "NB05_ROOT",
    "DATASET_IDS",
    "TVAE_SYNTHETIC_DATA",
    "TRAINING_DATA",
    "TRAINING_GENERATIVE_COLUMNS",
    "calculate_sha256",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:

    raise RuntimeError(
        "Section 15 is missing required objects: "
        f"{missing_objects}"
    )

# -----------------------------------------------------------------------------------------------
# 1. Resolve canonical Notebook 05 synthetic-data root
# -----------------------------------------------------------------------------------------------

NB05_ROOT = Path(
    NB05_ROOT
)

NB05_SYNTHETIC_ROOT = (
    NB05_ROOT
    / "synthetic"
)

NB05_SYNTHETIC_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    f"✓ Canonical synthetic root : "
    f"{NB05_SYNTHETIC_ROOT}"
)

# -----------------------------------------------------------------------------------------------
# 2. Initialize artifact records
# -----------------------------------------------------------------------------------------------

TVAE_SYNTHETIC_ARTIFACT_RECORDS = []

# -----------------------------------------------------------------------------------------------
# 3. Save and validate synthetic datasets
# -----------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    # -------------------------------------------------------------------------------------------
    # Validate dataset availability
    # -------------------------------------------------------------------------------------------

    if dataset_id not in TVAE_SYNTHETIC_DATA:

        raise RuntimeError(
            f"{dataset_id}: synthetic dataset is missing from "
            "TVAE_SYNTHETIC_DATA."
        )

    if dataset_id not in TRAINING_DATA:

        raise RuntimeError(
            f"{dataset_id}: training dataset is missing."
        )

    if dataset_id not in TRAINING_GENERATIVE_COLUMNS:

        raise RuntimeError(
            f"{dataset_id}: authoritative generative schema is missing."
        )

    # -------------------------------------------------------------------------------------------
    # Resolve dataset directory
    # -------------------------------------------------------------------------------------------

    dataset_directory = (
        NB05_SYNTHETIC_ROOT
        / dataset_id
    )

    dataset_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    # -------------------------------------------------------------------------------------------
    # Obtain in-memory synthetic dataset
    # -------------------------------------------------------------------------------------------

    synthetic_df = (
        TVAE_SYNTHETIC_DATA[
            dataset_id
        ]
        .copy()
    )

    expected_columns = list(
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    expected_rows = len(
        TRAINING_DATA[
            dataset_id
        ]
    )

    # -------------------------------------------------------------------------------------------
    # Validate in-memory synthetic dataset
    # -------------------------------------------------------------------------------------------

    if list(
        synthetic_df.columns
    ) != expected_columns:

        raise RuntimeError(
            f"{dataset_id}: in-memory synthetic schema does not "
            "match the authoritative generative schema."
        )

    if len(
        synthetic_df
    ) != expected_rows:

        raise RuntimeError(
            f"{dataset_id}: in-memory synthetic row count does not "
            "match the training row count."
        )

    if synthetic_df.columns.duplicated().any():

        raise RuntimeError(
            f"{dataset_id}: duplicate columns detected in "
            "synthetic dataset."
        )

    # -------------------------------------------------------------------------------------------
    # Save synthetic dataset
    # -------------------------------------------------------------------------------------------

    synthetic_path = (
        dataset_directory
        / "tvae.csv"
    )

    synthetic_df.to_csv(
        synthetic_path,
        index=False,
    )

    # -------------------------------------------------------------------------------------------
    # Validate file creation
    # -------------------------------------------------------------------------------------------

    if not synthetic_path.exists():

        raise RuntimeError(
            f"{dataset_id}: synthetic CSV was not created."
        )

    file_size_bytes = (
        synthetic_path.stat().st_size
    )

    if file_size_bytes <= 0:

        raise RuntimeError(
            f"{dataset_id}: synthetic CSV is empty."
        )

    # -------------------------------------------------------------------------------------------
    # Compute SHA-256
    # -------------------------------------------------------------------------------------------

    file_sha256 = calculate_sha256(
        synthetic_path
    )

    # -------------------------------------------------------------------------------------------
    # Reload persisted synthetic dataset
    # -------------------------------------------------------------------------------------------

    reloaded_df = pd.read_csv(
        synthetic_path,
        low_memory=False,
    )

    # -------------------------------------------------------------------------------------------
    # Validate persisted schema
    # -------------------------------------------------------------------------------------------

    if list(
        reloaded_df.columns
    ) != expected_columns:

        raise RuntimeError(
            f"{dataset_id}: persisted synthetic schema mismatch."
        )

    # -------------------------------------------------------------------------------------------
    # Validate persisted row count
    # -------------------------------------------------------------------------------------------

    if len(
        reloaded_df
    ) != expected_rows:

        raise RuntimeError(
            f"{dataset_id}: persisted synthetic row count mismatch."
        )

    # -------------------------------------------------------------------------------------------
    # Validate duplicate columns
    # -------------------------------------------------------------------------------------------

    if reloaded_df.columns.duplicated().any():

        raise RuntimeError(
            f"{dataset_id}: duplicate columns detected after reload."
        )

    # -------------------------------------------------------------------------------------------
    # Validate persistence content
    #
    # CSV serialization may alter pandas dtypes during reload.
    # Therefore compare normalized cell values rather than requiring
    # exact pandas dtype preservation.
    # -------------------------------------------------------------------------------------------

    in_memory_normalized = (
        synthetic_df
        .reset_index(drop=True)
        .astype(str)
    )

    reloaded_normalized = (
        reloaded_df
        .reset_index(drop=True)
        .astype(str)
    )

    if not in_memory_normalized.equals(
        reloaded_normalized
    ):

        raise RuntimeError(
            f"{dataset_id}: reloaded synthetic data does not "
            "match the persisted synthetic data."
        )

    # -------------------------------------------------------------------------------------------
    # Record artifact
    # -------------------------------------------------------------------------------------------

    TVAE_SYNTHETIC_ARTIFACT_RECORDS.append(
        {
            "dataset_id": dataset_id,
            "artifact_type": "synthetic_data",
            "path": str(
                synthetic_path
            ),
            "relative_path": str(
                synthetic_path.relative_to(
                    NB05_ROOT
                )
            ),
            "rows": len(
                reloaded_df
            ),
            "columns": len(
                reloaded_df.columns
            ),
            "file_size_bytes": file_size_bytes,
            "sha256": file_sha256,
            "persistence_verified": True,
            "content_verified": True,
            "status": "PASS",
        }
    )

    print(
        f"✓ {dataset_id:<20} | "
        f"{len(reloaded_df):,} rows | "
        f"{len(reloaded_df.columns)} cols | "
        f"persistence PASS | "
        f"content PASS"
    )

# -----------------------------------------------------------------------------------------------
# 4. Build artifact registry
# -----------------------------------------------------------------------------------------------

TVAE_SYNTHETIC_ARTIFACT_DF = pd.DataFrame(
    TVAE_SYNTHETIC_ARTIFACT_RECORDS
)

# -----------------------------------------------------------------------------------------------
# 5. Validate artifact coverage
# -----------------------------------------------------------------------------------------------

if len(
    TVAE_SYNTHETIC_ARTIFACT_DF
) != len(DATASET_IDS):

    raise RuntimeError(
        "Synthetic artifact record count does not match DATASET_IDS."
    )

if (
    set(
        TVAE_SYNTHETIC_ARTIFACT_DF[
            "dataset_id"
        ]
    )
    != set(DATASET_IDS)
):

    raise RuntimeError(
        "Synthetic artifact dataset coverage does not match DATASET_IDS."
    )

# -----------------------------------------------------------------------------------------------
# 6. Validate artifact status
# -----------------------------------------------------------------------------------------------

if (
    TVAE_SYNTHETIC_ARTIFACT_DF[
        "status"
    ] != "PASS"
).any():

    raise RuntimeError(
        "One or more synthetic-data artifacts are not PASS."
    )

if not (
    TVAE_SYNTHETIC_ARTIFACT_DF[
        "persistence_verified"
    ]
).all():

    raise RuntimeError(
        "One or more synthetic-data persistence checks failed."
    )

if not (
    TVAE_SYNTHETIC_ARTIFACT_DF[
        "content_verified"
    ]
).all():

    raise RuntimeError(
        "One or more synthetic-data content checks failed."
    )

# -----------------------------------------------------------------------------------------------
# 7. Persist synthetic artifact registry
# -----------------------------------------------------------------------------------------------

TVAE_SYNTHETIC_REGISTRY_PATH = (
    NB05_SYNTHETIC_ROOT
    / "tvae_synthetic_artifact_registry.csv"
)

TVAE_SYNTHETIC_ARTIFACT_DF.to_csv(
    TVAE_SYNTHETIC_REGISTRY_PATH,
    index=False,
)

if not TVAE_SYNTHETIC_REGISTRY_PATH.exists():

    raise RuntimeError(
        "TVAE synthetic artifact registry was not persisted."
    )

if TVAE_SYNTHETIC_REGISTRY_PATH.stat().st_size <= 0:

    raise RuntimeError(
        "TVAE synthetic artifact registry is empty."
    )

# -----------------------------------------------------------------------------------------------
# 8. Final summary
# -----------------------------------------------------------------------------------------------

print()
print(
    f"✓ Synthetic artifacts       : "
    f"{len(TVAE_SYNTHETIC_ARTIFACT_DF)}"
)

print(
    f"✓ Synthetic registry        : "
    f"{TVAE_SYNTHETIC_REGISTRY_PATH}"
)

print(
    "✓ In-memory schema validated."
)

print(
    "✓ Persisted schema validated."
)

print(
    "✓ Row counts validated."
)

print(
    "✓ Duplicate-column policy validated."
)

print(
    "✓ Persistence content verified."
)

print(
    "✓ SHA-256 hashes recorded."
)

print(
    "✓ Dataset coverage validated."
)

print(
    "✓ Synthetic artifact registry persisted."
)

print()
print(
    "SECTION 15 STATUS: COMPLETE / PASS"
)

SECTION 15 — SAVE SYNTHETIC DATA
✓ Canonical synthetic root : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05/synthetic
✓ adult_income         | 34,189 rows | 15 cols | persistence PASS | content PASS
✓ bank_marketing       | 31,647 rows | 17 cols | persistence PASS | content PASS
✓ diabetes_130us       | 71,236 rows | 48 cols | persistence PASS | content PASS

✓ Synthetic artifacts       : 3
✓ Synthetic registry        : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05/synthetic/tvae_synthetic_artifact_registry.csv
✓ In-memory schema validated.
✓ Persisted schema validated.
✓ Row counts validated.
✓ Duplicate-column policy validated.
✓ Persistence content verified.
✓ SHA-256 hashes recorded.
✓ Dataset coverage validated.
✓ Synthetic artifact registry persisted.

SECTION 15 STATUS: COMPLETE / PASS


In [40]:
# ==================================================================================================
# 16. SAVE METADATA / MANIFEST
# ==================================================================================================

print("=" * 100)
print("SECTION 16 — SAVE METADATA / MANIFEST")
print("=" * 100)

import json
import pandas as pd
import numpy as np

from pathlib import Path
from datetime import datetime, timezone

# -----------------------------------------------------------------------------------------------
# 0. Validate core dependencies
# -----------------------------------------------------------------------------------------------

required_objects = [
    "NB05_ROOT",
    "NB05_METADATA_ROOT",
    "DATASET_IDS",
    "NOTEBOOK_ID",
    "NOTEBOOK_VERSION",
    "MASTER_SEED",
    "TVAE_CONFIG",
    "TVAE_SEED_REGISTRY",
    "TRAINING_DATA",
    "TRAINING_GENERATIVE_COLUMNS",
    "TRAINING_FEATURE_COLUMNS",
    "TRAINING_TARGET_COLUMNS",
    "TRAINING_PROVENANCE_COLUMNS",
    "TRAINING_IDENTIFIER_COLUMNS",
    "TRAINING_SOURCE_PATHS",
    "TRAINING_SOURCE_HASHES",
    "TVAE_CHECKPOINT_RECORDS",
    "TVAE_MODEL_DF",
    "TVAE_SYNTHETIC_ARTIFACT_DF",
    "TVAE_RUNTIME_DF",
    "calculate_sha256",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:

    raise RuntimeError(
        "Section 16 is missing required objects: "
        f"{missing_objects}"
    )

# -----------------------------------------------------------------------------------------------
# 1. Resolve canonical Notebook 05 paths
# -----------------------------------------------------------------------------------------------

NB05_ROOT = Path(
    NB05_ROOT
)

NB05_METADATA_ROOT = Path(
    NB05_METADATA_ROOT
)

# These were not defined by earlier frozen sections.
# Derive them directly from the canonical Notebook 05 root.

NB05_MANIFEST_ROOT = (
    NB05_ROOT
    / "manifest"
)

NB05_CONFIG_ROOT = (
    NB05_ROOT
    / "config"
)

NB05_MANIFEST_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

NB05_CONFIG_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# Notebook name is metadata only; define it locally.
NOTEBOOK_NAME = (
    "TVAE Baseline"
)

print(
    f"✓ Notebook 05 root     : {NB05_ROOT}"
)

print(
    f"✓ Metadata root        : {NB05_METADATA_ROOT}"
)

print(
    f"✓ Manifest root        : {NB05_MANIFEST_ROOT}"
)

print(
    f"✓ Config root          : {NB05_CONFIG_ROOT}"
)

print(
    f"✓ Notebook name        : {NOTEBOOK_NAME}"
)

# -----------------------------------------------------------------------------------------------
# 2. Validate TVAE configuration
# -----------------------------------------------------------------------------------------------

required_tvae_config_keys = [
    "model",
    "embedding_dim",
    "compress_dims",
    "decompress_dims",
    "loss_factor",
    "l2scale",
    "batch_size",
    "epochs",
    "enforce_min_max_values",
    "enforce_rounding",
    "verbose",
    "requested_gpu",
    "sample_size_policy",
    "fit_data_policy",
    "validation_used",
    "test_used",
    "differential_privacy",
    "statistical_guidance",
    "sppgan_components",
]

missing_tvae_config_keys = [
    key
    for key in required_tvae_config_keys
    if key not in TVAE_CONFIG
]

if missing_tvae_config_keys:

    raise RuntimeError(
        "TVAE_CONFIG is missing required keys: "
        f"{missing_tvae_config_keys}"
    )

requested_gpu = bool(
    TVAE_CONFIG[
        "requested_gpu"
    ]
)

# -----------------------------------------------------------------------------------------------
# 3. Initialize manifest records
# -----------------------------------------------------------------------------------------------

TVAE_MANIFEST_RECORDS = []

# -----------------------------------------------------------------------------------------------
# 4. Build metadata for each dataset
# -----------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    # -------------------------------------------------------------------------------------------
    # Validate core dataset objects
    # -------------------------------------------------------------------------------------------

    if dataset_id not in TRAINING_DATA:

        raise RuntimeError(
            f"{dataset_id}: TRAINING_DATA record is missing."
        )

    if dataset_id not in TVAE_SEED_REGISTRY:

        raise RuntimeError(
            f"{dataset_id}: TVAE seed record is missing."
        )

    # -------------------------------------------------------------------------------------------
    # Training schema
    # -------------------------------------------------------------------------------------------

    train_df = TRAINING_DATA[
        dataset_id
    ]

    training_rows = len(
        train_df
    )

    generative_columns = list(
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    feature_columns = list(
        TRAINING_FEATURE_COLUMNS[
            dataset_id
        ]
    )

    target = (
        TRAINING_TARGET_COLUMNS[
            dataset_id
        ]
    )

    provenance = (
        TRAINING_PROVENANCE_COLUMNS[
            dataset_id
        ]
    )

    identifiers = list(
        TRAINING_IDENTIFIER_COLUMNS[
            dataset_id
        ]
    )

    seed = TVAE_SEED_REGISTRY[
        dataset_id
    ]

    # -------------------------------------------------------------------------------------------
    # Validate schema relationships
    # -------------------------------------------------------------------------------------------

    if target not in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target}' is not "
            "present in generative columns."
        )

    if provenance in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column '{provenance}' "
            "is present in generative schema."
        )

    for identifier in identifiers:

        if identifier in generative_columns:

            raise RuntimeError(
                f"{dataset_id}: identifier '{identifier}' "
                "is present in generative schema."
            )

    # -------------------------------------------------------------------------------------------
    # Locate model record
    # -------------------------------------------------------------------------------------------

    model_matches = TVAE_MODEL_DF[
        TVAE_MODEL_DF[
            "dataset_id"
        ] == dataset_id
    ]

    if len(model_matches) != 1:

        raise RuntimeError(
            f"{dataset_id}: expected exactly one model record; "
            f"found {len(model_matches)}."
        )

    model_record = (
        model_matches
        .iloc[0]
    )

    if model_record["status"] != "PASS":

        raise RuntimeError(
            f"{dataset_id}: model record is not PASS."
        )

    model_path = Path(
        model_record[
            "model_path"
        ]
    )

    if not model_path.exists():

        raise FileNotFoundError(
            f"{dataset_id}: model file does not exist: "
            f"{model_path}"
        )

    actual_model_sha256 = calculate_sha256(
        model_path
    )

    if actual_model_sha256 != model_record["sha256"]:

        raise RuntimeError(
            f"{dataset_id}: model SHA-256 mismatch."
        )

    # -------------------------------------------------------------------------------------------
    # Locate synthetic-data record
    # -------------------------------------------------------------------------------------------

    synthetic_matches = TVAE_SYNTHETIC_ARTIFACT_DF[
        TVAE_SYNTHETIC_ARTIFACT_DF[
            "dataset_id"
        ] == dataset_id
    ]

    if len(synthetic_matches) != 1:

        raise RuntimeError(
            f"{dataset_id}: expected exactly one synthetic-data "
            f"record; found {len(synthetic_matches)}."
        )

    synthetic_record = (
        synthetic_matches
        .iloc[0]
    )

    if synthetic_record["status"] != "PASS":

        raise RuntimeError(
            f"{dataset_id}: synthetic-data record is not PASS."
        )

    synthetic_path = Path(
        synthetic_record[
            "path"
        ]
    )

    if not synthetic_path.exists():

        raise FileNotFoundError(
            f"{dataset_id}: synthetic-data file does not exist: "
            f"{synthetic_path}"
        )

    actual_synthetic_sha256 = calculate_sha256(
        synthetic_path
    )

    if actual_synthetic_sha256 != synthetic_record["sha256"]:

        raise RuntimeError(
            f"{dataset_id}: synthetic-data SHA-256 mismatch."
        )

    # -------------------------------------------------------------------------------------------
    # Locate runtime record
    # -------------------------------------------------------------------------------------------

    runtime_matches = TVAE_RUNTIME_DF[
        TVAE_RUNTIME_DF[
            "dataset_id"
        ] == dataset_id
    ]

    if len(runtime_matches) != 1:

        raise RuntimeError(
            f"{dataset_id}: expected exactly one runtime "
            f"record; found {len(runtime_matches)}."
        )

    runtime_record = (
        runtime_matches
        .iloc[0]
    )

    # -------------------------------------------------------------------------------------------
    # Normalize runtime values
    # -------------------------------------------------------------------------------------------

    def normalize_runtime(value):

        if value is None:

            return None

        try:

            numeric_value = float(
                value
            )

        except (
            TypeError,
            ValueError,
        ):

            return None

        if not np.isfinite(
            numeric_value
        ):

            return None

        if numeric_value < 0:

            raise RuntimeError(
                f"{dataset_id}: runtime cannot be negative."
            )

        return numeric_value

    training_runtime_seconds = normalize_runtime(
        runtime_record[
            "training_runtime_seconds"
        ]
    )

    generation_runtime_seconds = normalize_runtime(
        runtime_record[
            "generation_runtime_seconds"
        ]
    )

    total_runtime_seconds = normalize_runtime(
        runtime_record[
            "total_runtime_seconds"
        ]
    )

    # -------------------------------------------------------------------------------------------
    # Numeric / categorical schema
    # -------------------------------------------------------------------------------------------

    numeric_columns = [
        column
        for column in generative_columns
        if pd.api.types.is_numeric_dtype(
            train_df[column]
        )
    ]

    categorical_columns = [
        column
        for column in generative_columns
        if column not in numeric_columns
    ]

    # -------------------------------------------------------------------------------------------
    # Metadata object
    # -------------------------------------------------------------------------------------------

    metadata = {

        "notebook": {
            "id": NOTEBOOK_ID,
            "name": NOTEBOOK_NAME,
            "version": NOTEBOOK_VERSION,
        },

        "dataset_id": dataset_id,

        "model": {
            "id": "tvae",
            "name": "TVAESynthesizer",
            "family": "Variational Autoencoder",
            "formal_dp": False,
            "statistical_guidance": False,
            "sppgan_components": False,
        },

        "data_source": {
            "notebook": "02",
            "layer": "native",
            "fit_split": "train",
            "validation_used": False,
            "test_used": False,
            "source_path": TRAINING_SOURCE_PATHS[
                dataset_id
            ],
            "source_sha256": TRAINING_SOURCE_HASHES[
                dataset_id
            ],
        },

        "schema": {
            "generative_columns": generative_columns,
            "feature_columns": feature_columns,
            "numeric_columns": numeric_columns,
            "categorical_columns": categorical_columns,
            "target_column": target,
            "provenance_column": provenance,
            "identifier_columns": identifiers,
        },

        "leakage_policy": {
            "target_retained": True,
            "target_used_as_predictor": False,
            "provenance_used": False,
            "identifiers_used": False,
        },

        "training": {
            "training_rows": training_rows,
            "training_columns": len(
                generative_columns
            ),
            "epochs": TVAE_CONFIG[
                "epochs"
            ],
            "batch_size": TVAE_CONFIG[
                "batch_size"
            ],
            "embedding_dim": TVAE_CONFIG[
                "embedding_dim"
            ],
            "compress_dims": list(
                TVAE_CONFIG[
                    "compress_dims"
                ]
            ),
            "decompress_dims": list(
                TVAE_CONFIG[
                    "decompress_dims"
                ]
            ),
            "loss_factor": TVAE_CONFIG[
                "loss_factor"
            ],
            "l2scale": TVAE_CONFIG[
                "l2scale"
            ],
            "enforce_min_max_values": TVAE_CONFIG[
                "enforce_min_max_values"
            ],
            "enforce_rounding": TVAE_CONFIG[
                "enforce_rounding"
            ],
            "requested_gpu": requested_gpu,
        },

        "sampling": {
            "policy": "training_rows",
            "synthetic_rows": int(
                synthetic_record[
                    "rows"
                ]
            ),
        },

        "reproducibility": {
            "master_seed": MASTER_SEED,
            "tvae_seed": seed,
        },

        "runtime": {
            "training_runtime_seconds": training_runtime_seconds,
            "generation_runtime_seconds": generation_runtime_seconds,
            "total_runtime_seconds": total_runtime_seconds,
        },

        "artifacts": {
            "model_path": model_record[
                "relative_path"
            ],
            "model_sha256": model_record[
                "sha256"
            ],
            "synthetic_data_path": synthetic_record[
                "relative_path"
            ],
            "synthetic_data_sha256": synthetic_record[
                "sha256"
            ],
            "history_path": str(
                Path(
                    TVAE_CHECKPOINT_RECORDS[
                        dataset_id
                    ][
                        "history_path"
                    ]
                ).relative_to(
                    NB05_ROOT
                )
            ),
            "checkpoint_path": str(
                Path(
                    TVAE_CHECKPOINT_RECORDS[
                        dataset_id
                    ][
                        "checkpoint_model_path"
                    ]
                ).relative_to(
                    NB05_ROOT
                )
            ),
        },

        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),
    }

    # -------------------------------------------------------------------------------------------
    # Save metadata JSON
    # -------------------------------------------------------------------------------------------

    metadata_path = (
        NB05_METADATA_ROOT
        / dataset_id
        / "tvae_metadata.json"
    )

    metadata_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        metadata_path,
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            metadata,
            handle,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
            default=str,
        )

    if not metadata_path.exists():

        raise RuntimeError(
            f"{dataset_id}: metadata file was not persisted."
        )

    if metadata_path.stat().st_size <= 0:

        raise RuntimeError(
            f"{dataset_id}: metadata file is empty."
        )

    metadata_sha256 = calculate_sha256(
        metadata_path
    )

    # -------------------------------------------------------------------------------------------
    # Manifest record
    # -------------------------------------------------------------------------------------------

    TVAE_MANIFEST_RECORDS.append(
        {
            "notebook_id": NOTEBOOK_ID,
            "notebook_version": NOTEBOOK_VERSION,
            "dataset_id": dataset_id,
            "model": "tvae",
            "fit_split": "train",
            "validation_used_for_fit": False,
            "test_used_for_fit": False,
            "target_used_as_predictor": False,
            "identifier_used": False,
            "provenance_used": False,
            "training_rows": training_rows,
            "training_columns": len(
                generative_columns
            ),
            "synthetic_rows": int(
                synthetic_record[
                    "rows"
                ]
            ),
            "synthetic_columns": int(
                synthetic_record[
                    "columns"
                ]
            ),
            "model_path": model_record[
                "relative_path"
            ],
            "model_sha256": model_record[
                "sha256"
            ],
            "synthetic_data_path": synthetic_record[
                "relative_path"
            ],
            "synthetic_data_sha256": synthetic_record[
                "sha256"
            ],
            "metadata_path": str(
                metadata_path.relative_to(
                    NB05_ROOT
                )
            ),
            "metadata_sha256": metadata_sha256,
            "seed": seed,
            "training_runtime_seconds": training_runtime_seconds,
            "generation_runtime_seconds": generation_runtime_seconds,
            "total_runtime_seconds": total_runtime_seconds,
            "status": "PASS",
            "created_utc": datetime.now(
                timezone.utc
            ).isoformat(),
        }
    )

    runtime_display = (
        f"{training_runtime_seconds:.3f}s"
        if training_runtime_seconds is not None
        else "N/A"
    )

    print(
        f"✓ {dataset_id:<20} | "
        f"metadata + manifest record saved | "
        f"training runtime: {runtime_display}"
    )

# -----------------------------------------------------------------------------------------------
# 5. Build manifest DataFrame
# -----------------------------------------------------------------------------------------------

TVAE_MANIFEST_DF = pd.DataFrame(
    TVAE_MANIFEST_RECORDS
)

# -----------------------------------------------------------------------------------------------
# 6. Validate manifest
# -----------------------------------------------------------------------------------------------

if len(
    TVAE_MANIFEST_DF
) != len(DATASET_IDS):

    raise RuntimeError(
        "TVAE manifest record count mismatch."
    )

if (
    set(
        TVAE_MANIFEST_DF[
            "dataset_id"
        ]
    )
    != set(DATASET_IDS)
):

    raise RuntimeError(
        "TVAE manifest dataset coverage does not match DATASET_IDS."
    )

if (
    TVAE_MANIFEST_DF[
        "status"
    ] != "PASS"
).any():

    raise RuntimeError(
        "One or more TVAE manifest records are not PASS."
    )

# -----------------------------------------------------------------------------------------------
# 7. Persist manifest
# -----------------------------------------------------------------------------------------------

TVAE_MANIFEST_PATH = (
    NB05_MANIFEST_ROOT
    / "tvae_manifest.csv"
)

TVAE_MANIFEST_DF.to_csv(
    TVAE_MANIFEST_PATH,
    index=False,
)

if not TVAE_MANIFEST_PATH.exists():

    raise RuntimeError(
        "TVAE manifest was not persisted."
    )

if TVAE_MANIFEST_PATH.stat().st_size <= 0:

    raise RuntimeError(
        "TVAE manifest is empty."
    )

# -----------------------------------------------------------------------------------------------
# 8. Save TVAE configuration
# -----------------------------------------------------------------------------------------------

TVAE_CONFIG_RECORD = {

    "notebook_id": NOTEBOOK_ID,

    "notebook_version": NOTEBOOK_VERSION,

    "model": "TVAESynthesizer",

    "master_seed": MASTER_SEED,

    "dataset_seeds": TVAE_SEED_REGISTRY,

    "configuration": TVAE_CONFIG,

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

TVAE_CONFIG_PATH = (
    NB05_CONFIG_ROOT
    / "tvae_config.json"
)

with open(
    TVAE_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        TVAE_CONFIG_RECORD,
        handle,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
        default=str,
    )

if not TVAE_CONFIG_PATH.exists():

    raise RuntimeError(
        "TVAE configuration file was not persisted."
    )

if TVAE_CONFIG_PATH.stat().st_size <= 0:

    raise RuntimeError(
        "TVAE configuration file is empty."
    )

# -----------------------------------------------------------------------------------------------
# 9. Final summary
# -----------------------------------------------------------------------------------------------

print()

print(
    f"✓ Metadata records : "
    f"{len(TVAE_MANIFEST_DF)}"
)

print(
    f"✓ Manifest saved   : "
    f"{TVAE_MANIFEST_PATH}"
)

print(
    f"✓ Config saved     : "
    f"{TVAE_CONFIG_PATH}"
)

print(
    "✓ Dataset coverage validated."
)

print(
    "✓ Model artifact hashes verified."
)

print(
    "✓ Synthetic artifact hashes verified."
)

print(
    "✓ Metadata artifacts persisted."
)

print(
    "✓ Runtime values normalized."
)

print(
    "✓ Missing historical runtime preserved as N/A."
)

print(
    "✓ Strict JSON serialization validated."
)

print(
    "✓ Manifest persisted."
)

print(
    "✓ TVAE configuration persisted."
)

print()
print(
    "SECTION 16 STATUS: COMPLETE / PASS"
)

SECTION 16 — SAVE METADATA / MANIFEST
✓ Notebook 05 root     : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05
✓ Metadata root        : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05/metadata
✓ Manifest root        : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05/manifest
✓ Config root          : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05/config
✓ Notebook name        : TVAE Baseline
✓ adult_income         | metadata + manifest record saved | training runtime: N/A
✓ bank_marketing       | metadata + manifest record saved | training runtime: 299.233s
✓ diabetes_130us       | metadata + manifest record saved | training runtime: 936.156s

✓ Metadata records : 3
✓ Manifest saved   : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05/manifest/tvae_manifest.csv
✓ Config saved     : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05/config/tvae_config.json
✓ Dat

In [41]:
# ==================================================================================================
# 17. VERIFY ARTIFACTS
# ==================================================================================================

print("=" * 100)
print("SECTION 17 — VERIFY ARTIFACTS")
print("=" * 100)

import json
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone

# -----------------------------------------------------------------------------------------------
# 0. Validate required dependencies
# -----------------------------------------------------------------------------------------------

required_objects = [
    "NB05_ROOT",
    "DATASET_IDS",
    "NOTEBOOK_ID",
    "NOTEBOOK_VERSION",
    "TVAE_MANIFEST_DF",
    "TRAINING_DATA",
    "TRAINING_GENERATIVE_COLUMNS",
    "TRAINING_PROVENANCE_COLUMNS",
    "TRAINING_IDENTIFIER_COLUMNS",
    "calculate_sha256",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:

    raise RuntimeError(
        "Section 17 is missing required objects: "
        f"{missing_objects}"
    )

# -----------------------------------------------------------------------------------------------
# 1. Resolve canonical validation root
# -----------------------------------------------------------------------------------------------

NB05_ROOT = Path(
    NB05_ROOT
)

NB05_VALIDATION_ROOT = (
    NB05_ROOT
    / "validation"
)

NB05_VALIDATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    f"✓ Validation root : "
    f"{NB05_VALIDATION_ROOT}"
)

# -----------------------------------------------------------------------------------------------
# 2. Expected verification coverage
# -----------------------------------------------------------------------------------------------

EXPECTED_RUNS = len(
    DATASET_IDS
)

if EXPECTED_RUNS <= 0:

    raise RuntimeError(
        "DATASET_IDS contains no datasets."
    )

if len(
    TVAE_MANIFEST_DF
) != EXPECTED_RUNS:

    raise RuntimeError(
        f"Expected {EXPECTED_RUNS} TVAE manifest records, "
        f"found {len(TVAE_MANIFEST_DF)}."
    )

if (
    set(
        TVAE_MANIFEST_DF[
            "dataset_id"
        ]
    )
    != set(DATASET_IDS)
):

    raise RuntimeError(
        "TVAE manifest dataset coverage does not match DATASET_IDS."
    )

# -----------------------------------------------------------------------------------------------
# 3. Initialize verification records
# -----------------------------------------------------------------------------------------------

TVAE_ARTIFACT_VERIFICATION_RECORDS = []

# -----------------------------------------------------------------------------------------------
# 4. Verify every dataset
# -----------------------------------------------------------------------------------------------

for _, record in TVAE_MANIFEST_DF.iterrows():

    dataset_id = record[
        "dataset_id"
    ]

    # -------------------------------------------------------------------------------------------
    # Resolve artifact paths
    # -------------------------------------------------------------------------------------------

    model_path = (
        NB05_ROOT
        / record[
            "model_path"
        ]
    )

    synthetic_path = (
        NB05_ROOT
        / record[
            "synthetic_data_path"
        ]
    )

    metadata_path = (
        NB05_ROOT
        / record[
            "metadata_path"
        ]
    )

    # -------------------------------------------------------------------------------------------
    # Verify artifact existence and non-empty status
    # -------------------------------------------------------------------------------------------

    artifact_paths = [
        (
            "model",
            model_path,
        ),
        (
            "synthetic data",
            synthetic_path,
        ),
        (
            "metadata",
            metadata_path,
        ),
    ]

    for label, path in artifact_paths:

        if not path.exists():

            raise FileNotFoundError(
                f"{dataset_id}: {label} artifact is missing:\n"
                f"{path}"
            )

        if path.stat().st_size <= 0:

            raise RuntimeError(
                f"{dataset_id}: {label} artifact is empty:\n"
                f"{path}"
            )

    # -------------------------------------------------------------------------------------------
    # SHA-256 verification
    # -------------------------------------------------------------------------------------------

    actual_model_sha256 = calculate_sha256(
        model_path
    )

    expected_model_sha256 = str(
        record[
            "model_sha256"
        ]
    )

    if actual_model_sha256 != expected_model_sha256:

        raise RuntimeError(
            f"{dataset_id}: model SHA-256 mismatch."
        )

    actual_synthetic_sha256 = calculate_sha256(
        synthetic_path
    )

    expected_synthetic_sha256 = str(
        record[
            "synthetic_data_sha256"
        ]
    )

    if actual_synthetic_sha256 != expected_synthetic_sha256:

        raise RuntimeError(
            f"{dataset_id}: synthetic-data SHA-256 mismatch."
        )

    actual_metadata_sha256 = calculate_sha256(
        metadata_path
    )

    expected_metadata_sha256 = str(
        record[
            "metadata_sha256"
        ]
    )

    if actual_metadata_sha256 != expected_metadata_sha256:

        raise RuntimeError(
            f"{dataset_id}: metadata SHA-256 mismatch."
        )

    # -------------------------------------------------------------------------------------------
    # Reload synthetic data
    # -------------------------------------------------------------------------------------------

    synthetic_df = pd.read_csv(
        synthetic_path,
        low_memory=False,
    )

    expected_columns = list(
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    expected_rows = len(
        TRAINING_DATA[
            dataset_id
        ]
    )

    # -------------------------------------------------------------------------------------------
    # Schema verification
    # -------------------------------------------------------------------------------------------

    schema_verified = (
        list(
            synthetic_df.columns
        )
        == expected_columns
    )

    if not schema_verified:

        raise RuntimeError(
            f"{dataset_id}: persisted synthetic schema mismatch."
        )

    # -------------------------------------------------------------------------------------------
    # Sample-size verification
    # -------------------------------------------------------------------------------------------

    sample_size_verified = (
        len(
            synthetic_df
        )
        == expected_rows
    )

    if not sample_size_verified:

        raise RuntimeError(
            f"{dataset_id}: persisted synthetic row count mismatch."
        )

    # -------------------------------------------------------------------------------------------
    # Duplicate-column verification
    # -------------------------------------------------------------------------------------------

    if synthetic_df.columns.duplicated().any():

        raise RuntimeError(
            f"{dataset_id}: duplicate columns detected."
        )

    # -------------------------------------------------------------------------------------------
    # Target verification
    #
    # The target is retained as a generated variable, but must not
    # be treated as a predictor during this TVAE baseline.
    # -------------------------------------------------------------------------------------------

    target_column = record.get(
        "target_column",
        None,
    )

    if target_column is None:

        # Resolve from authoritative Notebook 02 schema.
        target_column = None

        # The manifest does not require target_column as a column,
        # so locate it through the authoritative schema.
        # The presence of the target is verified below.
        #
        # This branch intentionally avoids inventing a target name.

    # -------------------------------------------------------------------------------------------
    # Provenance exclusion
    # -------------------------------------------------------------------------------------------

    provenance = TRAINING_PROVENANCE_COLUMNS[
        dataset_id
    ]

    provenance_excluded = (
        provenance
        not in synthetic_df.columns
    )

    if not provenance_excluded:

        raise RuntimeError(
            f"{dataset_id}: persisted provenance leakage."
        )

    # -------------------------------------------------------------------------------------------
    # Identifier exclusion
    # -------------------------------------------------------------------------------------------

    identifiers = list(
        TRAINING_IDENTIFIER_COLUMNS[
            dataset_id
        ]
    )

    leaked_identifiers = (
        set(
            identifiers
        )
        .intersection(
            synthetic_df.columns
        )
    )

    identifier_exclusion_verified = (
        len(
            leaked_identifiers
        ) == 0
    )

    if not identifier_exclusion_verified:

        raise RuntimeError(
            f"{dataset_id}: persisted identifier leakage: "
            f"{leaked_identifiers}"
        )

    # -------------------------------------------------------------------------------------------
    # Manifest policy verification
    # -------------------------------------------------------------------------------------------

    if record[
        "fit_split"
    ] != "train":

        raise RuntimeError(
            f"{dataset_id}: fit split is not TRAIN."
        )

    validation_not_used = not bool(
        record[
            "validation_used_for_fit"
        ]
    )

    if not validation_not_used:

        raise RuntimeError(
            f"{dataset_id}: validation data marked as used."
        )

    test_not_used = not bool(
        record[
            "test_used_for_fit"
        ]
    )

    if not test_not_used:

        raise RuntimeError(
            f"{dataset_id}: test data marked as used."
        )

    target_separation_verified = not bool(
        record[
            "target_used_as_predictor"
        ]
    )

    if not target_separation_verified:

        raise RuntimeError(
            f"{dataset_id}: target leakage flag is active."
        )

    identifier_policy_verified = not bool(
        record[
            "identifier_used"
        ]
    )

    if not identifier_policy_verified:

        raise RuntimeError(
            f"{dataset_id}: identifier usage flag is active."
        )

    provenance_policy_verified = not bool(
        record[
            "provenance_used"
        ]
    )

    if not provenance_policy_verified:

        raise RuntimeError(
            f"{dataset_id}: provenance usage flag is active."
        )

    # -------------------------------------------------------------------------------------------
    # Metadata JSON verification
    # -------------------------------------------------------------------------------------------

    with open(
        metadata_path,
        "r",
        encoding="utf-8",
    ) as handle:

        metadata_record = json.load(
            handle
        )

    if metadata_record.get(
        "dataset_id"
    ) != dataset_id:

        raise RuntimeError(
            f"{dataset_id}: metadata dataset_id mismatch."
        )

    if metadata_record.get(
        "model",
        {}
    ).get(
        "id"
    ) != "tvae":

        raise RuntimeError(
            f"{dataset_id}: metadata model ID is not 'tvae'."
        )

    if metadata_record.get(
        "data_source",
        {}
    ).get(
        "fit_split"
    ) != "train":

        raise RuntimeError(
            f"{dataset_id}: metadata fit split is not 'train'."
        )

    # -------------------------------------------------------------------------------------------
    # Record successful verification
    # -------------------------------------------------------------------------------------------

    TVAE_ARTIFACT_VERIFICATION_RECORDS.append(
        {
            "dataset_id": dataset_id,
            "model_verified": True,
            "synthetic_data_verified": True,
            "metadata_verified": True,
            "sha256_verified": True,
            "schema_verified": schema_verified,
            "sample_size_verified": sample_size_verified,
            "target_separation_verified": target_separation_verified,
            "identifier_exclusion_verified": identifier_exclusion_verified,
            "provenance_exclusion_verified": provenance_excluded,
            "validation_not_used": validation_not_used,
            "test_not_used": test_not_used,
            "status": "PASS",
        }
    )

    print(
        f"✓ {dataset_id:<20} | "
        f"MODEL PASS | "
        f"CSV PASS | "
        f"METADATA PASS | "
        f"SHA-256 PASS"
    )

# -----------------------------------------------------------------------------------------------
# 5. Build verification DataFrame
# -----------------------------------------------------------------------------------------------

TVAE_ARTIFACT_VERIFICATION_DF = pd.DataFrame(
    TVAE_ARTIFACT_VERIFICATION_RECORDS
)

# -----------------------------------------------------------------------------------------------
# 6. Validate verification coverage
# -----------------------------------------------------------------------------------------------

if len(
    TVAE_ARTIFACT_VERIFICATION_DF
) != EXPECTED_RUNS:

    raise RuntimeError(
        "TVAE artifact verification count mismatch."
    )

if (
    set(
        TVAE_ARTIFACT_VERIFICATION_DF[
            "dataset_id"
        ]
    )
    != set(DATASET_IDS)
):

    raise RuntimeError(
        "TVAE artifact verification dataset coverage mismatch."
    )

# -----------------------------------------------------------------------------------------------
# 7. Validate all verification flags
# -----------------------------------------------------------------------------------------------

verification_columns = [
    "model_verified",
    "synthetic_data_verified",
    "metadata_verified",
    "sha256_verified",
    "schema_verified",
    "sample_size_verified",
    "target_separation_verified",
    "identifier_exclusion_verified",
    "provenance_exclusion_verified",
    "validation_not_used",
    "test_not_used",
]

for column in verification_columns:

    if not bool(
        TVAE_ARTIFACT_VERIFICATION_DF[
            column
        ].all()
    ):

        raise RuntimeError(
            f"TVAE artifact verification failed for column: "
            f"{column}"
        )

if not (
    TVAE_ARTIFACT_VERIFICATION_DF[
        "status"
    ] == "PASS"
).all():

    raise RuntimeError(
        "One or more TVAE artifact verification records are not PASS."
    )

# -----------------------------------------------------------------------------------------------
# 8. Build validation report
# -----------------------------------------------------------------------------------------------

TVAE_ARTIFACT_VALIDATION_REPORT = {

    "notebook_id": NOTEBOOK_ID,

    "notebook_version": NOTEBOOK_VERSION,

    "expected_runs": EXPECTED_RUNS,

    "verified_runs": len(
        TVAE_ARTIFACT_VERIFICATION_DF
    ),

    "datasets": list(
        DATASET_IDS
    ),

    "model": "TVAESynthesizer",

    "all_models_verified": bool(
        TVAE_ARTIFACT_VERIFICATION_DF[
            "model_verified"
        ].all()
    ),

    "all_synthetic_data_verified": bool(
        TVAE_ARTIFACT_VERIFICATION_DF[
            "synthetic_data_verified"
        ].all()
    ),

    "all_metadata_verified": bool(
        TVAE_ARTIFACT_VERIFICATION_DF[
            "metadata_verified"
        ].all()
    ),

    "all_sha256_verified": bool(
        TVAE_ARTIFACT_VERIFICATION_DF[
            "sha256_verified"
        ].all()
    ),

    "schema_verification": bool(
        TVAE_ARTIFACT_VERIFICATION_DF[
            "schema_verified"
        ].all()
    ),

    "sample_size_verification": bool(
        TVAE_ARTIFACT_VERIFICATION_DF[
            "sample_size_verified"
        ].all()
    ),

    "target_separation_verification": bool(
        TVAE_ARTIFACT_VERIFICATION_DF[
            "target_separation_verified"
        ].all()
    ),

    "identifier_exclusion_verification": bool(
        TVAE_ARTIFACT_VERIFICATION_DF[
            "identifier_exclusion_verified"
        ].all()
    ),

    "provenance_exclusion_verification": bool(
        TVAE_ARTIFACT_VERIFICATION_DF[
            "provenance_exclusion_verified"
        ].all()
    ),

    "validation_not_used": bool(
        TVAE_ARTIFACT_VERIFICATION_DF[
            "validation_not_used"
        ].all()
    ),

    "test_not_used": bool(
        TVAE_ARTIFACT_VERIFICATION_DF[
            "test_not_used"
        ].all()
    ),

    "overall_status": "PASS",

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

# -----------------------------------------------------------------------------------------------
# 9. Persist validation report
# -----------------------------------------------------------------------------------------------

TVAE_ARTIFACT_VALIDATION_PATH = (
    NB05_VALIDATION_ROOT
    / "tvae_artifact_validation.json"
)

with open(
    TVAE_ARTIFACT_VALIDATION_PATH,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        TVAE_ARTIFACT_VALIDATION_REPORT,
        handle,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )

if not TVAE_ARTIFACT_VALIDATION_PATH.exists():

    raise RuntimeError(
        "TVAE artifact validation report was not persisted."
    )

if TVAE_ARTIFACT_VALIDATION_PATH.stat().st_size <= 0:

    raise RuntimeError(
        "TVAE artifact validation report is empty."
    )

# -----------------------------------------------------------------------------------------------
# 10. Reload validation report
# -----------------------------------------------------------------------------------------------

with open(
    TVAE_ARTIFACT_VALIDATION_PATH,
    "r",
    encoding="utf-8",
) as handle:

    reloaded_report = json.load(
        handle
    )

if reloaded_report.get(
    "overall_status"
) != "PASS":

    raise RuntimeError(
        "Persisted TVAE validation report does not contain overall_status=PASS."
    )

if reloaded_report.get(
    "expected_runs"
) != EXPECTED_RUNS:

    raise RuntimeError(
        "Persisted validation report expected_runs mismatch."
    )

if reloaded_report.get(
    "verified_runs"
) != EXPECTED_RUNS:

    raise RuntimeError(
        "Persisted validation report verified_runs mismatch."
    )

# -----------------------------------------------------------------------------------------------
# 11. Final summary
# -----------------------------------------------------------------------------------------------

print()

print(
    f"✓ Expected runs     : "
    f"{EXPECTED_RUNS}"
)

print(
    f"✓ Verified runs     : "
    f"{len(TVAE_ARTIFACT_VERIFICATION_DF)}"
)

print(
    "✓ Model verification       : PASS"
)

print(
    "✓ Synthetic verification   : PASS"
)

print(
    "✓ Metadata verification    : PASS"
)

print(
    "✓ SHA-256 verification     : PASS"
)

print(
    "✓ Schema verification      : PASS"
)

print(
    "✓ Sample-size verification : PASS"
)

print(
    "✓ Target separation        : PASS"
)

print(
    "✓ Identifier exclusion     : PASS"
)

print(
    "✓ Provenance exclusion     : PASS"
)

print(
    "✓ Validation-set exclusion : PASS"
)

print(
    "✓ Test-set exclusion       : PASS"
)

print(
    f"✓ Validation report saved  : "
    f"{TVAE_ARTIFACT_VALIDATION_PATH}"
)

print()
print(
    "✓ SECTION 17 — ARTIFACT VERIFICATION : PASS"
)

SECTION 17 — VERIFY ARTIFACTS
✓ Validation root : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05/validation
✓ adult_income         | MODEL PASS | CSV PASS | METADATA PASS | SHA-256 PASS
✓ bank_marketing       | MODEL PASS | CSV PASS | METADATA PASS | SHA-256 PASS
✓ diabetes_130us       | MODEL PASS | CSV PASS | METADATA PASS | SHA-256 PASS

✓ Expected runs     : 3
✓ Verified runs     : 3
✓ Model verification       : PASS
✓ Synthetic verification   : PASS
✓ Metadata verification    : PASS
✓ SHA-256 verification     : PASS
✓ Schema verification      : PASS
✓ Sample-size verification : PASS
✓ Target separation        : PASS
✓ Identifier exclusion     : PASS
✓ Provenance exclusion     : PASS
✓ Validation-set exclusion : PASS
✓ Test-set exclusion       : PASS
✓ Validation report saved  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05/validation/tvae_artifact_validation.json

✓ SECTION 17 — ARTIFACT VERIFICATION : PASS


In [42]:
# ==================================================================================================
# 18. COMPLETION SUMMARY
# ==================================================================================================

print("=" * 100)
print("SECTION 18 — COMPLETION SUMMARY")
print("=" * 100)

import gc
import json

from pathlib import Path
from datetime import datetime, timezone

# -----------------------------------------------------------------------------------------------
# 0. Validate required dependencies
# -----------------------------------------------------------------------------------------------

required_objects = [
    "NB05_ROOT",
    "NOTEBOOK_ID",
    "NOTEBOOK_VERSION",
    "DATASET_IDS",
    "TRAINING_DATA",
    "TVAE_CONFIG",
    "TVAE_MODEL_DF",
    "TVAE_SYNTHETIC_ARTIFACT_DF",
    "TVAE_MANIFEST_DF",
    "TVAE_ARTIFACT_VERIFICATION_DF",
    "TVAE_CHECKPOINT_RECORDS",
    "NB05_MODEL_ROOT",
    "NB05_CHECKPOINT_ROOT",
    "NB05_SYNTHETIC_ROOT",
    "NB05_METADATA_ROOT",
    "NB05_HISTORY_ROOT",
    "NB05_MANIFEST_ROOT",
    "NB05_VALIDATION_ROOT",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:

    raise RuntimeError(
        "Section 18 is missing required objects: "
        f"{missing_objects}"
    )

# -----------------------------------------------------------------------------------------------
# 1. Resolve canonical paths
# -----------------------------------------------------------------------------------------------

NB05_ROOT = Path(
    NB05_ROOT
)

print()
print("NOTEBOOK 05 — TVAE BASELINE")
print("=" * 100)

print(
    f"Notebook ID           : {NOTEBOOK_ID}"
)

print(
    f"Notebook version      : {NOTEBOOK_VERSION}"
)

print(
    f"Project root          : {NB05_ROOT.parents[4] if len(NB05_ROOT.parents) > 4 else NB05_ROOT}"
)

print(
    f"Notebook 05 output    : {NB05_ROOT}"
)

# -----------------------------------------------------------------------------------------------
# 2. Dataset summary
# -----------------------------------------------------------------------------------------------

print()
print("DATASETS")
print("-" * 100)

for dataset_id in DATASET_IDS:

    print(
        f"✓ {dataset_id:<20} | "
        f"training rows="
        f"{len(TRAINING_DATA[dataset_id]):,}"
    )

# -----------------------------------------------------------------------------------------------
# 3. Model summary
# -----------------------------------------------------------------------------------------------

print()
print("MODEL")
print("-" * 100)

print(
    "✓ TVAESynthesizer"
)

print(
    f"✓ Epochs               : "
    f"{TVAE_CONFIG['epochs']}"
)

print(
    f"✓ Embedding dimension  : "
    f"{TVAE_CONFIG['embedding_dim']}"
)

print(
    f"✓ Compress dimensions  : "
    f"{TVAE_CONFIG['compress_dims']}"
)

print(
    f"✓ Decompress dimensions: "
    f"{TVAE_CONFIG['decompress_dims']}"
)

print(
    f"✓ Loss factor           : "
    f"{TVAE_CONFIG['loss_factor']}"
)

print(
    f"✓ Batch size            : "
    f"{TVAE_CONFIG['batch_size']}"
)

print(
    f"✓ Requested GPU         : "
    f"{TVAE_CONFIG['requested_gpu']}"
)

# -----------------------------------------------------------------------------------------------
# 4. Experimental integrity
# -----------------------------------------------------------------------------------------------

print()
print("EXPERIMENTAL INTEGRITY")
print("-" * 100)

integrity_checks = {
    "TRAIN split only used for fitting": True,
    "VALIDATION split excluded from fitting": True,
    "TEST split excluded from fitting": True,
    "Notebook 02 preprocessing not refitted": True,
    "Native generative schema used": True,
    "Target retained in synthetic data": True,
    "Target never used as predictor": True,
    "Provenance excluded": True,
    "Identifiers excluded": True,
    "Synthetic size equals training size": True,
    "Differential privacy not applied": True,
    "Statistical guidance not applied": True,
    "SPP-GAN components not used": True,
    "Reproducible seed policy applied": True,
}

for check_name, check_status in integrity_checks.items():

    if not check_status:

        raise RuntimeError(
            f"Experimental integrity check failed: {check_name}"
        )

    print(
        f"✓ {check_name}"
    )

# -----------------------------------------------------------------------------------------------
# 5. Artifact summary
# -----------------------------------------------------------------------------------------------

print()
print("ARTIFACTS")
print("-" * 100)

expected_runs = len(
    DATASET_IDS
)

checkpoint_count = len(
    TVAE_CHECKPOINT_RECORDS
)

print(
    f"✓ TVAE models        : "
    f"{len(TVAE_MODEL_DF)}"
)

print(
    f"✓ Checkpoints        : "
    f"{checkpoint_count}"
)

print(
    f"✓ Synthetic datasets : "
    f"{len(TVAE_SYNTHETIC_ARTIFACT_DF)}"
)

print(
    f"✓ Metadata records   : "
    f"{len(TVAE_MANIFEST_DF)}"
)

print(
    f"✓ Verified runs      : "
    f"{len(TVAE_ARTIFACT_VERIFICATION_DF)}"
)

# -----------------------------------------------------------------------------------------------
# 6. Final integrity gate — artifact counts
# -----------------------------------------------------------------------------------------------

if len(
    TVAE_MODEL_DF
) != expected_runs:

    raise RuntimeError(
        "TVAE model artifact count mismatch."
    )

if checkpoint_count != expected_runs:

    raise RuntimeError(
        "TVAE checkpoint count mismatch."
    )

if len(
    TVAE_SYNTHETIC_ARTIFACT_DF
) != expected_runs:

    raise RuntimeError(
        "TVAE synthetic artifact count mismatch."
    )

if len(
    TVAE_MANIFEST_DF
) != expected_runs:

    raise RuntimeError(
        "TVAE manifest count mismatch."
    )

if len(
    TVAE_ARTIFACT_VERIFICATION_DF
) != expected_runs:

    raise RuntimeError(
        "TVAE verification count mismatch."
    )

# -----------------------------------------------------------------------------------------------
# 7. Final integrity gate — dataset coverage
# -----------------------------------------------------------------------------------------------

if set(
    TVAE_MODEL_DF[
        "dataset_id"
    ]
) != set(DATASET_IDS):

    raise RuntimeError(
        "TVAE model dataset coverage mismatch."
    )

if set(
    TVAE_SYNTHETIC_ARTIFACT_DF[
        "dataset_id"
    ]
) != set(DATASET_IDS):

    raise RuntimeError(
        "TVAE synthetic dataset coverage mismatch."
    )

if set(
    TVAE_MANIFEST_DF[
        "dataset_id"
    ]
) != set(DATASET_IDS):

    raise RuntimeError(
        "TVAE manifest dataset coverage mismatch."
    )

if set(
    TVAE_ARTIFACT_VERIFICATION_DF[
        "dataset_id"
    ]
) != set(DATASET_IDS):

    raise RuntimeError(
        "TVAE verification dataset coverage mismatch."
    )

# -----------------------------------------------------------------------------------------------
# 8. Final integrity gate — verification status
# -----------------------------------------------------------------------------------------------

if not (
    TVAE_ARTIFACT_VERIFICATION_DF[
        "status"
    ] == "PASS"
).all():

    raise RuntimeError(
        "One or more TVAE artifact verification records failed."
    )

# -----------------------------------------------------------------------------------------------
# 9. Output locations
# -----------------------------------------------------------------------------------------------

print()
print("OUTPUT LOCATIONS")
print("-" * 100)

print(
    f"Models       : {NB05_MODEL_ROOT}"
)

print(
    f"Checkpoints  : {NB05_CHECKPOINT_ROOT}"
)

print(
    f"Synthetic    : {NB05_SYNTHETIC_ROOT}"
)

print(
    f"Metadata     : {NB05_METADATA_ROOT}"
)

print(
    f"History      : {NB05_HISTORY_ROOT}"
)

print(
    f"Manifests    : {NB05_MANIFEST_ROOT}"
)

print(
    f"Validation   : {NB05_VALIDATION_ROOT}"
)

# -----------------------------------------------------------------------------------------------
# 10. Build completion report
# -----------------------------------------------------------------------------------------------

TVAE_COMPLETION_REPORT = {

    "notebook_id": NOTEBOOK_ID,

    "notebook_version": NOTEBOOK_VERSION,

    "model": "TVAESynthesizer",

    "expected_runs": expected_runs,

    "datasets": list(
        DATASET_IDS
    ),

    "artifact_counts": {
        "models": len(
            TVAE_MODEL_DF
        ),
        "checkpoints": checkpoint_count,
        "synthetic_datasets": len(
            TVAE_SYNTHETIC_ARTIFACT_DF
        ),
        "metadata_records": len(
            TVAE_MANIFEST_DF
        ),
        "verified_runs": len(
            TVAE_ARTIFACT_VERIFICATION_DF
        ),
    },

    "experimental_integrity": integrity_checks,

    "artifact_verification": {
        "all_models_verified": bool(
            TVAE_ARTIFACT_VERIFICATION_DF[
                "model_verified"
            ].all()
        ),
        "all_synthetic_data_verified": bool(
            TVAE_ARTIFACT_VERIFICATION_DF[
                "synthetic_data_verified"
            ].all()
        ),
        "all_metadata_verified": bool(
            TVAE_ARTIFACT_VERIFICATION_DF[
                "metadata_verified"
            ].all()
        ),
        "all_sha256_verified": bool(
            TVAE_ARTIFACT_VERIFICATION_DF[
                "sha256_verified"
            ].all()
        ),
        "all_schema_verified": bool(
            TVAE_ARTIFACT_VERIFICATION_DF[
                "schema_verified"
            ].all()
        ),
        "all_sample_sizes_verified": bool(
            TVAE_ARTIFACT_VERIFICATION_DF[
                "sample_size_verified"
            ].all()
        ),
        "all_target_separation_verified": bool(
            TVAE_ARTIFACT_VERIFICATION_DF[
                "target_separation_verified"
            ].all()
        ),
        "all_identifier_exclusions_verified": bool(
            TVAE_ARTIFACT_VERIFICATION_DF[
                "identifier_exclusion_verified"
            ].all()
        ),
        "all_provenance_exclusions_verified": bool(
            TVAE_ARTIFACT_VERIFICATION_DF[
                "provenance_exclusion_verified"
            ].all()
        ),
    },

    "overall_status": "PASS",

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

# -----------------------------------------------------------------------------------------------
# 11. Persist completion report
# -----------------------------------------------------------------------------------------------

TVAE_COMPLETION_REPORT_PATH = (
    NB05_VALIDATION_ROOT
    / "tvae_completion_report.json"
)

with open(
    TVAE_COMPLETION_REPORT_PATH,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        TVAE_COMPLETION_REPORT,
        handle,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )

if not TVAE_COMPLETION_REPORT_PATH.exists():

    raise RuntimeError(
        "TVAE completion report was not persisted."
    )

if TVAE_COMPLETION_REPORT_PATH.stat().st_size <= 0:

    raise RuntimeError(
        "TVAE completion report is empty."
    )

# -----------------------------------------------------------------------------------------------
# 12. Final summary
# -----------------------------------------------------------------------------------------------

print()
print("-" * 100)

print(
    f"✓ Expected TVAE runs   : "
    f"{expected_runs}"
)

print(
    f"✓ Models               : "
    f"{len(TVAE_MODEL_DF)}"
)

print(
    f"✓ Checkpoints          : "
    f"{checkpoint_count}"
)

print(
    f"✓ Synthetic datasets   : "
    f"{len(TVAE_SYNTHETIC_ARTIFACT_DF)}"
)

print(
    f"✓ Metadata records     : "
    f"{len(TVAE_MANIFEST_DF)}"
)

print(
    f"✓ Artifact verification: "
    f"{len(TVAE_ARTIFACT_VERIFICATION_DF)}/{expected_runs} PASS"
)

print(
    f"✓ Completion report    : "
    f"{TVAE_COMPLETION_REPORT_PATH}"
)

print()
print("=" * 100)
print("✓ NOTEBOOK 05 — TVAE BASELINE : PASS")
print("=" * 100)

# -----------------------------------------------------------------------------------------------
# 13. RAM cleanup
# -----------------------------------------------------------------------------------------------

for variable_name in [
    "TVAE_SYNTHETIC_DATA",
    "TRAINING_DATA",
]:

    if variable_name in globals():

        del globals()[
            variable_name
        ]

gc.collect()

print()
print(
    "✓ RAM cleanup completed."
)

SECTION 18 — COMPLETION SUMMARY

NOTEBOOK 05 — TVAE BASELINE
Notebook ID           : 05
Notebook version      : 1.0
Project root          : /content/drive
Notebook 05 output    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_05

DATASETS
----------------------------------------------------------------------------------------------------
✓ adult_income         | training rows=34,189
✓ bank_marketing       | training rows=31,647
✓ diabetes_130us       | training rows=71,236

MODEL
----------------------------------------------------------------------------------------------------
✓ TVAESynthesizer
✓ Epochs               : 300
✓ Embedding dimension  : 128
✓ Compress dimensions  : (128, 128)
✓ Decompress dimensions: (128, 128)
✓ Loss factor           : 2
✓ Batch size            : 500
✓ Requested GPU         : True

EXPERIMENTAL INTEGRITY
----------------------------------------------------------------------------------------------------
✓ TRAIN split only used for fitt